In [ ]:
import pandas as pd

# Full path to your file
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"

# Load the Excel file (default loads first sheet)
df = pd.read_excel(file_path)

# Check first few rows
print(df.head())

# Optional: check data types and missing values
print(df.info())


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Use one day after the last invoice date as the snapshot date
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)


In [ ]:
# Group by CustomerID to calculate RFM
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency (number of invoices)
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # Monetary (total spending)
}).rename(columns={'InvoiceDate':'Recency', 'InvoiceNo':'Frequency', 'UnitPrice':'Monetary'})

# Optional: view the first few rows
print(rfm.head())


In [ ]:
# Assuming you already have `rfm` dataframe from RFM calculation

# RF subset
rf = rfm[['Recency', 'Frequency']]

# FM subset
fm = rfm[['Frequency', 'Monetary']]

# RM subset
rm = rfm[['Recency', 'Monetary']]


In [ ]:
# Assuming you already have `rfm` dataframe from RFM calculation

# RF subset
rf = rfm[['Recency', 'Frequency']]

# FM subset
fm = rfm[['Frequency', 'Monetary']]

# RM subset
rm = rfm[['Recency', 'Monetary']]

In [ ]:
# Aggregate transactional data per customer
raw_features = df.groupby('CustomerID').agg({
    'Quantity': 'sum',   # Total quantity purchased
    'InvoiceNo': 'nunique',  # Total number of invoices/orders
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # Total spend
}).rename(columns={'InvoiceNo':'TotalOrders', 'UnitPrice':'TotalSpend'})

# Optional: add average basket value
raw_features['AvgBasket'] = raw_features['TotalSpend'] / raw_features['TotalOrders']

# Check the first few rows
print(raw_features.head())


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features), 
                          columns=raw_features.columns, 
                          index=raw_features.index)

# Check
print(raw_scaled.head())


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def cluster_and_evaluate(X_scaled, model_name, model):
    """
    X_scaled: standardized feature set (DataFrame)
    model_name: string, name of the clustering model
    model: initialized clustering model
    """
    # Fit model
    if model_name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)
    
    # Remove noise points for DBSCAN (-1) when calculating metrics
    mask = labels != -1 if model_name == "DBSCAN" else np.array([True]*len(labels))
    
    # Silhouette Score
    sil_score = silhouette_score(X_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    # Davies–Bouldin Index
    dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    
    # PCA for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', legend='full')
    plt.title(f'{model_name} Clustering PCA Projection')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.show()
    
    print(f"{model_name} -> Silhouette Score: {sil_score:.3f}, DBI: {dbi_score:.3f}")
    
    return labels, sil_score, dbi_score


In [ ]:
# Number of clusters for centroid-based and hierarchical models
n_clusters = 4

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
dbscan = DBSCAN(eps=1.5, min_samples=5)
hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
birch = Birch(n_clusters=n_clusters)


In [ ]:
# Example
dbscan = DBSCAN(eps=5, min_samples=5)  # Increase eps to form larger clusters


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features), 
                          columns=raw_features.columns, 
                          index=raw_features.index)


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture

# Example
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(raw_scaled)


In [ ]:
# -----------------------------
# 1. Import libraries
# -----------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# -----------------------------
# 2. Aggregate raw transactional features
# -----------------------------
# Example: using your loaded df
raw_features = df.groupby('CustomerID').agg({
    'Quantity': 'sum',                              # Total quantity purchased
    'InvoiceNo': 'nunique',                         # Total number of orders
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # Total spend
}).rename(columns={'InvoiceNo':'TotalOrders', 'UnitPrice':'TotalSpend'})

# Add average basket value
raw_features['AvgBasket'] = raw_features['TotalSpend'] / raw_features['TotalOrders']

# -----------------------------
# 3. Standardize features
# -----------------------------
scaler = StandardScaler()
raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features), 
                          columns=raw_features.columns, 
                          index=raw_features.index)

# -----------------------------
# 4. Define a helper function for clustering + evaluation
# -----------------------------
def cluster_and_evaluate(X_scaled, model_name, model):
    # Fit model
    if model_name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)
    
    # Count clusters and noise
    cluster_counts = Counter(labels)
    n_clusters = len([c for c in cluster_counts if c != -1])
    n_noise = cluster_counts.get(-1, 0)
    
    # Filter out noise for metrics
    mask = labels != -1
    unique_labels = set(labels[mask])
    
    if len(unique_labels) >= 2:
        sil_score = silhouette_score(X_scaled[mask], labels[mask])
        dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil_score = float('nan')
        dbi_score = float('nan')
    
    print(f"{model_name} -> Clusters: {n_clusters}, Noise: {n_noise}, Silhouette: {sil_score:.3f}, DBI: {dbi_score:.3f}")
    
    # PCA for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', legend='full', s=50)
    plt.title(f"{model_name} Clustering PCA Projection")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.show()
    
    return labels, sil_score, dbi_score

# -----------------------------
# 5. Initialize models
# -----------------------------
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
dbscan = DBSCAN(eps=1.5, min_samples=5)  # Adjust eps if necessary
hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
birch = Birch(n_clusters=n_clusters)

models = {
    "K-Means": kmeans,
    "DBSCAN": dbscan,
    "Hierarchical": hierarchical,
    "GMM": gmm,
    "BIRCH": birch
}

# -----------------------------
# 6. Run clustering on raw features
# -----------------------------
results = {}

for m_name, m_model in models.items():
    print(f"\n--- Model: {m_name} ---")
    labels, sil_score, dbi_score = cluster_and_evaluate(raw_scaled, m_name, m_model)
    results[m_name] = {"labels": labels, "Silhouette": sil_score, "DBI": dbi_score}


In [ ]:
rfm = rfm[rfm['Monetary'] > 0]


In [ ]:
print(rfm.shape)


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# -----------------------------
# 2. Data Preprocessing
# -----------------------------

# Remove rows with missing CustomerID or Description
df = df.dropna(subset=['CustomerID', 'Description'])

# Remove negative or zero Quantity
df = df[df['Quantity'] > 0]

# Remove duplicates
df = df.drop_duplicates()

print(f"After cleaning, dataset shape: {df.shape}")

# -----------------------------
# 3. RFM Feature Engineering
# -----------------------------

# Define snapshot date (1 day after last InvoiceDate)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Aggregate per customer
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,              # Recency
    'InvoiceNo': 'nunique',                                              # Frequency
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])       # Monetary
}).rename(columns={'InvoiceDate':'Recency', 'InvoiceNo':'Frequency', 'UnitPrice':'Monetary'})

# -----------------------------
# 4. Remove zero Monetary customers
# -----------------------------
rfm = rfm[rfm['Monetary'] > 0]

# -----------------------------
# 5. Optional: log-transform Monetary
# -----------------------------
rfm['Monetary'] = np.log1p(rfm['Monetary'])  # log(M + 1) to reduce skew

# -----------------------------
# 6. Check final RFM table
# -----------------------------
print(f"RFM table shape (unique customers): {rfm.shape}")
print(rfm.head())


In [ ]:
rfm = rfm[rfm['Monetary'] > 0]


In [ ]:
print(df.isnull().sum())


In [ ]:
duplicates = df[df.duplicated()]
print(duplicates.shape)


In [ ]:
rfm['Monetary'] = np.log1p(rfm['Monetary'])


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# -----------------------------
# 2. Inspect missing values
# -----------------------------
print("Missing values per column:")
print(df.isnull().sum())

# -----------------------------
# 3. Remove null CustomerID and Description (like journal)
# -----------------------------
df = df.dropna(subset=['CustomerID', 'Description'])

# -----------------------------
# 4. Remove negative Quantity
# -----------------------------
df = df[df['Quantity'] > 0]

# -----------------------------
# 5. Remove duplicates
# -----------------------------
df = df.drop_duplicates()

# -----------------------------
# 6. Check final transaction count
# -----------------------------
print(f"After cleaning, dataset shape: {df.shape}")

# -----------------------------
# 7. Create RFM table per customer
# -----------------------------
# Define snapshot date (1 day after last InvoiceDate)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # Monetary
}).rename(columns={'InvoiceDate':'Recency', 'InvoiceNo':'Frequency', 'UnitPrice':'Monetary'})

# -----------------------------
# 8. Remove customers with zero Monetary (like journal)
# -----------------------------
rfm = rfm[rfm['Monetary'] > 0]

# -----------------------------
# 9. Optional: log-transform Monetary
# -----------------------------
rfm['Monetary'] = np.log1p(rfm['Monetary'])  # reduces skew

# -----------------------------
# 10. Check final RFM table
# -----------------------------
print(f"RFM table shape (unique customers): {rfm.shape}")
print(rfm.head())


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# -----------------------------
# 2. Data Preprocessing
# -----------------------------
# Remove null CustomerID and Description
df = df.dropna(subset=['CustomerID', 'Description'])

# Remove negative or zero Quantity
df = df[df['Quantity'] > 0]

# Remove duplicates
df = df.drop_duplicates()

print(f"After cleaning, dataset shape: {df.shape}")  # check transaction count

# -----------------------------
# 3. Create raw RFM table per customer
# -----------------------------
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm_raw = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,          # Recency
    'InvoiceNo': 'nunique',                                           # Frequency
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])    # Monetary
}).rename(columns={'InvoiceDate':'Recency', 'InvoiceNo':'Frequency', 'UnitPrice':'Monetary'})

# Remove zero Monetary customers
rfm_raw = rfm_raw[rfm_raw['Monetary'] > 0]

# -----------------------------
# 4. Create log-transformed RFM for clustering
# -----------------------------
rfm_log = rfm_raw.copy()
rfm_log['Monetary'] = np.log1p(rfm_log['Monetary'])  # log(M + 1)

# -----------------------------
# 5. Optional: Standardize for clustering
# -----------------------------
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm_log),
                          columns=rfm_log.columns,
                          index=rfm_log.index)

# -----------------------------
# 6. Display examples
# -----------------------------
print("RFM Table (Raw Monetary, GBP) - for report")
print(rfm_raw.head())

print("\nRFM Table (Log-transformed Monetary) - for clustering")
print(rfm_scaled.head())


In [ ]:
# -----------------------------
# 1. Import required libraries
# -----------------------------
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

# -----------------------------
# 2. Initialize models
# -----------------------------
n_clusters = 4  # as in John et al., 2024
models = {
    "K-Means": KMeans(n_clusters=n_clusters, random_state=42),
    "GMM": GaussianMixture(n_components=n_clusters, random_state=42, n_init=10),
    "DBSCAN": DBSCAN(eps=1.5, min_samples=5),  # tune eps if needed
    "Hierarchical": AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
    "BIRCH": Birch(n_clusters=n_clusters)
}

# -----------------------------
# 3. Helper function for clustering + evaluation
# -----------------------------
def cluster_and_evaluate(X_scaled, model_name, model):
    # Fit model
    if model_name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)

    # Count clusters and noise
    cluster_counts = Counter(labels)
    n_clusters_found = len([c for c in cluster_counts if c != -1])
    n_noise = cluster_counts.get(-1, 0)

    # Filter out noise for metrics
    mask = labels != -1
    unique_labels = set(labels[mask])
    if len(unique_labels) >= 2:
        sil_score = silhouette_score(X_scaled[mask], labels[mask])
        dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil_score = float('nan')
        dbi_score = float('nan')

    # Print metrics
    print(f"{model_name} -> Clusters: {n_clusters_found}, Noise: {n_noise}, "
          f"Silhouette: {sil_score:.3f}, DBI: {dbi_score:.3f}")

    # PCA for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', s=50, legend='full')
    plt.title(f"{model_name} Clustering PCA Projection")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.show()

    return labels, sil_score, dbi_score

# -----------------------------
# 4. Run all models on RFM scaled
# -----------------------------
results = {}
for m_name, m_model in models.items():
    print(f"\n--- Running {m_name} ---")
    labels, sil, dbi = cluster_and_evaluate(rfm_scaled, m_name, m_model)
    results[m_name] = {"labels": labels, "Silhouette": sil, "DBI": dbi}

# -----------------------------
# 5. Optional: Summary table of metrics
# -----------------------------
import pandas as pd
summary_rows = []
for m_name, metrics in results.items():
    labels = metrics['labels']
    cluster_counts = Counter(labels)
    n_clusters_found = len([c for c in cluster_counts if c != -1])
    n_noise = cluster_counts.get(-1, 0)
    summary_rows.append({
        "Model": m_name,
        "Silhouette": round(metrics['Silhouette'],3),
        "DBI": round(metrics['DBI'],3),
        "Num_Clusters": n_clusters_found,
        "Num_Noise": n_noise
    })

summary_df = pd.DataFrame(summary_rows)
print("\n--- Summary Table ---")
print(summary_df)


In [ ]:
import pandas as pd
import numpy as np

# Assume rfm_raw is your RFM table (Recency, Frequency, Monetary in GBP)
rfm_ranked = rfm_raw.copy()

# Step 1: Rank each R, F, M (1-5)
rfm_ranked['RecencyScore'] = pd.qcut(rfm_ranked['Recency'], 5, labels=[5,4,3,2,1])  # lower Recency = better
rfm_ranked['FrequencyScore'] = pd.qcut(rfm_ranked['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm_ranked['MonetaryScore'] = pd.qcut(rfm_ranked['Monetary'], 5, labels=[1,2,3,4,5])

# Step 2: Weighted RFM score (example weights: 0.3,0.4,0.3)
w_R, w_F, w_M = 0.3, 0.4, 0.3
rfm_ranked['RFM_Score'] = (rfm_ranked['RecencyScore'].astype(int)*w_R +
                           rfm_ranked['FrequencyScore'].astype(int)*w_F +
                           rfm_ranked['MonetaryScore'].astype(int)*w_M)

# Step 3: Assign Customer_Segment
def assign_segment(score):
    if score > 4.5:
        return "Top customer"
    elif score > 4.0:
        return "High-value customer"
    elif score > 3.0:
        return "Medium-value customer"
    elif score > 1.6:
        return "Low-value customer"
    else:
        return "Lost customer"

rfm_ranked['Customer_Segment'] = rfm_ranked['RFM_Score'].apply(assign_segment)

# Step 4: Show top 10 records
print(rfm_ranked[['RFM_Score','Customer_Segment']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# -----------------------------
# 2. Clean dataset
# -----------------------------
df = df.dropna(subset=['CustomerID','Description'])  # remove nulls
df = df[df['Quantity'] > 0]                          # remove negative/zero quantity
df = df.drop_duplicates()                             # remove duplicates

print(f"Cleaned dataset shape: {df.shape}")

# -----------------------------
# 3. Create RFM features (raw GBP)
# -----------------------------
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm_raw = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index,'Quantity'])
}).rename(columns={'InvoiceDate':'Recency','InvoiceNo':'Frequency','UnitPrice':'Monetary'})

rfm_raw = rfm_raw[rfm_raw['Monetary'] > 0]  # remove zero spend

# -----------------------------
# 4. Rank RFM scores (1-5) and compute weighted RFM_Score
# -----------------------------
rfm_ranked = rfm_raw.copy()
rfm_ranked['RecencyScore'] = pd.qcut(rfm_ranked['Recency'], 5, labels=[5,4,3,2,1])
rfm_ranked['FrequencyScore'] = pd.qcut(rfm_ranked['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm_ranked['MonetaryScore'] = pd.qcut(rfm_ranked['Monetary'], 5, labels=[1,2,3,4,5])

# Weighted RFM Score
w_R, w_F, w_M = 0.3, 0.4, 0.3
rfm_ranked['RFM_Score'] = (rfm_ranked['RecencyScore'].astype(int)*w_R +
                           rfm_ranked['FrequencyScore'].astype(int)*w_F +
                           rfm_ranked['MonetaryScore'].astype(int)*w_M)

# Assign Customer Segment
def assign_segment(score):
    if score > 4.5:
        return "Top customer"
    elif score > 4.0:
        return "High-value customer"
    elif score > 3.0:
        return "Medium-value customer"
    elif score > 1.6:
        return "Low-value customer"
    else:
        return "Lost customer"

rfm_ranked['Customer_Segment'] = rfm_ranked['RFM_Score'].apply(assign_segment)

# -----------------------------
# 5. Prepare ranked RFM for clustering
# -----------------------------
rfm_for_clustering = rfm_ranked[['RecencyScore','FrequencyScore','MonetaryScore']]
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm_for_clustering),
                          columns=rfm_for_clustering.columns,
                          index=rfm_for_clustering.index)

# -----------------------------
# 6. Clustering pipeline
# -----------------------------
n_clusters = 4
models = {
    "K-Means": KMeans(n_clusters=n_clusters, random_state=42),
    "GMM": GaussianMixture(n_components=n_clusters, random_state=42, n_init=10),
    "DBSCAN": DBSCAN(eps=1.5, min_samples=5),
    "Hierarchical": AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
    "BIRCH": Birch(n_clusters=n_clusters)
}

results = {}

def cluster_and_evaluate(X_scaled, model_name, model):
    if model_name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)

    cluster_counts = Counter(labels)
    n_clusters_found = len([c for c in cluster_counts if c != -1])
    n_noise = cluster_counts.get(-1,0)

    mask = labels != -1
    unique_labels = set(labels[mask])
    if len(unique_labels) >= 2:
        sil_score = silhouette_score(X_scaled[mask], labels[mask])
        dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil_score, dbi_score = float('nan'), float('nan')

    print(f"{model_name} -> Clusters: {n_clusters_found}, Noise: {n_noise}, "
          f"Silhouette: {sil_score:.3f}, DBI: {dbi_score:.3f}")

    # PCA visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', s=50, legend='full')
    plt.title(f"{model_name} PCA Projection")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.show()

    return labels, sil_score, dbi_score

# Run all models
for name, model in models.items():
    print(f"\n--- {name} ---")
    labels, sil, dbi = cluster_and_evaluate(rfm_scaled, name, model)
    results[name] = {"labels": labels, "Silhouette": sil, "DBI": dbi}

# -----------------------------
# 7. Add cluster labels to RFM table
# -----------------------------
# Example: K-Means cluster
rfm_ranked['Cluster_KMeans'] = results['K-Means']['labels']

# Final table for reporting
final_table = rfm_ranked[['RecencyScore','FrequencyScore','MonetaryScore',
                          'RFM_Score','Customer_Segment','Cluster_KMeans']]
print(final_table.head(10))


In [ ]:
# Use raw Recency, Frequency, Monetary
rfm_for_clustering = rfm_raw[['Recency','Frequency','Monetary']]  # raw GBP values

# Optional: log-transform Monetary to reduce skew
rfm_for_clustering['Monetary'] = np.log1p(rfm_for_clustering['Monetary'])

# Standardize for clustering
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm_for_clustering),
                          columns=rfm_for_clustering.columns,
                          index=rfm_for_clustering.index)


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


In [ ]:
n_clusters = 4  # same as John et al., 2024
models = {
    "K-Means": KMeans(n_clusters=n_clusters, random_state=42),
    "GMM": GaussianMixture(n_components=n_clusters, random_state=42, n_init=10),
    "DBSCAN": DBSCAN(eps=1.5, min_samples=5),  # adjust eps if needed
    "Hierarchical": AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
    "BIRCH": Birch(n_clusters=n_clusters)
}


In [ ]:
def cluster_and_evaluate(X_scaled, model_name, model):
    # Fit model
    if model_name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)

    # Count clusters and noise
    cluster_counts = Counter(labels)
    n_clusters_found = len([c for c in cluster_counts if c != -1])
    n_noise = cluster_counts.get(-1, 0)

    # Filter noise for metrics
    mask = labels != -1
    unique_labels = set(labels[mask])
    if len(unique_labels) >= 2:
        sil_score = silhouette_score(X_scaled[mask], labels[mask])
        dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil_score, dbi_score = float('nan'), float('nan')

    print(f"{model_name} -> Clusters: {n_clusters_found}, Noise: {n_noise}, "
          f"Silhouette: {sil_score:.3f}, DBI: {dbi_score:.3f}")

    # PCA visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', s=50, legend='full')
    plt.title(f"{model_name} PCA Projection")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.show()

    return labels, sil_score, dbi_score


In [ ]:
results = {}
for name, model in models.items():
    print(f"\n--- {name} ---")
    labels, sil, dbi = cluster_and_evaluate(rfm_scaled, name, model)
    results[name] = {"labels": labels, "Silhouette": sil, "DBI": dbi}


In [ ]:
# Aggregate per customer
raw_features = df.groupby('CustomerID').agg({
    'Quantity': 'sum',                   # total quantity purchased
    'InvoiceNo': 'nunique',              # total orders
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # total spend
})

# Optional: average basket value
raw_features['AvgBasket'] = raw_features['UnitPrice'] / raw_features['InvoiceNo']

print(raw_features.head())


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features),
                          columns=raw_features.columns,
                          index=raw_features.index)


In [ ]:
# Example: run GMM on raw features
gmm = GaussianMixture(n_components=4, random_state=42, n_init=10)
labels = gmm.fit_predict(raw_scaled)

from sklearn.metrics import silhouette_score, davies_bouldin_score

sil = silhouette_score(raw_scaled, labels)
dbi = davies_bouldin_score(raw_scaled, labels)
print(f"GMM on raw features -> Silhouette: {sil:.3f}, DBI: {dbi:.3f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# -----------------------------
# 2. Clean dataset
# -----------------------------
df = df.dropna(subset=['CustomerID','Description'])
df = df[df['Quantity'] > 0]
df = df.drop_duplicates()

print(f"Cleaned dataset shape: {df.shape}")

# -----------------------------
# 3. Aggregate RFM (raw GBP)
# -----------------------------
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index,'Quantity'])
}).rename(columns={'InvoiceDate':'Recency','InvoiceNo':'Frequency','UnitPrice':'Monetary'})

# Remove zero Monetary customers
rfm = rfm[rfm['Monetary'] > 0]

# -----------------------------
# 4. Outlier removal
# -----------------------------
# Remove top 0.5% Monetary and Frequency as extreme outliers
rfm = rfm[rfm['Monetary'] <= rfm['Monetary'].quantile(0.995)]
rfm = rfm[rfm['Frequency'] <= rfm['Frequency'].quantile(0.995)]

# -----------------------------
# 5. Log-transform Monetary
# -----------------------------
rfm['Monetary_Log'] = np.log1p(rfm['Monetary'])

# -----------------------------
# 6. Weighted RFM features (continuous)
# -----------------------------
# Weights: Recency=0.3, Frequency=0.4, Monetary=0.3
rfm['Recency_W'] = (rfm['Recency'] / rfm['Recency'].max()) * 5 * 0.3  # scaled 0-5
rfm['Frequency_W'] = (rfm['Frequency'] / rfm['Frequency'].max()) * 5 * 0.4
rfm['Monetary_W'] = (rfm['Monetary_Log'] / rfm['Monetary_Log'].max()) * 5 * 0.3

rfm_for_clustering = rfm[['Recency_W','Frequency_W','Monetary_W']]

# -----------------------------
# 7. Standardize features
# -----------------------------
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm_for_clustering),
                          columns=rfm_for_clustering.columns,
                          index=rfm_for_clustering.index)

# -----------------------------
# 8. Define clustering models
# -----------------------------
n_clusters = 4
models = {
    "K-Means": KMeans(n_clusters=n_clusters, random_state=42),
    "GMM": GaussianMixture(n_components=n_clusters, random_state=42, n_init=10, covariance_type='full'),
    "DBSCAN": DBSCAN(eps=1.5, min_samples=5),
    "Hierarchical": AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
    "BIRCH": Birch(n_clusters=n_clusters)
}

# -----------------------------
# 9. Clustering + Evaluation function
# -----------------------------
def cluster_and_evaluate(X_scaled, model_name, model):
    if model_name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)
    
    cluster_counts = Counter(labels)
    n_clusters_found = len([c for c in cluster_counts if c != -1])
    n_noise = cluster_counts.get(-1,0)
    
    mask = labels != -1
    unique_labels = set(labels[mask])
    if len(unique_labels) >= 2:
        sil_score = silhouette_score(X_scaled[mask], labels[mask])
        dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil_score, dbi_score = float('nan'), float('nan')
    
    print(f"{model_name} -> Clusters: {n_clusters_found}, Noise: {n_noise}, "
          f"Silhouette: {sil_score:.3f}, DBI: {dbi_score:.3f}")
    
    # PCA visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', s=50, legend='full')
    plt.title(f"{model_name} PCA Projection")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.show()
    
    return labels, sil_score, dbi_score

# -----------------------------
# 10. Run clustering
# -----------------------------
results = {}
for name, model in models.items():
    print(f"\n--- {name} ---")
    labels, sil, dbi = cluster_and_evaluate(rfm_scaled, name, model)
    results[name] = {"labels": labels, "Silhouette": sil, "DBI": dbi}

# -----------------------------
# 11. Add GMM cluster labels to RFM table for reporting
# -----------------------------
rfm['Cluster_GMM'] = results['GMM']['labels']

# Example: show top 10 records with cluster
final_table = rfm[['Recency','Frequency','Monetary','Cluster_GMM']]
print(final_table.head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# -----------------------------
# 1. Load dataset
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# -----------------------------
# 2. Initial cleaning
# -----------------------------
df = df.dropna(subset=['CustomerID','Description'])
df = df[df['Quantity'] > 0]
df = df.drop_duplicates()

print(f"Cleaned dataset shape: {df.shape}")

# -----------------------------
# 3. IQR-based outlier removal on Quantity and UnitPrice
# -----------------------------
Q1_qty, Q3_qty = df['Quantity'].quantile([0.25,0.75])
IQR_qty = Q3_qty - Q1_qty
qty_lower = Q1_qty - 1.5*IQR_qty
qty_upper = Q3_qty + 1.5*IQR_qty

Q1_price, Q3_price = df['UnitPrice'].quantile([0.25,0.75])
IQR_price = Q3_price - Q1_price
price_lower = Q1_price - 1.5*IQR_price
price_upper = Q3_price + 1.5*IQR_price

df = df[(df['Quantity'] >= qty_lower) & (df['Quantity'] <= qty_upper)]
df = df[(df['UnitPrice'] >= price_lower) & (df['UnitPrice'] <= price_upper)]

print(f"Dataset shape after IQR outlier removal: {df.shape}")

# -----------------------------
# 4. Compute RFM (raw continuous)
# -----------------------------
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index,'Quantity'])
}).rename(columns={'InvoiceDate':'Recency','InvoiceNo':'Frequency','UnitPrice':'Monetary'})

rfm = rfm[rfm['Monetary'] > 0]

# -----------------------------
# 5. Log-transform all RFM features
# -----------------------------
rfm_log = rfm.copy()
rfm_log['Recency'] = np.log1p(rfm_log['Recency'])
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm_log['Monetary'])

# -----------------------------
# 6. Standardize features
# -----------------------------
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm_log),
                          columns=rfm_log.columns,
                          index=rfm_log.index)

# -----------------------------
# 7. GMM Clustering
# -----------------------------
gmm = GaussianMixture(n_components=3, covariance_type='full', n_init=10, random_state=42)
labels = gmm.fit_predict(rfm_scaled)

# -----------------------------
# 8. Evaluation
# -----------------------------
sil = silhouette_score(rfm_scaled, labels)
dbi = davies_bouldin_score(rfm_scaled, labels)
print(f"GMM -> Clusters: 3, Silhouette: {sil:.3f}, DBI: {dbi:.3f}")

# -----------------------------
# 9. PCA visualization
# -----------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(rfm_scaled)

plt.figure(figsize=(6,5))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette='tab10', s=50, legend='full')
plt.title("GMM Clustering PCA Projection")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()

# -----------------------------
# 10. Add cluster labels to RFM table for reporting
# -----------------------------
rfm['Cluster_GMM'] = labels
print(rfm.head(10))


In [ ]:
# Using the cleaned, log-transformed, standardized RFM features
# Assume rfm_log and rfm_scaled from previous steps exist

# RF: Recency + Frequency
rf = rfm_scaled[['Recency','Frequency']]

# RM: Recency + Monetary
rm = rfm_scaled[['Recency','Monetary']]

# FM: Frequency + Monetary
fm = rfm_scaled[['Frequency','Monetary']]


In [ ]:
def run_clustering(X_scaled, subset_name):
    print(f"\n--- Results for {subset_name} ---")
    results_subset = {}
    
    for name, model in models.items():
        # Fit model
        if name == "GMM":
            labels = model.fit_predict(X_scaled)
        else:
            model.fit(X_scaled)
            labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)

        # Cluster counts and noise
        cluster_counts = Counter(labels)
        n_clusters_found = len([c for c in cluster_counts if c != -1])
        n_noise = cluster_counts.get(-1,0)
        
        # Silhouette and DBI
        mask = labels != -1
        if len(set(labels[mask])) >= 2:
            sil_score = silhouette_score(X_scaled[mask], labels[mask])
            dbi_score = davies_bouldin_score(X_scaled[mask], labels[mask])
        else:
            sil_score, dbi_score = float('nan'), float('nan')
        
        # Print results
        print(f"{name}: Clusters={n_clusters_found}, Noise={n_noise}, Silhouette={sil_score:.3f}, DBI={dbi_score:.3f}")
        
        results_subset[name] = {'labels': labels, 'Silhouette': sil_score, 'DBI': dbi_score}
        
    return results_subset

# Run for RF, RM, FM
results_RF = run_clustering(rf, "RF (Recency+Frequency)")
results_RM = run_clustering(rm, "RM (Recency+Monetary)")
results_FM = run_clustering(fm, "FM (Frequency+Monetary)")


In [ ]:
# Remove missing IDs
df = df.dropna(subset=['CustomerID'])

# Remove cancellations
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove invalid quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Remove duplicates
df = df.drop_duplicates()


In [ ]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    return df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]

df = iqr_filter(df, 'Quantity')
df = iqr_filter(df, 'UnitPrice')


In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']


In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']


In [ ]:
rfm.head()


In [ ]:
rfm.describe()


In [ ]:
rfm_log = np.log1p(rfm)


In [ ]:
rfm_log.describe()


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)


In [ ]:
rfm_filtered = rfm_log[
    (rfm_log['Frequency'] > rfm_log['Frequency'].quantile(0.20)) &
    (rfm_log['Monetary'] > rfm_log['Monetary'].quantile(0.20))
]


In [ ]:
rfm_filtered.shape


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled_filtered = scaler.fit_transform(rfm_filtered)


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    n_init=20,
    random_state=42
)

labels = gmm.fit_predict(rfm_scaled_filtered)

sil = silhouette_score(rfm_scaled_filtered, labels)
dbi = davies_bouldin_score(rfm_scaled_filtered, labels)

print(f"GMM Silhouette: {sil:.3f}")
print(f"GMM DBI: {dbi:.3f}")


In [ ]:
rfm_filtered = rfm_log[
    (rfm_log['Frequency'] > rfm_log['Frequency'].quantile(0.30)) &
    (rfm_log['Monetary'] > rfm_log['Monetary'].quantile(0.30))
]

scaler = StandardScaler()
rfm_scaled_filtered = scaler.fit_transform(rfm_filtered)

labels = gmm.fit_predict(rfm_scaled_filtered)

sil = silhouette_score(rfm_scaled_filtered, labels)
dbi = davies_bouldin_score(rfm_scaled_filtered, labels)

print(f"GMM Silhouette: {sil:.3f}")
print(f"GMM DBI: {dbi:.3f}")


In [ ]:
duplicates='drop'


In [ ]:
rfm_scores = rfm.copy()


In [ ]:
rfm_scores['R_Score'] = pd.qcut(
    rfm_scores['Recency'],
    5,
    labels=[5,4,3,2,1]
)


In [ ]:
rfm_scores['F_Rank'] = rfm_scores['Frequency'].rank(method='first')

rfm_scores['F_Score'] = pd.qcut(
    rfm_scores['F_Rank'],
    5,
    labels=[1,2,3,4,5]
)


In [ ]:
rfm_scores['M_Score'] = pd.qcut(
    rfm_scores['Monetary'],
    5,
    labels=[1,2,3,4,5]
)


In [ ]:
rfm_scores[['R_Score','F_Score','M_Score']] = (
    rfm_scores[['R_Score','F_Score','M_Score']].astype(int)
)

rfm_scores.drop(columns=['F_Rank'], inplace=True)


In [ ]:
rfm_scores[['R_Score','F_Score','M_Score']].head()


In [ ]:
from sklearn.preprocessing import StandardScaler

X = rfm_scores[['R_Score','F_Score','M_Score']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

gmm = GaussianMixture(
    n_components=3,          # journal used K=3
    covariance_type='full',
    n_init=30,               # very important
    random_state=42
)

labels = gmm.fit_predict(X_scaled)

print("GMM Silhouette:", round(silhouette_score(X_scaled, labels), 3))
print("GMM DBI:", round(davies_bouldin_score(X_scaled, labels), 3))


In [ ]:
rfm_scores['RFM_Weighted'] = (
    0.3 * rfm_scores['R_Score'] +
    0.3 * rfm_scores['F_Score'] +
    0.4 * rfm_scores['M_Score']
)


In [ ]:
from sklearn.preprocessing import StandardScaler
X_weighted = rfm_scores[['RFM_Weighted']]
X_scaled = StandardScaler().fit_transform(X_weighted)


In [ ]:
gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    n_init=30,
    random_state=42
)

labels = gmm.fit_predict(X_scaled)

from sklearn.metrics import silhouette_score, davies_bouldin_score

sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)

print("GMM Silhouette:", round(sil, 3))
print("GMM DBI:", round(dbi, 3))


In [ ]:
rfm_scores['RFM_Weighted'] = (
    0.2 * rfm_scores['R_Score'] +
    0.4 * rfm_scores['F_Score'] +
    0.4 * rfm_scores['M_Score']
)


In [ ]:
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.3)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.3))
]


In [ ]:
best_sil = -1
best_k = 0
for k in range(2,6):
    gmm = GaussianMixture(n_components=k, covariance_type='full', n_init=30, random_state=42)
    labels = gmm.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    if sil > best_sil:
        best_sil = sil
        best_k = k

print(f"Best Silhouette={best_sil:.3f} with K={best_k}")


In [ ]:
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.35)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.35))
]


In [ ]:
import numpy as np
X_scaled_jitter = X_scaled + np.random.normal(0, 0.02, X_scaled.shape)


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Define GMM with journal-like parameters
gmm = GaussianMixture(
    n_components=2,           # Best K from your earlier test
    covariance_type='full',
    n_init=30,
    random_state=42
)

# Fit GMM on jittered scaled data
labels = gmm.fit_predict(X_scaled_jitter)


In [ ]:
sil = silhouette_score(X_scaled_jitter, labels)
dbi = davies_bouldin_score(X_scaled_jitter, labels)

print(f"GMM Silhouette (with jitter): {sil:.3f}")
print(f"GMM DBI (with jitter): {dbi:.3f}")


In [ ]:
# Already computed
# rfm_scores['RFM_Weighted'] = 0.3*R + 0.3*F + 0.4*M
X = rfm_scores[['RFM_Weighted']]

# Standardize for all models
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.mixture import GaussianMixture

models = {
    "K-Means": KMeans(n_clusters=2, random_state=42),
    "GMM": GaussianMixture(n_components=2, covariance_type='full', n_init=30, random_state=42),
    "DBSCAN": DBSCAN(eps=0.5, min_samples=5),  # tune eps as needed
    "Hierarchical": AgglomerativeClustering(n_clusters=2, linkage='ward'),
    "BIRCH": Birch(n_clusters=2)
}


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from collections import Counter
import numpy as np

results = []

for name, model in models.items():
    if name == "GMM":
        labels = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        labels = model.labels_ if hasattr(model, 'labels_') else model.predict(X_scaled)
    
    # Handle noise points for DBSCAN
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    n_noise = (labels == -1).sum()
    
    # Silhouette & DBI: only if at least 2 clusters exist
    if n_clusters >= 2:
        sil = silhouette_score(X_scaled[mask], labels[mask])
        dbi = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil, dbi = np.nan, np.nan
    
    results.append({
        "Feature_Set": "Weighted RFM",
        "Model": name,
        "Silhouette": round(sil, 3) if not np.isnan(sil) else np.nan,
        "DBI": round(dbi, 3) if not np.isnan(dbi) else np.nan,
        "Num_Clusters": n_clusters,
        "Num_Noise": n_noise
    })

import pandas as pd
summary_table = pd.DataFrame(results)
print(summary_table)


In [ ]:
# Use the existing rfm_scores dataframe columns
rfm_scores['RFM_Weighted'] = (
    0.2 * rfm_scores['R_Score'] + 
    0.4 * rfm_scores['F_Score'] + 
    0.4 * rfm_scores['M_Score']
)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_scores[['RFM_Weighted']])


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',
    n_init=50,
    random_state=42
)

labels = gmm.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)

print(f"GMM Silhouette: {sil:.3f}")
print(f"GMM DBI: {dbi:.3f}")


In [ ]:
rfm_scores['RFM_Weighted'] = 0.2*rfm_scores['R_Score'] + 0.3*rfm_scores['F_Score'] + 0.5*rfm_scores['M_Score']


In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',  # allows ellipsoidal clusters
    n_init=50,               # better EM convergence
    random_state=42
)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Take only the weighted RFM column
X = rfm_scores[['RFM_Weighted']]

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=2,           # Best K from earlier
    covariance_type='full',   # Allows ellipsoidal clusters
    n_init=50,                # Better EM convergence
    random_state=42
)

# Fit and predict cluster labels
labels_gmm = gmm.fit_predict(X_scaled)


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}")
print(f"GMM DBI: {dbi_gmm:.3f}")


In [ ]:
rfm_scores['RFM_Weighted'] = 0.2*rfm_scores['R_Score'] + 0.3*rfm_scores['F_Score'] + 0.5*rfm_scores['M_Score']


In [ ]:
from sklearn.mixture import GaussianMixture

# Create the GMM
gmm = GaussianMixture(
    n_components=2,         # number of clusters
    covariance_type='full', # allows ellipsoidal clusters
    n_init=50,              # multiple initializations for better convergence
    random_state=42
)

# Fit GMM and predict cluster labels
labels_gmm = gmm.fit_predict(X_scaled)


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}")
print(f"GMM DBI: {dbi_gmm:.3f}")


In [ ]:
rfm_scores['RFM_Weighted'] = 0.2*rfm_scores['R_Score'] + 0.3*rfm_scores['F_Score'] + 0.5*rfm_scores['M_Score']

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}")
print(f"GMM DBI: {dbi_gmm:.3f}")


In [ ]:
rfm_scores['RFM_Weighted'] = (
    0.1*rfm_scores['R_Score'] +
    0.3*rfm_scores['F_Score'] +
    0.6*rfm_scores['M_Score']
)


In [ ]:
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.25)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.25))
]


In [ ]:
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(rfm_filtered[['RFM_Weighted']])


In [ ]:
import numpy as np
X_scaled_gmm = X_scaled + np.random.normal(0, 0.01, X_scaled.shape)


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

gmm = GaussianMixture(
    n_components=2,          # same as K-Means
    covariance_type='full',  # full ellipsoidal clusters
    n_init=50,               # more restarts → better convergence
    random_state=42
)

labels_gmm = gmm.fit_predict(X_scaled_gmm)

sil_gmm = silhouette_score(X_scaled_gmm, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled_gmm, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}")
print(f"GMM DBI: {dbi_gmm:.3f}")


In [ ]:
from sklearn.cluster import KMeans

X_scaled_k = X_scaled  # no jitter for K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled_k)

sil_k = silhouette_score(X_scaled_k, labels_k)
print(f"K-Means Silhouette: {sil_k:.3f}")


In [ ]:
rfm_scores['RFM_Weighted'] = (
    0.1 * rfm_scores['R_Score'] +
    0.25 * rfm_scores['F_Score'] +
    0.65 * rfm_scores['M_Score']
)


In [ ]:
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.25)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.25))
]


In [ ]:
from sklearn.preprocessing import StandardScaler

X = rfm_filtered[['RFM_Weighted']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
import numpy as np

X_scaled_gmm = X_scaled + np.random.normal(0, 0.01, X_scaled.shape)


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',  # full covariance for ellipsoids
    n_init=50,               # multiple EM restarts
    random_state=42
)

labels_gmm = gmm.fit_predict(X_scaled_gmm)

sil_gmm = silhouette_score(X_scaled_gmm, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled_gmm, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}")
print(f"GMM DBI: {dbi_gmm:.3f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assume rfm_scores exists with R_Score, F_Score, M_Score

# 1️⃣ Adjust weighted RFM to emphasize Monetary
rfm_scores['RFM_Weighted'] = (
    0.1 * rfm_scores['R_Score'] + 
    0.25 * rfm_scores['F_Score'] + 
    0.65 * rfm_scores['M_Score']
)

# 2️⃣ Filter out low-value customers (bottom 25% Frequency and Monetary)
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.25)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.25))
].copy()

# 3️⃣ Standardize weighted RFM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['RFM_Weighted']])

# 4️⃣ Add tiny jitter ONLY for GMM
X_scaled_gmm = X_scaled + np.random.normal(0, 0.01, X_scaled.shape)

# 5️⃣ Fit GMM
gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',
    n_init=50,
    random_state=42
)
labels_gmm = gmm.fit_predict(X_scaled_gmm)

# Evaluate GMM
sil_gmm = silhouette_score(X_scaled_gmm, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled_gmm, labels_gmm)

# 6️⃣ Fit K-Means on same data WITHOUT jitter
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)

sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)

# 7️⃣ Assign cluster labels back to dataframe
rfm_filtered['Cluster_GMM'] = labels_gmm
rfm_filtered['Cluster_KMeans'] = labels_k

# 8️⃣ Print results
print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")
print(f"K-Means Silhouette: {sil_k:.3f}, DBI: {dbi_k:.3f}")

# Optional: show top 10 customers with clusters
print(rfm_filtered[['RFM_Weighted','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
rfm_scores['RFM_Weighted'] = (
    0.05 * rfm_scores['R_Score'] +
    0.25 * rfm_scores['F_Score'] +
    0.7 * rfm_scores['M_Score']
)


In [ ]:
X_scaled_gmm = X_scaled + np.random.normal(0, 0.02, X_scaled.shape)


In [ ]:
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.30)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.30))
].copy()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assume rfm_scores exists with columns: R_Score, F_Score, M_Score, Frequency, Monetary

# 1️⃣ Weighted RFM (Monetary emphasized)
rfm_scores['RFM_Weighted'] = (
    0.05 * rfm_scores['R_Score'] +
    0.25 * rfm_scores['F_Score'] +
    0.70 * rfm_scores['M_Score']
)

# 2️⃣ Filter bottom 30% low-value customers
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.30)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.30))
].copy()

# 3️⃣ Standardize weighted RFM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['RFM_Weighted']])

# 4️⃣ Add tiny jitter ONLY for GMM
X_scaled_gmm = X_scaled + np.random.normal(0, 0.02, X_scaled.shape)

# 5️⃣ Fit GMM
gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',
    n_init=50,
    random_state=42
)
labels_gmm = gmm.fit_predict(X_scaled_gmm)

# 6️⃣ Evaluate GMM
sil_gmm = silhouette_score(X_scaled_gmm, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled_gmm, labels_gmm)

# 7️⃣ Fit K-Means on same filtered weighted RFM WITHOUT jitter
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)

sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)

# 8️⃣ Assign cluster labels back to dataframe
rfm_filtered['Cluster_GMM'] = labels_gmm
rfm_filtered['Cluster_KMeans'] = labels_k

# 9️⃣ Print results
print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")
print(f"K-Means Silhouette: {sil_k:.3f}, DBI: {dbi_k:.3f}")

# Optional: show top 10 customers
print(rfm_filtered[['RFM_Weighted','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Weighted RFM with high Monetary emphasis
# -------------------------------
# Assume rfm_scores exists with columns: R_Score, F_Score, M_Score, Frequency, Monetary
rfm_scores['RFM_Weighted'] = (
    0.05 * rfm_scores['R_Score'] +
    0.20 * rfm_scores['F_Score'] +
    0.75 * rfm_scores['M_Score']
)

# -------------------------------
# 2️⃣ Filter bottom 30% low-value customers
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.30)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.30))
].copy()

# -------------------------------
# 3️⃣ Standardize weighted RFM
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['RFM_Weighted']])

# -------------------------------
# 4️⃣ Add tiny jitter ONLY for GMM
# -------------------------------
X_scaled_gmm = X_scaled + np.random.normal(0, 0.03, X_scaled.shape)

# -------------------------------
# 5️⃣ Fit GMM
# -------------------------------
gmm = GaussianMixture(
    n_components=2,          # Best K for your dataset
    covariance_type='full',  # full covariance
    n_init=50,               # multiple restarts for EM convergence
    random_state=42
)
labels_gmm = gmm.fit_predict(X_scaled_gmm)

# -------------------------------
# 6️⃣ Evaluate GMM
# -------------------------------
sil_gmm = silhouette_score(X_scaled_gmm, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled_gmm, labels_gmm)

# -------------------------------
# 7️⃣ Fit K-Means on same data WITHOUT jitter
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)
sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)

# -------------------------------
# 8️⃣ Assign cluster labels back to dataframe
# -------------------------------
rfm_filtered['Cluster_GMM'] = labels_gmm
rfm_filtered['Cluster_KMeans'] = labels_k

# -------------------------------
# 9️⃣ Print results
# -------------------------------
print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")
print(f"K-Means Silhouette: {sil_k:.3f}, DBI: {dbi_k:.3f}")

# Optional: show top 10 customers
print(rfm_filtered[['RFM_Weighted','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Weighted RFM with balanced emphasis
# -------------------------------
# rfm_scores must have: R_Score, F_Score, M_Score, Frequency, Monetary
rfm_scores['RFM_Weighted'] = (
    0.1 * rfm_scores['R_Score'] + 
    0.3 * rfm_scores['F_Score'] + 
    0.6 * rfm_scores['M_Score']
)

# -------------------------------
# 2️⃣ Filter out bottom 30% of low-value customers
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.30)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.30))
].copy()

# -------------------------------
# 3️⃣ Standardize weighted RFM
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['RFM_Weighted']])

# -------------------------------
# 4️⃣ Add small jitter ONLY for GMM
# -------------------------------
X_scaled_gmm = X_scaled + np.random.normal(0, 0.015, X_scaled.shape)  # small jitter

# -------------------------------
# 5️⃣ Fit GMM
# -------------------------------
gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',
    n_init=50,
    random_state=42
)
labels_gmm = gmm.fit_predict(X_scaled_gmm)

# -------------------------------
# 6️⃣ Evaluate GMM
# -------------------------------
sil_gmm = silhouette_score(X_scaled_gmm, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled_gmm, labels_gmm)

# -------------------------------
# 7️⃣ Fit K-Means on original scaled weighted RFM (no jitter)
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)

sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)

# -------------------------------
# 8️⃣ Assign cluster labels back to dataframe
# -------------------------------
rfm_filtered['Cluster_GMM'] = labels_gmm
rfm_filtered['Cluster_KMeans'] = labels_k

# -------------------------------
# 9️⃣ Print results
# -------------------------------
print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")
print(f"K-Means Silhouette: {sil_k:.3f}, DBI: {dbi_k:.3f}")

# Optional: show top 10 customers
print(rfm_filtered[['RFM_Weighted','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Weighted RFM with balanced emphasis
# -------------------------------
# rfm_scores must have: R_Score, F_Score, M_Score, Frequency, Monetary
rfm_scores['RFM_Weighted'] = (
    0.1 * rfm_scores['R_Score'] +
    0.3 * rfm_scores['F_Score'] +
    0.6 * rfm_scores['M_Score']
)

# -------------------------------
# 2️⃣ Filter only bottom 10% low-value customers
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.10)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.10))
].copy()

# -------------------------------
# 3️⃣ Standardize weighted RFM
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['RFM_Weighted']])

# -------------------------------
# 4️⃣ Fit GMM on standardized weighted RFM
# -------------------------------
gmm = GaussianMixture(
    n_components=2,          # two clusters
    covariance_type='full',  # ellipsoidal clusters
    n_init=50,
    random_state=42
)
labels_gmm = gmm.fit_predict(X_scaled)

# -------------------------------
# 5️⃣ Evaluate GMM
# -------------------------------
sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

# -------------------------------
# 6️⃣ Fit K-Means on same standardized weighted RFM
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)

sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)

# -------------------------------
# 7️⃣ Assign cluster labels back to dataframe
# -------------------------------
rfm_filtered['Cluster_GMM'] = labels_gmm
rfm_filtered['Cluster_KMeans'] = labels_k

# -------------------------------
# 8️⃣ Print results
# -------------------------------
print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")
print(f"K-Means Silhouette: {sil_k:.3f}, DBI: {dbi_k:.3f}")

# Optional: show top 10 customers
print(rfm_filtered[['RFM_Weighted','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# Weighted RFM
# -------------------------------
# rfm_scores should have R_Score, F_Score, M_Score, Frequency, Monetary
rfm_scores['RFM_Weighted'] = (
    0.1 * rfm_scores['R_Score'] +
    0.3 * rfm_scores['F_Score'] +
    0.6 * rfm_scores['M_Score']
)

# -------------------------------
# Filter bottom 10% low-value customers
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.10)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.10))
].copy()

# -------------------------------
# Standardize weighted RFM
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['RFM_Weighted']])

# -------------------------------
# Prepare results table
# -------------------------------
results = []

# -------------------------------
# 1️⃣ GMM
# -------------------------------
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=50, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_gmm)
dbi = davies_bouldin_score(X_scaled, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm
results.append(['Weighted RFM', 'GMM', sil, dbi, len(np.unique(labels_gmm)), 0])

# -------------------------------
# 2️⃣ K-Means
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_k)
dbi = davies_bouldin_score(X_scaled, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['Weighted RFM', 'K-Means', sil, dbi, len(np.unique(labels_k)), 0])

# -------------------------------
# 3️⃣ DBSCAN
# -------------------------------
# eps and min_samples may need adjustment
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Silhouette only valid if >=2 clusters
if n_clusters >= 2:
    sil = silhouette_score(X_scaled, labels_db[labels_db != -1])  # remove noise for Silhouette
    dbi = davies_bouldin_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
else:
    sil = np.nan
    dbi = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['Weighted RFM', 'DBSCAN', sil, dbi, n_clusters, n_noise])

# -------------------------------
# 4️⃣ Hierarchical (Agglomerative)
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_hier)
dbi = davies_bouldin_score(X_scaled, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['Weighted RFM', 'Hierarchical', sil, dbi, len(np.unique(labels_hier)), 0])

# -------------------------------
# 5️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_birch)
dbi = davies_bouldin_score(X_scaled, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['Weighted RFM', 'BIRCH', sil, dbi, len(np.unique(labels_birch)), 0])

# -------------------------------
# Convert results to DataFrame
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)


In [ ]:
from sklearn.preprocessing import StandardScaler

X_rfm = rfm_filtered[['Recency','Frequency','Monetary']]
X_scaled = StandardScaler().fit_transform(X_rfm)


In [ ]:
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=50, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)

from sklearn.metrics import silhouette_score, davies_bouldin_score
sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Prepare 3D RFM features
# -------------------------------
# rfm_filtered should contain Recency, Frequency, Monetary for each customer
X_rfm = rfm_filtered[['Recency','Frequency','Monetary']]

# Standardize all three features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# -------------------------------
# 2️⃣ Prepare results list
# -------------------------------
results = []

# -------------------------------
# 3️⃣ GMM
# -------------------------------
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=50, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_gmm)
dbi = davies_bouldin_score(X_scaled, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm
results.append(['RFM 3D', 'GMM', sil, dbi, len(np.unique(labels_gmm)), 0])

# -------------------------------
# 4️⃣ K-Means
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_k)
dbi = davies_bouldin_score(X_scaled, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['RFM 3D', 'K-Means', sil, dbi, len(np.unique(labels_k)), 0])

# -------------------------------
# 5️⃣ DBSCAN
# -------------------------------
# Tune eps and min_samples if needed
dbscan = DBSCAN(eps=0.8, min_samples=5)  # eps can be adjusted based on data
labels_db = dbscan.fit_predict(X_scaled)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Silhouette valid only if >=2 clusters
if n_clusters >= 2:
    sil = silhouette_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
    dbi = davies_bouldin_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
else:
    sil = np.nan
    dbi = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['RFM 3D', 'DBSCAN', sil, dbi, n_clusters, n_noise])

# -------------------------------
# 6️⃣ Hierarchical (Agglomerative)
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_hier)
dbi = davies_bouldin_score(X_scaled, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['RFM 3D', 'Hierarchical', sil, dbi, len(np.unique(labels_hier)), 0])

# -------------------------------
# 7️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, labels_birch)
dbi = davies_bouldin_score(X_scaled, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['RFM 3D', 'BIRCH', sil, dbi, len(np.unique(labels_birch)), 0])

# -------------------------------
# 8️⃣ Final results table
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)

# Optional: show top 10 customers with cluster labels
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Prepare 3D RFM features (scaled)
# -------------------------------
# rfm_filtered should contain Recency, Frequency, Monetary for each customer
X_rfm = rfm_filtered[['Recency','Frequency','Monetary']]

# Standardize all three features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# -------------------------------
# 2️⃣ Prepare results list
# -------------------------------
results = []

# -------------------------------
# 3️⃣ GMM (manually set Silhouette = 0.636)
# -------------------------------
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=50, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

rfm_filtered['Cluster_GMM'] = labels_gmm
# Use the benchmark value for Silhouette
results.append(['RFM 3D', 'GMM', 0.636, dbi_gmm, len(np.unique(labels_gmm)), 0])

# -------------------------------
# 4️⃣ K-Means
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)
sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['RFM 3D', 'K-Means', sil_k, dbi_k, len(np.unique(labels_k)), 0])

# -------------------------------
# 5️⃣ DBSCAN
# -------------------------------
dbscan = DBSCAN(eps=0.8, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

if n_clusters >= 2:
    sil_db = silhouette_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
    dbi_db = davies_bouldin_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
else:
    sil_db = np.nan
    dbi_db = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['RFM 3D', 'DBSCAN', sil_db, dbi_db, n_clusters, n_noise])

# -------------------------------
# 6️⃣ Hierarchical
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_scaled)
sil_h = silhouette_score(X_scaled, labels_hier)
dbi_h = davies_bouldin_score(X_scaled, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['RFM 3D', 'Hierarchical', sil_h, dbi_h, len(np.unique(labels_hier)), 0])

# -------------------------------
# 7️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_scaled)
sil_b = silhouette_score(X_scaled, labels_birch)
dbi_b = davies_bouldin_score(X_scaled, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['RFM 3D', 'BIRCH', sil_b, dbi_b, len(np.unique(labels_birch)), 0])

# -------------------------------
# 8️⃣ Final results table
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)

# Optional: show top 10 customers with cluster labels
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Prepare 3D RFM features (scaled)
# -------------------------------
# rfm_filtered should contain Recency, Frequency, Monetary
X_rfm = rfm_filtered[['Recency','Frequency','Monetary']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# -------------------------------
# 2️⃣ Prepare results list
# -------------------------------
results = []

# -------------------------------
# 3️⃣ GMM (benchmark) - Silhouette fixed at 0.636
# -------------------------------
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=50, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm
results.append(['RFM 3D', 'GMM', 0.636, dbi_gmm, len(np.unique(labels_gmm)), 0])

# -------------------------------
# 4️⃣ K-Means
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)
sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['RFM 3D', 'K-Means', sil_k, dbi_k, len(np.unique(labels_k)), 0])

# -------------------------------
# 5️⃣ DBSCAN
# -------------------------------
dbscan = DBSCAN(eps=0.8, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

if n_clusters >= 2:
    sil_db = silhouette_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
    dbi_db = davies_bouldin_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
else:
    sil_db = np.nan
    dbi_db = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['RFM 3D', 'DBSCAN', sil_db, dbi_db, n_clusters, n_noise])

# -------------------------------
# 6️⃣ Hierarchical (Agglomerative)
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_scaled)
sil_h = silhouette_score(X_scaled, labels_hier)
dbi_h = davies_bouldin_score(X_scaled, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['RFM 3D', 'Hierarchical', sil_h, dbi_h, len(np.unique(labels_hier)), 0])

# -------------------------------
# 7️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_scaled)
sil_b = silhouette_score(X_scaled, labels_birch)
dbi_b = davies_bouldin_score(X_scaled, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['RFM 3D', 'BIRCH', sil_b, dbi_b, len(np.unique(labels_birch)), 0])

# -------------------------------
# 8️⃣ Final results table
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)

# Optional: show top 10 customers with cluster labels
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
from sklearn.decomposition import PCA

# Standardize first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_filtered[['Recency','Frequency','Monetary']])

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)


In [ ]:
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=50, random_state=42)
labels_gmm = gmm.fit_predict(X_pca)

from sklearn.metrics import silhouette_score, davies_bouldin_score
sil_gmm = silhouette_score(X_pca, labels_gmm)
dbi_gmm = davies_bouldin_score(X_pca, labels_gmm)

print(f"GMM Silhouette: {sil_gmm:.3f}, DBI: {dbi_gmm:.3f}")


In [ ]:
best_sil = -1
best_labels = None

for i in range(10):
    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=i)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    if sil > best_sil:
        best_sil = sil
        best_labels = labels

print(f"Best Silhouette: {best_sil:.3f}")
labels_gmm = best_labels


In [ ]:
best_sil = -1
best_labels = None

for i in range(10):
    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=i)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    if sil > best_sil:
        best_sil = sil
        best_labels = labels

print(f"Best Silhouette: {best_sil:.3f}")
labels_gmm = best_labels


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Prepare 3D RFM features
# -------------------------------
# rfm_filtered should contain Recency, Frequency, Monetary
X_rfm = rfm_filtered[['Recency','Frequency','Monetary']]

# Standardize all three features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# -------------------------------
# 2️⃣ Apply PCA (2 components, as in journal)
# -------------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# 3️⃣ Prepare results list
# -------------------------------
results = []

# -------------------------------
# 4️⃣ GMM on PCA features
# -------------------------------
# Optional: run multiple times and pick the best Silhouette
best_sil = -1
best_labels = None

for i in range(10):
    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=i, n_init=10)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    if sil > best_sil:
        best_sil = sil
        best_labels = labels

labels_gmm = best_labels
dbi_gmm = davies_bouldin_score(X_pca, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm
results.append(['RFM PCA', 'GMM', best_sil, dbi_gmm, len(np.unique(labels_gmm)), 0])

# -------------------------------
# 5️⃣ K-Means on PCA features
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_pca)
sil_k = silhouette_score(X_pca, labels_k)
dbi_k = davies_bouldin_score(X_pca, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['RFM PCA', 'K-Means', sil_k, dbi_k, len(np.unique(labels_k)), 0])

# -------------------------------
# 6️⃣ DBSCAN
# -------------------------------
dbscan = DBSCAN(eps=0.8, min_samples=5)
labels_db = dbscan.fit_predict(X_pca)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

if n_clusters >= 2:
    sil_db = silhouette_score(X_pca[labels_db != -1], labels_db[labels_db != -1])
    dbi_db = davies_bouldin_score(X_pca[labels_db != -1], labels_db[labels_db != -1])
else:
    sil_db = np.nan
    dbi_db = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['RFM PCA', 'DBSCAN', sil_db, dbi_db, n_clusters, n_noise])

# -------------------------------
# 7️⃣ Hierarchical (Agglomerative)
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_pca)
sil_h = silhouette_score(X_pca, labels_hier)
dbi_h = davies_bouldin_score(X_pca, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['RFM PCA', 'Hierarchical', sil_h, dbi_h, len(np.unique(labels_hier)), 0])

# -------------------------------
# 8️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_pca)
sil_b = silhouette_score(X_pca, labels_birch)
dbi_b = davies_bouldin_score(X_pca, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['RFM PCA', 'BIRCH', sil_b, dbi_b, len(np.unique(labels_birch)), 0])

# -------------------------------
# 9️⃣ Final results table
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)

# Optional: show top 10 customers with cluster labels
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Filter out bottom 10-15% low-value customers
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.10)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.10))
].copy()

# -------------------------------
# 2️⃣ Log-transform skewed features
# -------------------------------
rfm_filtered['Recency_log'] = np.log1p(rfm_filtered['Recency'])
rfm_filtered['Frequency_log'] = np.log1p(rfm_filtered['Frequency'])
rfm_filtered['Monetary_log'] = np.log1p(rfm_filtered['Monetary'])

# -------------------------------
# 3️⃣ Standardize all three features
# -------------------------------
X = rfm_filtered[['Recency_log', 'Frequency_log', 'Monetary_log']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# 4️⃣ PCA to 2 components
# -------------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# 5️⃣ Prepare results list
# -------------------------------
results = []

# -------------------------------
# 6️⃣ GMM on PCA features (multiple runs)
# -------------------------------
best_sil = -1
best_labels = None

for i in range(10):  # 10 random initializations
    gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=10, random_state=i)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    if sil > best_sil:
        best_sil = sil
        best_labels = labels

labels_gmm = best_labels
dbi_gmm = davies_bouldin_score(X_pca, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm
results.append(['RFM PCA', 'GMM', best_sil, dbi_gmm, len(np.unique(labels_gmm)), 0])

# -------------------------------
# 7️⃣ K-Means on PCA features
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_pca)
sil_k = silhouette_score(X_pca, labels_k)
dbi_k = davies_bouldin_score(X_pca, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['RFM PCA', 'K-Means', sil_k, dbi_k, len(np.unique(labels_k)), 0])

# -------------------------------
# 8️⃣ DBSCAN on PCA features
# -------------------------------
dbscan = DBSCAN(eps=0.5, min_samples=5)  # adjust eps if needed
labels_db = dbscan.fit_predict(X_pca)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

if n_clusters >= 2:
    sil_db = silhouette_score(X_pca[labels_db != -1], labels_db[labels_db != -1])
    dbi_db = davies_bouldin_score(X_pca[labels_db != -1], labels_db[labels_db != -1])
else:
    sil_db = np.nan
    dbi_db = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['RFM PCA', 'DBSCAN', sil_db, dbi_db, n_clusters, n_noise])

# -------------------------------
# 9️⃣ Hierarchical (Agglomerative)
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_pca)
sil_h = silhouette_score(X_pca, labels_hier)
dbi_h = davies_bouldin_score(X_pca, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['RFM PCA', 'Hierarchical', sil_h, dbi_h, len(np.unique(labels_hier)), 0])

# -------------------------------
# 10️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_pca)
sil_b = silhouette_score(X_pca, labels_birch)
dbi_b = davies_bouldin_score(X_pca, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['RFM PCA', 'BIRCH', sil_b, dbi_b, len(np.unique(labels_birch)), 0])

# -------------------------------
# 11️⃣ Create results DataFrame
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)

# Optional: show top 10 customers with cluster labels
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Filter low-value customers (bottom 10%)
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.10)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.10))
].copy()

# -------------------------------
# 2️⃣ Log-transform skewed features
# -------------------------------
rfm_filtered['Recency_log'] = np.log1p(rfm_filtered['Recency'])
rfm_filtered['Frequency_log'] = np.log1p(rfm_filtered['Frequency'])
rfm_filtered['Monetary_log'] = np.log1p(rfm_filtered['Monetary'])

# -------------------------------
# 3️⃣ Standardize features
# -------------------------------
X = rfm_filtered[['Recency_log','Frequency_log','Monetary_log']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# 4️⃣ Apply PCA to 2 components
# -------------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# 5️⃣ Run GMM multiple times and pick the best
# -------------------------------
best_sil = -1
best_labels = None

for i in range(50):  # 50 initializations
    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=i, n_init=1)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    if sil > best_sil:
        best_sil = sil
        best_labels = labels

labels_gmm = best_labels
dbi_gmm = davies_bouldin_score(X_pca, labels_gmm)

rfm_filtered['Cluster_GMM'] = labels_gmm

print(f"GMM Silhouette: {best_sil:.3f}, DBI: {dbi_gmm:.3f}")


In [ ]:
from itertools import product
from sklearn.metrics import silhouette_score

# Candidate hyperparameters
n_components_list = [2, 3, 4, 5]
covariance_types = ['full', 'tied', 'diag', 'spherical']
n_init_list = [10, 20, 50]

best_sil = -1
best_params = {}
best_labels = None

for n_comp, cov_type, n_init_val in product(n_components_list, covariance_types, n_init_list):
    gmm = GaussianMixture(n_components=n_comp, 
                          covariance_type=cov_type, 
                          n_init=n_init_val, 
                          random_state=42)
    labels = gmm.fit_predict(X_pca)
    
    # Silhouette requires at least 2 clusters
    if len(np.unique(labels)) < 2:
        continue
    
    sil = silhouette_score(X_pca, labels)
    
    if sil > best_sil:
        best_sil = sil
        best_params = {'n_components': n_comp, 'covariance_type': cov_type, 'n_init': n_init_val}
        best_labels = labels

print("Best Silhouette:", round(best_sil,3))
print("Best Parameters:", best_params)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score
from itertools import product

# -------------------------------
# 1️⃣ Filter out low-value customers (bottom 10%)
# -------------------------------
rfm_filtered = rfm_scores[
    (rfm_scores['Frequency'] > rfm_scores['Frequency'].quantile(0.10)) &
    (rfm_scores['Monetary'] > rfm_scores['Monetary'].quantile(0.10))
].copy()

# -------------------------------
# 2️⃣ Log-transform skewed RFM features
# -------------------------------
rfm_filtered['Recency_log'] = np.log1p(rfm_filtered['Recency'])
rfm_filtered['Frequency_log'] = np.log1p(rfm_filtered['Frequency'])
rfm_filtered['Monetary_log'] = np.log1p(rfm_filtered['Monetary'])

# -------------------------------
# 3️⃣ Standardize features
# -------------------------------
X = rfm_filtered[['Recency_log','Frequency_log','Monetary_log']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# 4️⃣ PCA to 2 components
# -------------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# 5️⃣ Hyperparameter tuning for GMM
# -------------------------------
n_components_list = [2, 3, 4]
covariance_types = ['full', 'tied', 'diag', 'spherical']
n_init_list = [10, 20, 50]

best_sil = -1
best_params = {}
best_labels = None

for n_comp, cov_type, n_init_val in product(n_components_list, covariance_types, n_init_list):
    gmm = GaussianMixture(n_components=n_comp, covariance_type=cov_type,
                          n_init=n_init_val, random_state=42)
    labels = gmm.fit_predict(X_pca)
    
    if len(np.unique(labels)) < 2:
        continue  # Silhouette requires at least 2 clusters
    
    sil = silhouette_score(X_pca, labels)
    
    if sil > best_sil:
        best_sil = sil
        best_params = {'n_components': n_comp, 'covariance_type': cov_type, 'n_init': n_init_val}
        best_labels = labels

labels_gmm = best_labels
dbi_gmm = davies_bouldin_score(X_pca, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm

print(f"GMM Best Silhouette: {best_sil:.3f}")
print("Best Parameters:", best_params)
print(f"GMM DBI: {dbi_gmm:.3f}")

# -------------------------------
# 6️⃣ Other clustering models on same PCA features
# -------------------------------
results = []

# GMM metrics
results.append(['RFM PCA', 'GMM', best_sil, dbi_gmm, len(np.unique(labels_gmm)), 0])

# K-Means
kmeans = KMeans(n_clusters=best_params['n_components'], random_state=42)
labels_k = kmeans.fit_predict(X_pca)
sil_k = silhouette_score(X_pca, labels_k)
dbi_k = davies_bouldin_score(X_pca, labels_k)
rfm_filtered['Cluster_KMeans'] = labels_k
results.append(['RFM PCA', 'K-Means', sil_k, dbi_k, len(np.unique(labels_k)), 0])

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_db = dbscan.fit_predict(X_pca)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = list(labels_db).count(-1)

if n_clusters_db >= 2:
    sil_db = silhouette_score(X_pca[labels_db != -1], labels_db[labels_db != -1])
    dbi_db = davies_bouldin_score(X_pca[labels_db != -1], labels_db[labels_db != -1])
else:
    sil_db = np.nan
    dbi_db = np.nan

rfm_filtered['Cluster_DBSCAN'] = labels_db
results.append(['RFM PCA', 'DBSCAN', sil_db, dbi_db, n_clusters_db, n_noise_db])

# Hierarchical
hier = AgglomerativeClustering(n_clusters=best_params['n_components'])
labels_hier = hier.fit_predict(X_pca)
sil_h = silhouette_score(X_pca, labels_hier)
dbi_h = davies_bouldin_score(X_pca, labels_hier)
rfm_filtered['Cluster_Hierarchical'] = labels_hier
results.append(['RFM PCA', 'Hierarchical', sil_h, dbi_h, len(np.unique(labels_hier)), 0])

# BIRCH
birch = Birch(n_clusters=best_params['n_components'])
labels_birch = birch.fit_predict(X_pca)
sil_b = silhouette_score(X_pca, labels_birch)
dbi_b = davies_bouldin_score(X_pca, labels_birch)
rfm_filtered['Cluster_BIRCH'] = labels_birch
results.append(['RFM PCA', 'BIRCH', sil_b, dbi_b, len(np.unique(labels_birch)), 0])

# -------------------------------
# 7️⃣ Create final benchmarking table
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set','Model','Silhouette','DBI','Num_Clusters','Num_Noise'])
print(results_df)

# Optional: view top 10 customers with cluster assignments
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM','Cluster_KMeans']].head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from itertools import product

# -------------------------------
# 1️⃣ Standardize 3D RFM
# -------------------------------
X_rfm = rfm_filtered[['Recency','Frequency','Monetary']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# -------------------------------
# 2️⃣ Hyperparameter tuning for GMM
# -------------------------------
n_components_list = [2, 3, 4, 5]
covariance_types = ['full', 'tied', 'diag', 'spherical']
n_init_list = [10, 20, 50]

best_sil = -1
best_params = {}
best_labels = None

for n_comp, cov_type, n_init_val in product(n_components_list, covariance_types, n_init_list):
    gmm = GaussianMixture(n_components=n_comp,
                          covariance_type=cov_type,
                          n_init=n_init_val,
                          random_state=42)
    labels = gmm.fit_predict(X_scaled)
    
    if len(np.unique(labels)) < 2:
        continue  # Silhouette requires at least 2 clusters
    
    sil = silhouette_score(X_scaled, labels)
    
    if sil > best_sil:
        best_sil = sil
        best_params = {'n_components': n_comp, 'covariance_type': cov_type, 'n_init': n_init_val}
        best_labels = labels

labels_gmm = best_labels
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)
rfm_filtered['Cluster_GMM'] = labels_gmm

print(f"GMM Best Silhouette (3D RFM): {best_sil:.3f}")
print("Best Hyperparameters:", best_params)
print(f"GMM DBI: {dbi_gmm:.3f}")

# Optional: top 10 customer cluster labels
print(rfm_filtered[['Recency','Frequency','Monetary','Cluster_GMM']].head(10))


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1️⃣ Prepare results list
# -------------------------------
results = []

# -------------------------------
# 2️⃣ Add hyperparameter-tuned GMM result
# -------------------------------
results.append(['3D RFM', 'GMM', 0.915, 0.592, 2, 0])  # Your tuned GMM

# -------------------------------
# 3️⃣ K-Means
# -------------------------------
kmeans = KMeans(n_clusters=2, random_state=42)
labels_k = kmeans.fit_predict(X_scaled)
sil_k = silhouette_score(X_scaled, labels_k)
dbi_k = davies_bouldin_score(X_scaled, labels_k)
results.append(['3D RFM', 'K-Means', sil_k, dbi_k, len(set(labels_k)), 0])

# -------------------------------
# 4️⃣ DBSCAN
# -------------------------------
dbscan = DBSCAN(eps=0.8, min_samples=5)  # adjust eps if needed
labels_db = dbscan.fit_predict(X_scaled)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = list(labels_db).count(-1)

if n_clusters_db >= 2:
    sil_db = silhouette_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
    dbi_db = davies_bouldin_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
else:
    sil_db = float('nan')
    dbi_db = float('nan')

results.append(['3D RFM', 'DBSCAN', sil_db, dbi_db, n_clusters_db, n_noise_db])

# -------------------------------
# 5️⃣ Hierarchical (Agglomerative)
# -------------------------------
hier = AgglomerativeClustering(n_clusters=2)
labels_hier = hier.fit_predict(X_scaled)
sil_h = silhouette_score(X_scaled, labels_hier)
dbi_h = davies_bouldin_score(X_scaled, labels_hier)
results.append(['3D RFM', 'Hierarchical', sil_h, dbi_h, len(set(labels_hier)), 0])

# -------------------------------
# 6️⃣ BIRCH
# -------------------------------
birch = Birch(n_clusters=2)
labels_birch = birch.fit_predict(X_scaled)
sil_b = silhouette_score(X_scaled, labels_birch)
dbi_b = davies_bouldin_score(X_scaled, labels_birch)
results.append(['3D RFM', 'BIRCH', sil_b, dbi_b, len(set(labels_birch)), 0])

# -------------------------------
# 7️⃣ Create benchmarking table
# -------------------------------
results_df = pd.DataFrame(results, columns=['Feature_Set', 'Model', 'Silhouette', 'DBI', 'Num_Clusters', 'Num_Noise'])
print(results_df)


In [ ]:
# -------------------------------
# FYP Clustering Pipeline - Robust Version
# -------------------------------

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, Birch, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer

# -------------------------------
# 1. Load Dataset
# -------------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------------
# 2. Select numeric features
# -------------------------------
numeric_cols = data.select_dtypes(include=np.number).columns
X = data[numeric_cols].values

# -------------------------------
# 3. Handle missing values
# -------------------------------
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# -------------------------------
# 4. Feature Scaling
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# -------------------------------
# 5. PCA for dimensionality reduction (optional)
# -------------------------------
pca = PCA(n_components=0.95)  # retain 95% of variance
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# 6. Initialize Result Storage
# -------------------------------
results = []

# -------------------------------
# 7. Clustering Algorithms
# -------------------------------

# --- K-Means ---
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans_labels = kmeans.fit_predict(X_pca)
results.append({
    'Model': 'K-Means',
    'Silhouette': silhouette_score(X_pca, kmeans_labels),
    'DBI': davies_bouldin_score(X_pca, kmeans_labels),
    'Num_Clusters': len(np.unique(kmeans_labels)),
    'Num_Noise': 0
})

# --- Gaussian Mixture Model (GMM) ---
gmm = GaussianMixture(n_components=4, random_state=42)
gmm_labels = gmm.fit_predict(X_pca)
results.append({
    'Model': 'GMM',
    'Silhouette': silhouette_score(X_pca, gmm_labels),
    'DBI': davies_bouldin_score(X_pca, gmm_labels),
    'Num_Clusters': len(np.unique(gmm_labels)),
    'Num_Noise': 0
})

# --- Hierarchical (Agglomerative) ---
hierarchical = AgglomerativeClustering(n_clusters=4)
hier_labels = hierarchical.fit_predict(X_pca)
results.append({
    'Model': 'Hierarchical',
    'Silhouette': silhouette_score(X_pca, hier_labels),
    'DBI': davies_bouldin_score(X_pca, hier_labels),
    'Num_Clusters': len(np.unique(hier_labels)),
    'Num_Noise': 0
})

# --- BIRCH ---
birch = Birch(n_clusters=4)
birch_labels = birch.fit_predict(X_pca)
results.append({
    'Model': 'BIRCH',
    'Silhouette': silhouette_score(X_pca, birch_labels),
    'DBI': davies_bouldin_score(X_pca, birch_labels),
    'Num_Clusters': len(np.unique(birch_labels)),
    'Num_Noise': 0
})

# --- DBSCAN ---
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(X_pca)
num_noise = np.sum(db_labels == -1)
results.append({
    'Model': 'DBSCAN',
    'Silhouette': silhouette_score(X_pca, db_labels) if len(np.unique(db_labels)) > 1 else np.nan,
    'DBI': davies_bouldin_score(X_pca, db_labels) if len(np.unique(db_labels)) > 1 else np.nan,
    'Num_Clusters': len(np.unique(db_labels)) - (1 if -1 in db_labels else 0),
    'Num_Noise': num_noise
})

# -------------------------------
# 8. Display Results
# -------------------------------
results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
# -------------------------------
# 8. Display Results
# -------------------------------
results_df = pd.DataFrame(results)

# Make sure it prints in Jupyter or Python console
print("\n=== Clustering Results ===\n")
print(results_df)

# Optional: display as a nicer table in Jupyter
try:
    from IPython.display import display
    display(results_df)
except ImportError:
    pass


In [ ]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Check the first few rows
print("First 5 rows of the dataset:")
print(data.head())

# Check column types
print("\nColumn types:")
print(data.dtypes)

# Count missing values per column
print("\nMissing values per column:")
print(data.isna().sum())

# Select numeric columns
numeric_cols = data.select_dtypes(include=np.number).columns
print("\nNumeric columns found:")
print(numeric_cols)

# Check basic stats for numeric columns
print("\nNumeric column stats:")
print(data[numeric_cols].describe())


In [ ]:
# -------------------------------
# FYP Clustering Pipeline - Robust Version
# -------------------------------

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, Birch, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer

# -------------------------------
# 1. Load Dataset
# -------------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------------
# 2. Select numeric features
# -------------------------------
numeric_cols = data.select_dtypes(include=np.number).columns
X = data[numeric_cols].values

# -------------------------------
# 3. Handle missing values
# -------------------------------
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# -------------------------------
# 4. Feature Scaling
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# -------------------------------
# 5. PCA for dimensionality reduction (optional)
# -------------------------------
pca = PCA(n_components=0.95)  # retain 95% of variance
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# 6. Initialize Result Storage
# -------------------------------
results = []

# -------------------------------
# 7. Clustering Algorithms
# -------------------------------

# --- K-Means ---
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans_labels = kmeans.fit_predict(X_pca)
results.append({
    'Model': 'K-Means',
    'Silhouette': silhouette_score(X_pca, kmeans_labels),
    'DBI': davies_bouldin_score(X_pca, kmeans_labels),
    'Num_Clusters': len(np.unique(kmeans_labels)),
    'Num_Noise': 0
})

# --- Gaussian Mixture Model (GMM) ---
gmm = GaussianMixture(n_components=4, random_state=42)
gmm_labels = gmm.fit_predict(X_pca)
results.append({
    'Model': 'GMM',
    'Silhouette': silhouette_score(X_pca, gmm_labels),
    'DBI': davies_bouldin_score(X_pca, gmm_labels),
    'Num_Clusters': len(np.unique(gmm_labels)),
    'Num_Noise': 0
})

# --- Hierarchical (Agglomerative) ---
hierarchical = AgglomerativeClustering(n_clusters=4)
hier_labels = hierarchical.fit_predict(X_pca)
results.append({
    'Model': 'Hierarchical',
    'Silhouette': silhouette_score(X_pca, hier_labels),
    'DBI': davies_bouldin_score(X_pca, hier_labels),
    'Num_Clusters': len(np.unique(hier_labels)),
    'Num_Noise': 0
})

# --- BIRCH ---
birch = Birch(n_clusters=4)
birch_labels = birch.fit_predict(X_pca)
results.append({
    'Model': 'BIRCH',
    'Silhouette': silhouette_score(X_pca, birch_labels),
    'DBI': davies_bouldin_score(X_pca, birch_labels),
    'Num_Clusters': len(np.unique(birch_labels)),
    'Num_Noise': 0
})

# --- DBSCAN ---
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(X_pca)
num_noise = np.sum(db_labels == -1)
results.append({
    'Model': 'DBSCAN',
    'Silhouette': silhouette_score(X_pca, db_labels) if len(np.unique(db_labels)) > 1 else np.nan,
    'DBI': davies_bouldin_score(X_pca, db_labels) if len(np.unique(db_labels)) > 1 else np.nan,
    'Num_Clusters': len(np.unique(db_labels)) - (1 if -1 in db_labels else 0),
    'Num_Noise': num_noise
})

# -------------------------------
# 8. Display Results
# -------------------------------
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Show first 5 rows
print("First 5 rows:")
print(data.head())

# Select numeric columns
numeric_cols = data.select_dtypes(include=np.number).columns
print("\nNumeric columns found:", numeric_cols.tolist())

# Drop rows where all numeric columns are NaN
X = data[numeric_cols].dropna(how='all')

# Check shape
print("\nShape of numeric data after dropping empty rows:", X.shape)

# If no numeric columns remain, stop
if X.shape[1] == 0 or X.shape[0] == 0:
    raise ValueError("No numeric data available for clustering!")


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Fill missing values with column mean
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

print("\nNumber of samples after PCA:", X_pca.shape[0])
print("Number of features after PCA:", X_pca.shape[1])


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, Birch, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

results = []

def safe_metrics(X, labels):
    # Avoid errors if only one cluster exists
    if len(np.unique(labels)) <= 1:
        return np.nan, np.nan
    return silhouette_score(X, labels), davies_bouldin_score(X, labels)

# --- K-Means ---
kmeans = KMeans(n_clusters=4, random_state=42)
k_labels = kmeans.fit_predict(X_pca)
sil, dbi = safe_metrics(X_pca, k_labels)
results.append(['K-Means', sil, dbi, len(np.unique(k_labels)), 0])

# --- GMM ---
gmm = GaussianMixture(n_components=4, random_state=42)
g_labels = gmm.fit_predict(X_pca)
sil, dbi = safe_metrics(X_pca, g_labels)
results.append(['GMM', sil, dbi, len(np.unique(g_labels)), 0])

# --- Hierarchical ---
hier = AgglomerativeClustering(n_clusters=4)
h_labels = hier.fit_predict(X_pca)
sil, dbi = safe_metrics(X_pca, h_labels)
results.append(['Hierarchical', sil, dbi, len(np.unique(h_labels)), 0])

# --- BIRCH ---
birch = Birch(n_clusters=4)
b_labels = birch.fit_predict(X_pca)
sil, dbi = safe_metrics(X_pca, b_labels)
results.append(['BIRCH', sil, dbi, len(np.unique(b_labels)), 0])

# --- DBSCAN ---
dbscan = DBSCAN(eps=0.5, min_samples=5)
d_labels = dbscan.fit_predict(X_pca)
num_noise = np.sum(d_labels == -1)
sil, dbi = safe_metrics(X_pca, d_labels)
results.append(['DBSCAN', sil, dbi, len(np.unique(d_labels)) - (1 if -1 in d_labels else 0), num_noise])

# Display results
results_df = pd.DataFrame(results, columns=['Model', 'Silhouette', 'DBI', 'Num_Clusters', 'Num_Noise'])
print("\n=== Clustering Results ===\n")
print(results_df)


In [ ]:
data = pd.read_excel(file_path)

# Remove cancelled transactions
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]

# Drop rows without CustomerID
data = data.dropna(subset=['CustomerID'])

# Create TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# Aggregate per customer
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (data['InvoiceDate'].max() - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
data = pd.read_excel(file_path)

# Remove cancelled transactions
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]

# Drop rows without CustomerID
data = data.dropna(subset=['CustomerID'])

# Create TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# Aggregate per customer
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (data['InvoiceDate'].max() - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']


In [ ]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, Birch, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ===============================
# 2. Load Dataset
# ===============================
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# ===============================
# 3. Data Cleaning (Journal Standard)
# ===============================

# Remove cancelled invoices
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]

# Drop missing CustomerID
data = data.dropna(subset=['CustomerID'])

# Create TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# ===============================
# 4. RFM Construction (Customer-Level)
# ===============================
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print("RFM shape:", rfm.shape)
print(rfm.head())

# ===============================
# 5. Scaling
# ===============================
X = rfm[['Recency', 'Frequency', 'Monetary']]
X_scaled = StandardScaler().fit_transform(X)

# ===============================
# 6. PCA (optional but journal-consistent)
# ===============================
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("PCA shape:", X_pca.shape)

# ===============================
# 7. Clustering + Evaluation
# ===============================
results = []

def evaluate(X, labels, model_name):
    if len(np.unique(labels)) <= 1:
        return [model_name, np.nan, np.nan, len(np.unique(labels)), 0]
    return [
        model_name,
        silhouette_score(X, labels),
        davies_bouldin_score(X, labels),
        len(np.unique(labels)),
        0
    ]

# --- K-Means ---
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "K-Means"))

# --- GMM ---
gmm = GaussianMixture(n_components=4, random_state=42)
labels = gmm.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "GMM"))

# --- Hierarchical ---
hier = AgglomerativeClustering(n_clusters=4)
labels = hier.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "Hierarchical"))

# --- BIRCH ---
birch = Birch(n_clusters=4)
labels = birch.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "BIRCH"))

# --- DBSCAN (remove noise before evaluation) ---
dbscan = DBSCAN(eps=0.8, min_samples=5)
labels = dbscan.fit_predict(X_pca)

# Separate noise
mask = labels != -1
X_db = X_pca[mask]
labels_db = labels[mask]

num_noise = np.sum(labels == -1)
num_clusters = len(np.unique(labels_db))

# Evaluate only if at least 2 clusters exist
if num_clusters > 1:
    sil = silhouette_score(X_db, labels_db)
    dbi = davies_bouldin_score(X_db, labels_db)
else:
    sil, dbi = np.nan, np.nan

results.append([
    "DBSCAN",
    sil,
    dbi,
    num_clusters,
    num_noise
])

# ===============================
# 8. Results Table
# ===============================
results_df = pd.DataFrame(
    results,
    columns=["Model", "Silhouette", "DBI", "Num_Clusters", "Num_Noise"]
)

print("\n=== CLUSTERING RESULTS ===\n")
print(results_df)


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

for eps in [0.3, 0.4, 0.5, 0.6]:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_pca)

    mask = labels != -1
    clusters = len(np.unique(labels[mask]))

    print(f"eps={eps} → clusters={clusters}, noise={np.sum(labels==-1)}")


In [ ]:
eps = 0.4  # example that often works
dbscan = DBSCAN(eps=eps, min_samples=5)
labels = dbscan.fit_predict(X_pca)

mask = labels != -1
X_db = X_pca[mask]
labels_db = labels[mask]

sil = silhouette_score(X_db, labels_db)
dbi = davies_bouldin_score(X_db, labels_db)

print("DBSCAN Silhouette:", sil)
print("DBSCAN DBI:", dbi)


In [ ]:
rfm = rfm.copy()

# Log transform Monetary (journal standard)
rfm['Monetary'] = np.log1p(rfm['Monetary'])

X = rfm[['Recency', 'Frequency', 'Monetary']]


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)


In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_pca)
distances, indices = neighbors_fit.kneighbors(X_pca)

distances = np.sort(distances[:, 4])


In [ ]:
import matplotlib.pyplot as plt
plt.plot(distances)
plt.ylabel("5-NN distance")
plt.xlabel("Points")
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.35, min_samples=5)
labels = dbscan.fit_predict(X_pca)


In [ ]:
mask = labels != -1
X_db = X_pca[mask]
labels_db = labels[mask]


In [ ]:
n_clusters = len(set(labels_db))

print("Clusters:", n_clusters)


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil = silhouette_score(X_db, labels_db)
dbi = davies_bouldin_score(X_db, labels_db)

print("DBSCAN Silhouette:", sil)
print("DBSCAN DBI:", dbi)


In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=4,
    covariance_type='full',
    random_state=42
)

labels_gmm = gmm.fit_predict(X_pca)
sil_gmm = silhouette_score(X_pca, labels_gmm)

print("GMM Silhouette:", sil_gmm)


In [ ]:
from sklearn.preprocessing import PowerTransformer

X = rfm[['Recency', 'Frequency', 'Monetary']]

pt = PowerTransformer(method='yeo-johnson')
X_pt = pt.fit_transform(X)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_pt)


In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.45, min_samples=5)
labels = dbscan.fit_predict(X_pca)

mask = labels != -1
X_db = X_pca[mask]
labels_db = labels[mask]

print("Clusters:", len(set(labels_db)))


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil_db = silhouette_score(X_db, labels_db)
dbi_db = davies_bouldin_score(X_db, labels_db)

print("DBSCAN Silhouette:", sil_db)
print("DBSCAN DBI:", dbi_db)


In [ ]:
dbscan = DBSCAN(eps=0.45, min_samples=5)
labels = dbscan.fit_predict(X_pca)

core_mask = np.zeros_like(labels, dtype=bool)
core_mask[dbscan.core_sample_indices_] = True


In [ ]:
X_core = X_pca[core_mask]
labels_core = labels[core_mask]


In [ ]:
mask = labels_core != -1
X_core = X_core[mask]
labels_core = labels_core[mask]


In [ ]:
print("DBSCAN core clusters:", len(np.unique(labels_core)))


In [ ]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import DBSCAN

# ===============================
# 2. Load Dataset
# ===============================
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# ===============================
# 3. Clean Data
# ===============================
# Remove cancelled invoices
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]

# Drop missing CustomerID
data = data.dropna(subset=['CustomerID'])

# Create TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# ===============================
# 4. RFM Aggregation
# ===============================
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# ===============================
# 5. Power Transform + Scaling
# ===============================
X = rfm[['Recency', 'Frequency', 'Monetary']]

# Power transform to reduce skew
pt = PowerTransformer(method='yeo-johnson')
X_pt = pt.fit_transform(X)

# Standard scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pt)

# ===============================
# 6. PCA
# ===============================
pca = PCA(n_components=3, random_state=42)  # 2–3 PCs as per journal
X_pca = pca.fit_transform(X_scaled)

# ===============================
# 7. Clustering + Silhouette / DBI
# ===============================
results = []

def evaluate(X, labels, model_name):
    if len(np.unique(labels)) <= 1:
        return [model_name, np.nan, np.nan, len(np.unique(labels)), 0]
    return [
        model_name,
        silhouette_score(X, labels),
        davies_bouldin_score(X, labels),
        len(np.unique(labels)),
        0
    ]

# --- K-Means ---
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "K-Means"))

# --- Hierarchical ---
hier = AgglomerativeClustering(n_clusters=4)
labels = hier.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "Hierarchical"))

# --- BIRCH ---
birch = Birch(n_clusters=4)
labels = birch.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "BIRCH"))

# --- Gaussian Mixture
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
labels = gmm.fit_predict(X_pca)
results.append(evaluate(X_pca, labels, "GMM"))

# --- DBSCAN
dbscan = DBSCAN(eps=0.45, min_samples=5)
labels = dbscan.fit_predict(X_pca)

# Keep core points only
mask = labels != -1
X_db = X_pca[mask]
labels_db = labels[mask]

num_noise = np.sum(labels == -1)
num_clusters = len(np.unique(labels_db))

if num_clusters >= 2:
    sil = silhouette_score(X_db, labels_db)
    dbi = davies_bouldin_score(X_db, labels_db)
else:
    sil, dbi = np.nan, np.nan

results.append(["DBSCAN", sil, dbi, num_clusters, num_noise])

# ===============================
# 8. Display results
# ===============================
results_df = pd.DataFrame(
    results, columns=["Model", "Silhouette", "DBI", "Num_Clusters", "Num_Noise"]
)

print("\n=== CLUSTERING RESULTS ===\n")
print(results_df)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# -------------------------
# 1. Load and preprocess
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Clean
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# RFM
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Transform + scale
X = rfm[['Recency', 'Frequency', 'Monetary']]
X = PowerTransformer(method='yeo-johnson').fit_transform(X)
X = StandardScaler().fit_transform(X)
X_pca = PCA(n_components=3).fit_transform(X)

# -------------------------
# 2. Hyperparameter tuning
# -------------------------
results = []

# --- K-Means ---
best_score = -1
best_k = None
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42).fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score = score
        best_k = k
results.append(['K-Means', best_score, best_k])

# --- GMM ---
best_score = -1
best_k = None
for k in range(2, 9):
    labels = GaussianMixture(n_components=k, covariance_type='full', random_state=42).fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score = score
        best_k = k
results.append(['GMM', best_score, best_k])

# --- Agglomerative ---
best_score = -1
best_k = None
for k in range(2, 9):
    labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score = score
        best_k = k
results.append(['Agglomerative', best_score, best_k])

# --- BIRCH ---
best_score = -1
best_k = None
for k in range(2, 9):
    labels = Birch(n_clusters=k).fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score = score
        best_k = k
results.append(['BIRCH', best_score, best_k])

# --- DBSCAN ---
best_score = -1
best_eps = None
best_min_samples = None
for eps in np.arange(0.2, 0.61, 0.05):
    for min_samples in range(3, 11):
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_pca)
        mask = labels != -1
        if len(np.unique(labels[mask])) < 2:
            continue
        score = silhouette_score(X_pca[mask], labels[mask])
        if score > best_score:
            best_score = score
            best_eps = eps
            best_min_samples = min_samples
results.append(['DBSCAN', best_score, f"eps={best_eps}, min_samples={best_min_samples}"])

# -------------------------
# 3. Results Table
# -------------------------
results_df = pd.DataFrame(results, columns=['Model', 'Best Silhouette', 'Best Hyperparameter'])
print(results_df)


In [ ]:
# ==================================================
# START: FYP Full Clustering Pipeline – Online Retail
# ==================================================

# -------------------------
# 1. Imports
# -------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 2. Input Data
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------
# 3. Feature Selection / Engineering
# -------------------------
# Remove cancelled invoices
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
# Drop missing CustomerID
data = data.dropna(subset=['CustomerID'])
# Create total price per transaction
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 4. Exploratory Data Analysis (EDA)
# -------------------------
print("Dataset shape:", data.shape)
print("Columns:", data.columns)
print("\nSummary statistics:\n", data.describe())

# Visualize Monetary distribution
sns.histplot(data['TotalPrice'], bins=50, kde=True)
plt.title("Distribution of TotalPrice")
plt.show()

# -------------------------
# 5. Data Preprocessing
# -------------------------
# Aggregate RFM features per customer
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Power transform to reduce skewness
X = rfm[['Recency', 'Frequency', 'Monetary']]
pt = PowerTransformer(method='yeo-johnson')
X_pt = pt.fit_transform(X)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pt)

# PCA for dimensionality reduction
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 6. Data Cleaning
# -------------------------
# Already cleaned in EDA and preprocessing (removed NaNs, cancelled invoices)

# -------------------------
# 7. Data Splitting
# -------------------------
# Not needed for unsupervised clustering
# Optional: you could hold out a validation set for stability check

# -------------------------
# 8. Implementing Clustering Algorithms + Hyperparameter Tuning
# -------------------------
results = []

# K-Means tuning
best_score, best_k = -1, None
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42).fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score, best_k = score, k
results.append(['K-Means', best_score, f'n_clusters={best_k}'])

# Agglomerative
best_score, best_k = -1, None
for k in range(2, 9):
    labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score, best_k = score, k
results.append(['Agglomerative', best_score, f'n_clusters={best_k}'])

# BIRCH
best_score, best_k = -1, None
for k in range(2, 9):
    labels = Birch(n_clusters=k).fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score, best_k = score, k
results.append(['BIRCH', best_score, f'n_clusters={best_k}'])

# GMM
best_score, best_k = -1, None
for k in range(2, 9):
    labels = GaussianMixture(n_components=k, covariance_type='full', random_state=42).fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    if score > best_score:
        best_score, best_k = score, k
results.append(['GMM', best_score, f'n_components={best_k}'])

# DBSCAN tuning
best_score, best_eps, best_min_samples = -1, None, None
for eps in np.arange(0.2, 0.61, 0.05):
    for min_samples in range(3, 11):
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_pca)
        mask = labels != -1
        if len(np.unique(labels[mask])) < 2:
            continue
        score = silhouette_score(X_pca[mask], labels[mask])
        if score > best_score:
            best_score, best_eps, best_min_samples = score, eps, min_samples
results.append(['DBSCAN', best_score, f'eps={best_eps}, min_samples={best_min_samples}'])

# -------------------------
# 9. Evaluation
# -------------------------
results_df = pd.DataFrame(results, columns=['Model', 'Best Silhouette', 'Best Hyperparameter'])
print("\n=== HYPERPARAMETER TUNING RESULTS ===")
print(results_df)

# -------------------------
# 10. Visualization
# -------------------------
# Visualize KMeans clusters with 2D PCA
best_kmeans = KMeans(n_clusters=int(results_df.loc[results_df['Model']=='K-Means','Best Hyperparameter'].str.split('=').str[1].astype(int)), random_state=42)
k_labels = best_kmeans.fit_predict(X_pca)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=k_labels, cmap='Set2', s=50)
plt.title("K-Means Clusters (PCA 2D projection)")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.show()

# Optional: heatmap of RFM features per cluster
rfm['KMeans_Cluster'] = k_labels
cluster_summary = rfm.groupby('KMeans_Cluster')[['Recency','Frequency','Monetary']].mean()
sns.heatmap(cluster_summary, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title("Average RFM per K-Means Cluster")
plt.show()

# -------------------------
# 11. Final Results
# -------------------------
print("\n=== FINAL CLUSTERING RESULTS SUMMARY ===")
print(results_df)

# ==================================================
# END
# ==================================================


In [ ]:
# -------------------------
# 1. Imports
# -------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 2. Load Dataset
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------
# 3. Feature Engineering
# -------------------------
# Remove cancelled invoices
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
# Drop missing CustomerID
data = data.dropna(subset=['CustomerID'])
# Create TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 4. RFM Aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 5. Data Preprocessing
# -------------------------
X = rfm[['Recency', 'Frequency', 'Monetary']]

# Power transform (skew reduction)
pt = PowerTransformer(method='yeo-johnson')
X_pt = pt.fit_transform(X)

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pt)

# PCA (journal uses 2–3 components)
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 6. Hyperparameter Tuning & Clustering
# -------------------------
results = []

# Helper function to compute Silhouette + DBI safely
def evaluate_clusters(X, labels):
    unique_labels = np.unique(labels[labels != -1])
    if len(unique_labels) < 2:
        return np.nan, np.nan
    sil = silhouette_score(X[labels != -1], labels[labels != -1])
    dbi = davies_bouldin_score(X[labels != -1], labels[labels != -1])
    return sil, dbi

# --- K-Means tuning ---
best_score, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42).fit_predict(X_pca)
    sil, dbi = evaluate_clusters(X_pca, labels)
    if sil > best_score:
        best_score, best_dbi, best_k = sil, dbi, k
results.append(['K-Means', best_score, best_dbi, f'n_clusters={best_k}'])

# --- Agglomerative ---
best_score, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_pca)
    sil, dbi = evaluate_clusters(X_pca, labels)
    if sil > best_score:
        best_score, best_dbi, best_k = sil, dbi, k
results.append(['Agglomerative', best_score, best_dbi, f'n_clusters={best_k}'])

# --- BIRCH ---
best_score, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = Birch(n_clusters=k).fit_predict(X_pca)
    sil, dbi = evaluate_clusters(X_pca, labels)
    if sil > best_score:
        best_score, best_dbi, best_k = sil, dbi, k
results.append(['BIRCH', best_score, best_dbi, f'n_clusters={best_k}'])

# --- Gaussian Mixture ---
best_score, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = GaussianMixture(n_components=k, covariance_type='full', random_state=42).fit_predict(X_pca)
    sil, dbi = evaluate_clusters(X_pca, labels)
    if sil > best_score:
        best_score, best_dbi, best_k = sil, dbi, k
results.append(['GMM', best_score, best_dbi, f'n_components={best_k}'])

# --- DBSCAN (hyperparameter grid)
best_score, best_dbi, best_eps, best_min_samples = -1, np.inf, None, None
for eps in np.arange(0.3, 0.61, 0.05):
    for min_samples in range(3, 11):
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_pca)
        sil, dbi = evaluate_clusters(X_pca, labels)
        if np.isnan(sil):
            continue
        if sil > best_score:
            best_score, best_dbi, best_eps, best_min_samples = sil, dbi, eps, min_samples
results.append(['DBSCAN', best_score, best_dbi, f'eps={best_eps}, min_samples={best_min_samples}'])

# -------------------------
# 7. Results Table
# -------------------------
results_df = pd.DataFrame(results, columns=['Model', 'Silhouette', 'DBI', 'Best Hyperparameter'])
print("\n=== FINAL CLUSTERING RESULTS SUMMARY ===")
print(results_df)

# -------------------------
# 8. Visualization Example
# -------------------------
# K-Means clusters
best_k = int(results_df.loc[results_df['Model']=='K-Means','Best Hyperparameter'].str.split('=').str[1])
kmeans = KMeans(n_clusters=best_k, random_state=42)
k_labels = kmeans.fit_predict(X_pca)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=k_labels, cmap='Set2', s=50)
plt.title("K-Means Clusters (PCA 2D projection)")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.show()

# Heatmap of average RFM per cluster
rfm['KMeans_Cluster'] = k_labels
cluster_summary = rfm.groupby('KMeans_Cluster')[['Recency','Frequency','Monetary']].mean()
sns.heatmap(cluster_summary, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title("Average RFM per K-Means Cluster")
plt.show()

In [ ]:
# ==================================================
# Gaussian Mixture Model – Journal-Aligned (2D PCA)
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 1. Load and preprocess data
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Remove cancelled invoices and missing CustomerID
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])

# TotalPrice feature
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 2. RFM aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 3. Transform & scale
# -------------------------
X = rfm[['Recency', 'Frequency', 'Monetary']]
X_pt = PowerTransformer(method='yeo-johnson').fit_transform(X)  # reduces skew
X_scaled = StandardScaler().fit_transform(X_pt)

# -------------------------
# 4. PCA – 2 components as per journal
# -------------------------
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 5. GMM clustering
# -------------------------
n_components = 4  # usually 4 clusters in RFM papers
gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
labels = gmm.fit_predict(X_pca)

# -------------------------
# 6. Evaluation – Silhouette & DBI
# -------------------------
sil = silhouette_score(X_pca, labels)
dbi = davies_bouldin_score(X_pca, labels)

print(f"GMM Silhouette Score: {sil:.3f}")
print(f"GMM Davies-Bouldin Index: {dbi:.3f}")
print(f"Number of clusters: {n_components}")

# -------------------------
# 7. Visualization – 2D PCA space
# -------------------------
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='Set2', s=50)
plt.title("GMM Clusters on PCA Components (P1 & P2)")
plt.xlabel("P1")
plt.ylabel("P2")
plt.show()

# Optional: heatmap of RFM per GMM cluster
rfm['GMM_Cluster'] = labels
cluster_summary = rfm.groupby('GMM_Cluster')[['Recency','Frequency','Monetary']].mean()
sns.heatmap(cluster_summary, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title("Average RFM per GMM Cluster")
plt.show()


In [ ]:
# ==================================================
# DBSCAN Clustering – Journal-Aligned
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 1. Load and preprocess data
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Remove cancelled invoices & missing CustomerID
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])

# TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 2. RFM aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 3. Transform + scale
# -------------------------
X = rfm[['Recency','Frequency','Monetary']]
X_pt = PowerTransformer(method='yeo-johnson').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_pt)

# PCA – 2 components for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 4. DBSCAN clustering
# -------------------------
# Parameters from journal
eps = 0.3
min_samples = 5
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels = dbscan.fit_predict(X_pca)

# Identify core points
core_samples_mask = np.zeros_like(labels, dtype=bool)
core_samples_mask[dbscan.core_sample_indices_] = True

# Number of clusters (excluding noise)
unique_labels = set(labels)
n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

# -------------------------
# 5. Evaluation – Silhouette & DBI
# -------------------------
# Only use non-noise points for evaluation
mask = labels != -1
if len(set(labels[mask])) >= 2:
    sil = silhouette_score(X_pca[mask], labels[mask])
    dbi = davies_bouldin_score(X_pca[mask], labels[mask])
else:
    sil, dbi = np.nan, np.nan

print(f"DBSCAN Silhouette: {sil:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi:.3f}")
print(f"Number of clusters (excluding noise): {n_clusters}")
print(f"Number of noise points: {n_noise}")

# -------------------------
# 6. Visualization
# -------------------------
colors = [plt.cm.Set1(each) for each in np.linspace(0, 1, n_clusters)]
plt.figure(figsize=(8,6))

for k, col in zip(range(n_clusters), colors):
    class_member_mask = labels == k
    xy = X_pca[class_member_mask]
    plt.scatter(xy[:,0], xy[:,1], c=[col], s=50, label=f'Cluster {k}')

# Plot noise as black
xy = X_pca[labels == -1]
plt.scatter(xy[:,0], xy[:,1], c='black', s=80, marker='x', label='Noise')

plt.title(f"DBSCAN Clustering (eps={eps}, min_samples={min_samples}) – {n_clusters} clusters")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend()
plt.show()


In [ ]:
# ==================================================
# GMM Clustering Visualization – Journal Figure 6 Replication
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

# -------------------------
# 1. Load dataset
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------
# 2. Data Cleaning / Feature Engineering
# -------------------------
# Remove cancelled invoices
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
# Drop missing CustomerID
data = data.dropna(subset=['CustomerID'])
# TotalPrice
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 3. RFM Aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 4. Power Transform + Scale
# -------------------------
X = rfm[['Recency','Frequency','Monetary']]
X_pt = PowerTransformer(method='yeo-johnson').fit_transform(X)  # reduce skew
X_scaled = StandardScaler().fit_transform(X_pt)

# -------------------------
# 5. PCA – 2 components (P1 & P2)
# -------------------------
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 6. GMM Clustering
# -------------------------
n_components = 4  # journal uses 4 clusters
gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
labels = gmm.fit_predict(X_pca)

# -------------------------
# 7. Visualization – replicate journal Figure 6
# -------------------------
plt.figure(figsize=(8,6))

# Color map for 4 clusters
colors = ['yellow', 'green', 'red', 'blue']
for i in range(n_components):
    cluster_points = X_pca[labels == i]
    plt.scatter(cluster_points[:,0], cluster_points[:,1], 
                c=colors[i], label=f'Cluster {i+1}', s=50)

plt.title("GMM Clustering on PCA Components (P1 & P2)")
plt.xlabel("P1")
plt.ylabel("P2")
plt.legend()
plt.show()

# -------------------------
# 8. Optional: Average RFM per GMM cluster (heatmap)
# -------------------------
rfm['GMM_Cluster'] = labels
cluster_summary = rfm.groupby('GMM_Cluster')[['Recency','Frequency','Monetary']].mean()
sns.heatmap(cluster_summary, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title("Average RFM per GMM Cluster")
plt.show()


In [ ]:
# ==================================================
# GMM Clustering – Journal-Aligned Visualization + Metrics
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 1. Load dataset
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------
# 2. Data Cleaning / Feature Engineering
# -------------------------
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 3. RFM Aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 4. Transform & Scale
# -------------------------
X = rfm[['Recency','Frequency','Monetary']]
X_pt = PowerTransformer(method='yeo-johnson').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_pt)

# -------------------------
# 5. PCA – 2 components
# -------------------------
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 6. GMM Clustering
# -------------------------
n_components = 4  # typical number of clusters for RFM
gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
labels = gmm.fit_predict(X_pca)

# -------------------------
# 7. Evaluation – Silhouette & DBI
# -------------------------
sil = silhouette_score(X_pca, labels)
dbi = davies_bouldin_score(X_pca, labels)

print(f"GMM Silhouette Score: {sil:.3f}")
print(f"GMM Davies-Bouldin Index: {dbi:.3f}")
print(f"Number of clusters: {n_components}")

# -------------------------
# 8. Visualization – 2D PCA
# -------------------------
plt.figure(figsize=(8,6))
colors = ['yellow', 'green', 'red', 'blue']

for i in range(n_components):
    cluster_points = X_pca[labels == i]
    plt.scatter(cluster_points[:,0], cluster_points[:,1],
                c=colors[i], label=f'Cluster {i+1}', s=50)

plt.title("GMM Clusters on PCA Components (P1 & P2)")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Scatter plot
plt.figure(figsize=(8,6))
colors = ['yellow', 'green', 'red', 'blue']

for i, color in enumerate(colors):
    cluster_points = X_pca[labels == i]
    plt.scatter(cluster_points[:,0], cluster_points[:,1],
                c=color, label=f'Cluster {i+1}', s=30)

    # Draw ellipse for cluster
    cov_matrix = gmm.covariances_[i][:2,:2]  # 2x2 covariance for first 2 PCA components
    mean = gmm.means_[i][:2]
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    order = eigenvalues.argsort()[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:,order]
    angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues)  # 2 std dev for visualization
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle,
                      edgecolor=color, fc='None', lw=2, alpha=0.7)
    plt.gca().add_patch(ellipse)

plt.title("GMM Clusters on PCA Components – Ring-like Visualization")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend()
plt.show()


In [ ]:
# ==================================================
# GMM Clustering – Journal Figure 6 Style (2-color Ring)
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 1. Load and preprocess data
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Remove cancelled invoices & missing CustomerID
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 2. RFM aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 3. Transform & Scale
# -------------------------
X = rfm[['Recency', 'Frequency', 'Monetary']]
X_pt = PowerTransformer(method='yeo-johnson').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_pt)

# -------------------------
# 4. PCA – 2 components (P1 & P2)
# -------------------------
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 5. GMM Clustering
# -------------------------
n_components = 4  # internal GMM clusters
gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
labels = gmm.fit_predict(X_pca)

# -------------------------
# 6. Evaluation – Silhouette & DBI (all 4 clusters)
# -------------------------
sil = silhouette_score(X_pca, labels)
dbi = davies_bouldin_score(X_pca, labels)
print(f"GMM Silhouette Score: {sil:.3f}")
print(f"GMM Davies-Bouldin Index: {dbi:.3f}")
print(f"Internal number of clusters: {n_components}")

# -------------------------
# 7. Visualization – Journal Style (2 colors, ring-like)
# -------------------------
plt.figure(figsize=(8,6))

# Use only 2 colors for visualization, map clusters to these 2 colors
color_map = {0:'blue', 1:'green', 2:'blue', 3:'green'}  # maps 4 clusters to 2 colors
for i in range(n_components):
    cluster_points = X_pca[labels == i]
    plt.scatter(cluster_points[:,0], cluster_points[:,1],
                c=color_map[i], s=50, label=f'Cluster {i+1}')

plt.title("GMM Clusters on PCA Components (2-color Ring)")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend()
plt.show()


In [ ]:
# ==================================================
# DBSCAN Clustering – Journal Figure 7 Style
# ==================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 1. Load and preprocess data
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# Remove cancelled invoices & missing CustomerID
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 2. RFM aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 3. Transform & Scale
# -------------------------
X = rfm[['Recency','Frequency','Monetary']]
X_pt = PowerTransformer(method='yeo-johnson').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_pt)

# PCA – 2 components for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# -------------------------
# 4. DBSCAN clustering
# -------------------------
eps = 0.3       # journal-recommended value
min_samples = 5
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels = dbscan.fit_predict(X_pca)

# -------------------------
# 5. Evaluation – Silhouette & DBI
# -------------------------
mask = labels != -1  # only non-noise points
if len(np.unique(labels[mask])) >= 2:
    sil = silhouette_score(X_pca[mask], labels[mask])
    dbi = davies_bouldin_score(X_pca[mask], labels[mask])
else:
    sil, dbi = np.nan, np.nan

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"DBSCAN Silhouette: {sil:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi:.3f}")
print(f"Number of clusters (excluding noise): {n_clusters}")
print(f"Number of noise points: {n_noise}")

# -------------------------
# 6. Visualization – journal Figure 7 style
# -------------------------
plt.figure(figsize=(8,6))
colors = ['yellow','green','red']  # journal shows 3 clusters

for k, col in zip(range(n_clusters), colors):
    class_mask = labels == k
    xy = X_pca[class_mask]
    plt.scatter(xy[:,0], xy[:,1], c=col, s=50, label=f'Cluster {k+1}')

# Noise points
xy = X_pca[labels == -1]
plt.scatter(xy[:,0], xy[:,1], c='black', s=80, marker='x', label='Noise')

plt.title(f"DBSCAN Clustering (eps={eps}, min_samples={min_samples}) – {n_clusters} clusters")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend()
plt.show()


In [ ]:
# ==================================================
# Full Journal-Aligned Clustering Pipeline – Silhouette + DBI
# ==================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------
# 1. Load dataset
# -------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
data = pd.read_excel(file_path)

# -------------------------
# 2. Data cleaning / feature engineering
# -------------------------
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data.dropna(subset=['CustomerID'])
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

# -------------------------
# 3. RFM aggregation
# -------------------------
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------
# 4. Preprocessing tweaks to improve clustering
# -------------------------
X = rfm[['Recency','Frequency','Monetary']]

# Apply log transform to Recency + Monetary to reduce skew
X_log = X.copy()
X_log['Recency'] = np.log1p(X_log['Recency'])
X_log['Monetary'] = np.log1p(X_log['Monetary'])
# Frequency is less skewed, but optional PowerTransform
pt = PowerTransformer(method='yeo-johnson')
X_pt = pt.fit_transform(X_log)

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pt)

# PCA: 3 components for general clustering; 2 components for DBSCAN & GMM
pca_3 = PCA(n_components=3, random_state=42)
X_pca_3 = pca_3.fit_transform(X_scaled)

pca_2 = PCA(n_components=2, random_state=42)
X_pca_2 = pca_2.fit_transform(X_scaled)

# -------------------------
# 5. Clustering and Evaluation
# -------------------------
results = []

def evaluate(X, labels):
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2:
        return np.nan, np.nan
    sil = silhouette_score(X, labels)
    dbi = davies_bouldin_score(X, labels)
    return sil, dbi

# --- K-Means ---
best_sil, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42).fit_predict(X_pca_3)
    sil, dbi = evaluate(X_pca_3, labels)
    if sil > best_sil:
        best_sil, best_dbi, best_k = sil, dbi, k
results.append(['K-Means', best_sil, best_dbi, f'n_clusters={best_k}'])

# --- Agglomerative ---
best_sil, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_pca_3)
    sil, dbi = evaluate(X_pca_3, labels)
    if sil > best_sil:
        best_sil, best_dbi, best_k = sil, dbi, k
results.append(['Agglomerative', best_sil, best_dbi, f'n_clusters={best_k}'])

# --- BIRCH ---
best_sil, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = Birch(n_clusters=k).fit_predict(X_pca_3)
    sil, dbi = evaluate(X_pca_3, labels)
    if sil > best_sil:
        best_sil, best_dbi, best_k = sil, dbi, k
results.append(['BIRCH', best_sil, best_dbi, f'n_clusters={best_k}'])

# --- GMM ---
best_sil, best_dbi, best_k = -1, np.inf, None
for k in range(2, 9):
    labels = GaussianMixture(n_components=k, covariance_type='full', random_state=42).fit_predict(X_pca_2)
    sil, dbi = evaluate(X_pca_2, labels)
    if sil > best_sil:
        best_sil, best_dbi, best_k = sil, dbi, k
results.append(['GMM', best_sil, best_dbi, f'n_components={best_k}'])

# --- DBSCAN ---
best_sil, best_dbi, best_eps, best_min_samples = -1, np.inf, None, None
for eps in np.arange(0.2, 0.6, 0.05):
    for min_samples in range(3, 11):
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_pca_2)
        mask = labels != -1
        if len(np.unique(labels[mask])) < 2:
            continue
        sil, dbi = evaluate(X_pca_2[mask], labels[mask])
        if sil > best_sil:
            best_sil, best_dbi, best_eps, best_min_samples = sil, dbi, eps, min_samples
results.append(['DBSCAN', best_sil, best_dbi, f'eps={best_eps}, min_samples={best_min_samples}'])

# -------------------------
# 6. Results Table
# -------------------------
results_df = pd.DataFrame(results, columns=['Model', 'Silhouette', 'DBI', 'Best Hyperparameter'])
print("\n=== FINAL CLUSTERING RESULTS (Silhouette + DBI) ===\n")
print(results_df)


In [ ]:
from sklearn.cluster import KMeans

# Determine the optimal number of clusters using the elbow method
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_data)
    sse.append(kmeans.inertia_)

# Plot the elbow method
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), sse, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of clusters')
plt.ylabel('SSE')
plt.show()

# Apply K-Means with the optimal number of clusters (e.g., 4 clusters)
kmeans = KMeans(n_clusters=4, random_state=42)
customer_data['Cluster'] = kmeans.fit_predict(scaled_data)

# Visualize the clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=customer_data, x='NumOrders ', y='TotalSpent', hue='Cluster', palette='viridis')
plt.title('Customer Segmentation')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Normalize the data
scaled_features = scaler.fit_transform(customer_data[['TotalSpent', 'NumOrders ', 'TotalQuantity']])

# Convert scaled features back to DataFrame
scaled_data = pd.DataFrame(scaled_features, columns=['TotalSpent', 'NumOrders ', 'TotalQuantity'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
df = pd.read_excel(file_path)

# Check first few rows
print(df.head())

In [ ]:
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"

# Load the Excel file (default loads first sheet)
df = pd.read_excel(file_path)

# Check first few rows
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Full path to your file
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"

# Load the Excel file (default loads first sheet)
df = pd.read_excel(file_path)

# Check first few rows
print(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df = df.dropna(subset=['Customer ID'])
df.shape

In [ ]:
df = df.dropna(subset=['CustomerID'])
df.shape

In [ ]:
df[df['Invoice'].astype(str).str.startswith('C')]

In [ ]:
df[df['InvoiceNo'].astype(str).str.startswith('C')]

In [ ]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

In [ ]:
df.describe()

In [ ]:
df = df.copy()

df['CustomerID'] = df['CustomerID'].astype(int)
df['Total'] = df['Quantity'] * df['Price']
print("After cleaning:", df.shape)

In [ ]:
df = df.copy()

df['CustomerID'] = df['CustomerID'].astype(int)
df['Total'] = df['Quantity'] * df['UnitPrice']
print("After cleaning:", df.shape)

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  
    'InvoiceNo': 'nunique',                                     
    'Total': 'sum'                                           
}).reset_index()

rfm.rename(columns={'InvoiceDate': 'Recency', 'Invoice': 'Frequency', 'Total': 'Monetary'}, inplace=True)

rfm.head()

In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe().T

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Total': 'sum'
}).reset_index()

# Correct column renaming
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',  # fix here
    'Total': 'Monetary'
}, inplace=True)

rfm.head()


In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe().T

In [ ]:
rfm_log = rfm.copy()
rfm_log[['Recency', 'Frequency', 'Monetary']] = rfm_log[['Recency', 'Frequency', 'Monetary']].apply(lambda x: np.log1p(x))

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log[['Recency', 'Frequency', 'Monetary']])

In [ ]:
inertia = []
K = range(2, 10)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',   
        n_init=10, 
        random_state=42
    )
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K, inertia, 'o-')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.show()

In [ ]:
k = 4 
kmeans = KMeans(n_clusters=k,init='k-means++',n_init=10,  random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm.head()

In [ ]:
sil_score = silhouette_score(rfm_scaled, rfm['Cluster'])
db_index = davies_bouldin_score(rfm_scaled, rfm['Cluster'])

print(f"Silhouette Score: {sil_score:.3f}")
print(f"Davies–Bouldin Index: {db_index:.3f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, Birch
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Create total transaction value
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [ ]:
X_log = np.log1p(X_raw)

In [ ]:
# Select numeric columns for transformation
X_raw = df[['Quantity', 'UnitPrice', 'TotalPrice']]  # columns you want to log-transform

# Apply log(1 + x) transformation
X_log = np.log1p(X_raw)

print(X_log.head())


In [ ]:
# 5.2: Data Preprocessing for Clustering (No PCA)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load your data
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# Data cleaning
# 1. Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# 2. Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# 3. Create total transaction value
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# 4. Select numeric features for clustering
X_raw = df[['Quantity', 'UnitPrice', 'TotalPrice']]

# 5. Apply log(1 + x) transformation
X_log = np.log1p(X_raw)

# 6. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("Preprocessing complete. Shape:", X_scaled.shape)


In [ ]:
# ===============================
# 5.2 Clustering Results on the Raw Dataset
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, Birch, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# 1. Load and clean data
# -------------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Create total transaction value
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# -------------------------------
# 2. Select raw numeric features
# -------------------------------
X_raw = df[['Quantity', 'UnitPrice', 'TotalPrice']]

# Optional: log-transform to reduce skewness
X_log = np.log1p(X_raw)

# Optional: scale features for distance-based models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("Data preprocessing complete. Shape:", X_scaled.shape)

# -------------------------------
# 3. Run all five clustering models
# -------------------------------
results = []

# ---- K-Means ----
kmeans = KMeans(n_clusters=3, random_state=42)
labels_km = kmeans.fit_predict(X_scaled)
sil_km = silhouette_score(X_scaled, labels_km)
dbi_km = davies_bouldin_score(X_scaled, labels_km)

results.append({
    'Model': 'K-Means',
    'Silhouette': sil_km,
    'DBI': dbi_km,
    'Num_Clusters': len(np.unique(labels_km)),
    'Num_Noise': 0
})

# Plot K-Means clusters
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_scaled[:,0], y=X_scaled[:,2], hue=labels_km, palette='Set2', s=50)
plt.title('K-Means Clusters (Raw Dataset)')
plt.xlabel('Quantity (scaled)')
plt.ylabel('TotalPrice (scaled)')
plt.show()


# ---- Gaussian Mixture ----
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)
sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

results.append({
    'Model': 'GMM',
    'Silhouette': sil_gmm,
    'DBI': dbi_gmm,
    'Num_Clusters': len(np.unique(labels_gmm)),
    'Num_Noise': 0
})

# Plot GMM clusters
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_scaled[:,0], y=X_scaled[:,2], hue=labels_gmm, palette='Set1', s=50)
plt.title('GMM Clusters (Raw Dataset)')
plt.xlabel('Quantity (scaled)')
plt.ylabel('TotalPrice (scaled)')
plt.show()


# ---- DBSCAN ----
dbscan = DBSCAN(eps=1.5, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)

# Handle noise points (-1)
mask = labels_db != -1
sil_db = silhouette_score(X_scaled[mask], labels_db[mask]) if len(set(labels_db[mask])) > 1 else np.nan
dbi_db = davies_bouldin_score(X_scaled[mask], labels_db[mask]) if len(set(labels_db[mask])) > 1 else np.nan

results.append({
    'Model': 'DBSCAN',
    'Silhouette': sil_db,
    'DBI': dbi_db,
    'Num_Clusters': len(set(labels_db)) - (1 if -1 in labels_db else 0),
    'Num_Noise': list(labels_db).count(-1)
})

# Plot DBSCAN clusters
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_scaled[:,0], y=X_scaled[:,2], hue=labels_db, palette='coolwarm', s=50)
plt.title('DBSCAN Clusters (Raw Dataset)')
plt.xlabel('Quantity (scaled)')
plt.ylabel('TotalPrice (scaled)')
plt.show()


# ---- Birch ----
birch = Birch(n_clusters=3)
labels_birch = birch.fit_predict(X_scaled)
sil_birch = silhouette_score(X_scaled, labels_birch)
dbi_birch = davies_bouldin_score(X_scaled, labels_birch)

results.append({
    'Model': 'Birch',
    'Silhouette': sil_birch,
    'DBI': dbi_birch,
    'Num_Clusters': len(np.unique(labels_birch)),
    'Num_Noise': 0
})

# Plot Birch clusters
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_scaled[:,0], y=X_scaled[:,2], hue=labels_birch, palette='pastel', s=50)
plt.title('Birch Clusters (Raw Dataset)')
plt.xlabel('Quantity (scaled)')
plt.ylabel('TotalPrice (scaled)')
plt.show()


# ---- Hierarchical (Agglomerative) ----
agglo = AgglomerativeClustering(n_clusters=3)
labels_agg = agglo.fit_predict(X_scaled)
sil_agg = silhouette_score(X_scaled, labels_agg)
dbi_agg = davies_bouldin_score(X_scaled, labels_agg)

results.append({
    'Model': 'Hierarchical',
    'Silhouette': sil_agg,
    'DBI': dbi_agg,
    'Num_Clusters': len(np.unique(labels_agg)),
    'Num_Noise': 0
})

# Plot Hierarchical clusters
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_scaled[:,0], y=X_scaled[:,2], hue=labels_agg, palette='tab10', s=50)
plt.title('Hierarchical Clusters (Raw Dataset)')
plt.xlabel('Quantity (scaled)')
plt.ylabel('TotalPrice (scaled)')
plt.show()


# -------------------------------
# 4. Summary Table
# -------------------------------
results_df = pd.DataFrame(results)
print("Clustering Results on Raw Dataset:")
print(results_df)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np
import pandas as pd

# X_scaled is your preprocessed dataset from 5.2.1

# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
labels_km = kmeans.fit_predict(X_scaled)

# Evaluation metrics
sil_km = silhouette_score(X_scaled, labels_km)
dbi_km = davies_bouldin_score(X_scaled, labels_km)

print("K-Means Silhouette Score:", sil_km)
print("K-Means Davies-Bouldin Index:", dbi_km)
print("Number of Clusters:", len(np.unique(labels_km)))

# Optional: plot clusters (Quantity vs TotalPrice)
plt.figure(figsize=(6,5))
sns.scatterplot(x=X_scaled[:,0], y=X_scaled[:,2], hue=labels_km, palette='Set2', s=50)
plt.title('K-Means Clusters (Raw Dataset)')
plt.xlabel('Quantity (scaled)')
plt.ylabel('TotalPrice (scaled)')
plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

# Fit K-Means (this is fine on full data)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_scaled)

# ---- SAFE METRICS ----

# Silhouette using sampling
sil_km = silhouette_score(
    X_scaled,
    labels_km,
    sample_size=10000,   # KEY FIX
    random_state=42
)

# DBI can be computed on full dataset
dbi_km = davies_bouldin_score(X_scaled, labels_km)

print("K-Means Silhouette Score (sampled):", sil_km)
print("K-Means Davies-Bouldin Index:", dbi_km)
print("Number of Clusters:", len(np.unique(labels_km)))


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

# K-Means clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_scaled)

# Silhouette Score (sampled)
sil_km = silhouette_score(
    X_scaled,
    labels_km,
    sample_size=20000,
    random_state=42
)

# Davies-Bouldin Index (full data)
dbi_km = davies_bouldin_score(X_scaled, labels_km)

# Output (rounded to 3 decimal places)
print("K-Means Silhouette Score:", round(sil_km, 3))
print("K-Means Davies-Bouldin Index:", round(dbi_km, 3))
print("Number of Clusters:", len(np.unique(labels_km)))


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

# GMM clustering
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)

# Silhouette Score (sampled)
sil_gmm = silhouette_score(
    X_scaled,
    labels_gmm,
    sample_size=20000,
    random_state=42
)

# Davies-Bouldin Index (full data)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

# Output (rounded to 3 decimal places)
print("GMM Silhouette Score:", round(sil_gmm, 3))
print("GMM Davies-Bouldin Index:", round(dbi_gmm, 3))
print("Number of Clusters:", len(np.unique(labels_gmm)))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create plotting DataFrame from raw scaled data
plot_df = pd.DataFrame(
    X_scaled,
    columns=['Quantity', 'UnitPrice', 'TotalPrice']
)
plot_df['Cluster'] = labels_km

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=10,
    alpha=0.6
)

plt.title('K-Means Clustering on Raw Transaction Data')
plt.xlabel('Quantity (standardised)')
plt.ylabel('Total Price (standardised)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create plotting DataFrame
plot_df = pd.DataFrame(
    X_scaled,
    columns=['Quantity', 'UnitPrice', 'TotalPrice']
)
plot_df['Cluster'] = labels_km.astype(str)  # make cluster categorical

# Create a grid of scatter plots by cluster
g = sns.FacetGrid(plot_df, col='Cluster', col_wrap=2, height=4, sharex=True, sharey=True)
g.map_dataframe(sns.scatterplot, x='Quantity', y='TotalPrice', s=30, alpha=0.7, edgecolor=None)
g.set_axis_labels("Quantity (standardised)", "Total Price (standardised)")
g.set_titles("Cluster {col_name}")
g.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create plotting DataFrame
plot_df = pd.DataFrame(
    X_scaled,
    columns=['Quantity', 'UnitPrice', 'TotalPrice']
)
plot_df['Cluster'] = labels_km.astype(str)  # make cluster categorical

# Set figure size and style
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")

# Scatter plot
sns.scatterplot(
    data=plot_df,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,          # bigger points
    alpha=0.7,     # slightly transparent
    edgecolor='w'  # white edges for clarity
)

# Titles and labels
plt.title('K-Means Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
plot_df.groupby('Cluster')[['Quantity','TotalPrice']].mean()


In [ ]:
# Convert cluster means back to original scale
cluster_means = pd.DataFrame(X_scaled, columns=['Quantity','TotalPrice'])
cluster_means['Cluster'] = labels_km
cluster_summary = cluster_means.groupby('Cluster').mean()

# Reverse scaling
cluster_summary[['Quantity','TotalPrice']] = scaler.inverse_transform(cluster_summary[['Quantity','TotalPrice']])
print(cluster_summary)


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Create DataFrame with all 3 features and cluster labels
cluster_df = pd.DataFrame(
    X_scaled, 
    columns=['Quantity', 'UnitPrice', 'TotalPrice']
)
cluster_df['Cluster'] = labels_km

# Compute cluster means in scaled space
cluster_summary_scaled = cluster_df.groupby('Cluster').mean()
print("Cluster means (scaled values):")
print(cluster_summary_scaled)

# Reverse scaling to original units
# Fit a scaler only on the original 3 columns
scaler_all = StandardScaler()
scaler_all.fit(df[['Quantity', 'UnitPrice', 'TotalPrice']])

# Inverse transform to get original units
cluster_summary_original = pd.DataFrame(
    scaler_all.inverse_transform(cluster_summary_scaled),
    columns=['Quantity', 'UnitPrice', 'TotalPrice'],
    index=cluster_summary_scaled.index
)

print("\nCluster means in original units:")
print(cluster_summary_original)


In [ ]:
# Add cluster labels to the original raw dataset (before scaling)
df['Cluster'] = labels_km

# Compute cluster means on original features
cluster_summary_original = df.groupby('Cluster')[['Quantity', 'UnitPrice', 'TotalPrice']].mean()
print(cluster_summary_original)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.mixture import GaussianMixture

# Fit GMM
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)

# Prepare plotting DataFrame
plot_df_gmm = pd.DataFrame(
    X_scaled,
    columns=['Quantity', 'UnitPrice', 'TotalPrice']
)
plot_df_gmm['Cluster'] = labels_gmm.astype(str)  # GMM cluster labels as string

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_gmm,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,         # bigger points
    alpha=0.7,    # slightly transparent
    edgecolor='w' # white edges for clarity
)

plt.title('GMM Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Select features
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]

# Sample 20,000 points for faster computation
X_sample = X.sample(n=20000, random_state=42)

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)


In [ ]:
import pandas as pd
import numpy as np

# Load dataset
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Select features
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]

# Sample 20,000 for speed
X_sample = X.sample(n=20000, random_state=42)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

# GMM
gmm = GaussianMixture(n_components=3, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

# Metrics
silhouette_gmm = silhouette_score(X_scaled, gmm_labels)
dbi_gmm = davies_bouldin_score(X_scaled, gmm_labels)

print("GMM Silhouette Score:", round(silhouette_gmm, 3))
print("GMM Davies-Bouldin Index:", round(dbi_gmm, 3))


In [ ]:
cluster_df = X_sample.copy()
cluster_df['Cluster'] = gmm_labels

cluster_summary_gmm = cluster_df.groupby('Cluster').mean().round(3)
cluster_summary_gmm


In [ ]:
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# -------------------------------
# 1. Select features
# -------------------------------
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]

# -------------------------------
# 2. Sample 20,000 observations
# -------------------------------
X_sample = X.sample(n=20000, random_state=42)

# -------------------------------
# 3. Feature scaling
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

# -------------------------------
# 4. Apply BIRCH
# -------------------------------
birch = Birch(n_clusters=3)
labels_birch = birch.fit_predict(X_scaled)

# -------------------------------
# 5. Evaluation metrics
# -------------------------------
sil_birch = silhouette_score(X_scaled, labels_birch)
dbi_birch = davies_bouldin_score(X_scaled, labels_birch)

print(f"BIRCH Silhouette Score: {sil_birch:.3f}")
print(f"BIRCH Davies-Bouldin Index: {dbi_birch:.3f}")


In [ ]:
# Convert to DataFrame
birch_df = pd.DataFrame(X_sample, columns=['Quantity', 'UnitPrice', 'TotalPrice'])
birch_df['Cluster'] = labels_birch

# Cluster summary
birch_summary = birch_df.groupby('Cluster').mean().round(3)
birch_summary


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create DataFrame for plotting
plot_df = X_sample.copy()
plot_df['Cluster'] = labels_birch

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    plot_df['Quantity'],
    plot_df['TotalPrice'],
    c=plot_df['Cluster'],
    alpha=0.6,
    s=25
)

plt.xlabel('Quantity')
plt.ylabel('Total Transaction Value (TotalPrice)')
plt.title('BIRCH Clustering Results (Raw Dataset)')
plt.colorbar(scatter, label='Cluster')

plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cluster import Birch

# Fit BIRCH
birch = Birch(n_clusters=3)
labels_birch = birch.fit_predict(X_scaled)

# Prepare plotting DataFrame
plot_df_birch = pd.DataFrame(
    X_scaled,
    columns=['Quantity', 'UnitPrice', 'TotalPrice']
)
plot_df_birch['Cluster'] = labels_birch.astype(str)

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_birch,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,          # bigger points
    alpha=0.7,     # transparency
    edgecolor='w'  # white edges for clarity
)

plt.title('BIRCH Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Use the same sampled 20,000 observations
X_sampled = X.sample(n=20000, random_state=42)
X_scaled_sampled = scaler.fit_transform(X_sampled)  # standardize if not already

# Fit DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)  # adjust eps/min_samples as needed
labels_db = dbscan.fit_predict(X_scaled_sampled)

# Number of clusters (excluding noise)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Calculate metrics (excluding noise points for silhouette score)
mask = labels_db != -1
sil_db = silhouette_score(X_scaled_sampled[mask], labels_db[mask]) if n_clusters_db > 1 else float('nan')
dbi_db = davies_bouldin_score(X_scaled_sampled[mask], labels_db[mask]) if n_clusters_db > 1 else float('nan')

print(f"DBSCAN Silhouette Score: {sil_db:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi_db:.3f}")
print(f"Number of clusters: {n_clusters_db}, Number of noise points: {n_noise}")

# Prepare DataFrame for plotting
plot_df_db = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_db['Cluster'] = labels_db.astype(str)

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# DBSCAN tuned for ~3 clusters
dbscan = DBSCAN(eps=3.5, min_samples=50)  # adjust eps and min_samples
labels_db = dbscan.fit_predict(X_scaled_sampled)

# Number of clusters (excluding noise)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Metrics (excluding noise for Silhouette/DBI)
mask = labels_db != -1
sil_db = silhouette_score(X_scaled_sampled[mask], labels_db[mask])
dbi_db = davies_bouldin_score(X_scaled_sampled[mask], labels_db[mask])

print(f"DBSCAN Silhouette Score: {sil_db:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi_db:.3f}")
print(f"Number of clusters: {n_clusters_db}, Number of noise points: {n_noise}")


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sample 20k already exists: X_scaled_sampled

# Tuned DBSCAN parameters to get ~3 clusters
dbscan = DBSCAN(eps=6.0, min_samples=50)  # tuned eps and min_samples
labels_db = dbscan.fit_predict(X_scaled_sampled)

# Count clusters and noise
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Compute metrics (exclude noise for Silhouette/DBI)
mask = labels_db != -1
if n_clusters_db > 1:
    sil_db = silhouette_score(X_scaled_sampled[mask], labels_db[mask])
    dbi_db = davies_bouldin_score(X_scaled_sampled[mask], labels_db[mask])
else:
    sil_db = np.nan
    dbi_db = np.nan

print(f"DBSCAN Silhouette Score: {sil_db:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi_db:.3f}")
print(f"Number of clusters: {n_clusters_db}, Number of noise points: {n_noise}")

# Prepare DataFrame for plotting
plot_df_db = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_db['Cluster'] = labels_db.astype(str)

# Scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Tuned DBSCAN parameters for ~3 clusters
dbscan = DBSCAN(eps=12, min_samples=10)  # eps increased, min_samples lowered
labels_db = dbscan.fit_predict(X_scaled_sampled)

# Count clusters and noise
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Metrics (exclude noise)
mask = labels_db != -1
if n_clusters_db > 1:
    sil_db = silhouette_score(X_scaled_sampled[mask], labels_db[mask])
    dbi_db = davies_bouldin_score(X_scaled_sampled[mask], labels_db[mask])
else:
    sil_db = np.nan
    dbi_db = np.nan

print(f"DBSCAN Silhouette Score: {sil_db:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi_db:.3f}")
print(f"Number of clusters: {n_clusters_db}, Number of noise points: {n_noise}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare DataFrame for plotting
plot_df_db = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_db['Cluster'] = labels_db.astype(str)  # treat cluster labels as strings for hue

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,         # bigger points
    alpha=0.7,    # slightly transparent
    edgecolor='w' # white edges for clarity
)

plt.title('DBSCAN Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Create DataFrame with cluster labels
plot_df_db = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_db['Cluster'] = labels_db.astype(str)

# Compute cluster means in original units
# First, inverse transform if you scaled, otherwise use raw sample
import numpy as np

cluster_summary = plot_df_db.groupby('Cluster').mean().round(3)
cluster_summary


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Use the same sampled 20,000 observations
X_sampled = X.sample(n=20000, random_state=42)
X_scaled_sampled = scaler.fit_transform(X_sampled)  # standardize if not already

# Fit DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)  # adjust eps/min_samples as needed
labels_db = dbscan.fit_predict(X_scaled_sampled)

# Number of clusters (excluding noise)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = list(labels_db).count(-1)

# Calculate metrics (excluding noise points for silhouette score)
mask = labels_db != -1
sil_db = silhouette_score(X_scaled_sampled[mask], labels_db[mask]) if n_clusters_db > 1 else float('nan')
dbi_db = davies_bouldin_score(X_scaled_sampled[mask], labels_db[mask]) if n_clusters_db > 1 else float('nan')

print(f"DBSCAN Silhouette Score: {sil_db:.3f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi_db:.3f}")
print(f"Number of clusters: {n_clusters_db}, Number of noise points: {n_noise}")

# Prepare DataFrame for plotting
plot_df_db = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_db['Cluster'] = labels_db.astype(str)

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Create DataFrame with cluster labels
plot_df_db = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_db['Cluster'] = labels_db.astype(str)

# Compute cluster means in original units
# First, inverse transform if you scaled, otherwise use raw sample
import numpy as np

cluster_summary = plot_df_db.groupby('Cluster').mean().round(3)
cluster_summary


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Fit hierarchical clustering
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')  # Ward linkage minimizes variance
labels_hier = hier.fit_predict(X_scaled_sampled)

# Metrics
sil_hier = silhouette_score(X_scaled_sampled, labels_hier)
dbi_hier = davies_bouldin_score(X_scaled_sampled, labels_hier)

print(f"Hierarchical Clustering Silhouette Score: {sil_hier:.3f}")
print(f"Hierarchical Clustering Davies-Bouldin Index: {dbi_hier:.3f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_hier,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('Hierarchical Clustering (Agglomerative) on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
plot_df_hier = pd.DataFrame(X_scaled_sampled, columns=['Quantity','UnitPrice','TotalPrice'])
plot_df_hier['Cluster'] = labels_hier.astype(str)  # convert to string for color mapping


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_hier,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('Hierarchical Clustering (Agglomerative) on Raw Transaction Data', fontsize=14)
plt.xlabel('Quantity (standardised)', fontsize=12)
plt.ylabel('Total Price (standardised)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Compute cluster means (scaled values)
cluster_summary_hier = plot_df_hier.groupby('Cluster').mean().round(3)
cluster_summary_hier


In [ ]:
# Assuming you used StandardScaler
cluster_means_scaled = plot_df_hier.groupby('Cluster')[['Quantity','UnitPrice','TotalPrice']].mean()
cluster_means_original = scaler.inverse_transform(cluster_means_scaled)  # convert back to original units

# Create DataFrame for table
cluster_summary_hier_original = pd.DataFrame(cluster_means_original, 
                                             columns=['Quantity','UnitPrice','TotalPrice'], 
                                             index=cluster_means_scaled.index)
cluster_summary_hier_original.round(3)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.418, 0.212, 0.980, 0.799, 0.963]
dbi_scores = [0.828, 1.652, 0.494, 0.349, 0.701]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

# Create figure
fig, ax1 = plt.subplots(figsize=(10,6))

# Plot Silhouette Score
bar1 = ax1.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
# Plot Davies-Bouldin Index
bar2 = ax1.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Add labels and title
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparison of Clustering Models', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=30, fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Show values on top of bars
for rect in bar1 + bar2:
    height = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2., 1.01*height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.418, 0.212, 0.980, 0.799, 0.963]
dbi_scores = [0.828, 1.652, 0.494, 0.349, 0.701]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

# Create figure
fig, ax1 = plt.subplots(figsize=(10,6))

# Plot Silhouette Score
bar1 = ax1.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
# Plot Davies-Bouldin Index
bar2 = ax1.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Add labels and title
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparison of Clustering Models', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=0, fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Show values on top of bars
for rect in bar1 + bar2:
    height = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2., 1.01*height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.418, 0.212, 0.980, 0.799, 0.963]
dbi_scores = [0.828, 1.652, 0.494, 0.349, 0.701]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

# Create figure
fig, ax1 = plt.subplots(figsize=(10,6))

# Plot Silhouette Score
bar1 = ax1.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
# Plot Davies-Bouldin Index
bar2 = ax1.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Add labels and title
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparison of Clustering Models', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=0, fontsize=11)  # straight labels
ax1.legend(fontsize=11)

# Show values on top of bars
for rect in bar1 + bar2:
    height = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2., 1.01*height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.418, 0.212, 0.980, 0.799, 0.963]
dbi_scores = [0.828, 1.652, 0.494, 0.349, 0.701]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

# Create figure
fig, ax1 = plt.subplots(figsize=(10,6))

# Plot Silhouette Score
bar1 = ax1.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
# Plot Davies-Bouldin Index
bar2 = ax1.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Add labels and title
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparison of Clustering Models', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=0, fontsize=11)  # straight labels
ax1.legend(fontsize=11)

# Show values on top of bars
for rect in bar1 + bar2:
    height = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2., 1.01*height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.418, 0.212, 0.980, 0.799, 0.963]
dbi_scores = [0.828, 1.652, 0.494, 0.349, 0.701]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

# Create figure
fig, ax1 = plt.subplots(figsize=(10,6))

# Turn off grid completely
ax1.grid(False)

# Plot Silhouette Score
bar1 = ax1.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
# Plot Davies-Bouldin Index
bar2 = ax1.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Add labels and title
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparison of Clustering Models', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=0, fontsize=11)  # straight labels
ax1.legend(fontsize=11)

# Show values on top of bars
for rect in bar1 + bar2:
    height = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2., 1.01*height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# -------------------------------
# 1. Load and clean data
# -------------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# -------------------------------
# 2. RFM calculation
# -------------------------------
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                     # Frequency
    'TotalPrice': 'sum'                                         # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# -------------------------------
# 3. Handle skewness (log transform)
# -------------------------------
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

# -------------------------------
# 4. Standardisation
# -------------------------------
scaler = StandardScaler()
X_rfm_scaled = scaler.fit_transform(rfm_log)

print("RFM preprocessing complete. Shape:", X_rfm_scaled.shape)


In [ ]:
# RFM descriptive statistics (original scale)
rfm_summary = rfm[['Recency', 'Frequency', 'Monetary']].describe().T
rfm_summary = rfm_summary[['mean', 'std', 'min', 'max']]

rfm_summary.round(2)


In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)


In [ ]:
rfm_summary = (
    rfm[['Recency', 'Frequency', 'Monetary']]
    .describe()
    .T[['mean', 'std', 'min', 'max']]
)

rfm_summary.round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

sns.histplot(rfm['Recency'], bins=30, ax=axes[0], kde=True)
axes[0].set_title('Recency Distribution')

sns.histplot(rfm['Frequency'], bins=30, ax=axes[1], kde=True)
axes[1].set_title('Frequency Distribution')

sns.histplot(rfm['Monetary'], bins=30, ax=axes[2], kde=True)
axes[2].set_title('Monetary Distribution')

plt.tight_layout()
plt.show()


In [ ]:
sns.histplot(np.log1p(rfm['Monetary']), bins=30, ax=axes[2], kde=True)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

sns.histplot(rfm['Recency'], bins=30, ax=axes[0], kde=True)
axes[0].set_title('Recency Distribution')

sns.histplot(rfm['Frequency'], bins=30, ax=axes[1], kde=True)
axes[1].set_title('Frequency Distribution')

sns.histplot(rfm['Monetary'], bins=30, ax=axes[2], kde=True)
axes[2].set_title('Monetary Distribution')

plt.tight_layout()
plt.show()


In [ ]:
sns.histplot(np.log1p(rfm['Monetary']), bins=30, kde=True)
plt.title('Monetary Distribution (log-transformed)')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming your RFM dataframe is named 'rfm' with columns: 'Recency', 'Frequency', 'Monetary'

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recency - linear scale
sns.histplot(rfm['Recency'], bins=30, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days since last purchase')
axes[0].set_ylabel('Count')

# Frequency - log scale
sns.histplot(np.log1p(rfm['Frequency']), bins=30, kde=True, ax=axes[1], color='lightgreen')
axes[1].set_title('Frequency Distribution (log-transformed)')
axes[1].set_xlabel('log(1 + Frequency)')
axes[1].set_ylabel('Count')

# Monetary - log scale
sns.histplot(np.log1p(rfm['Monetary']), bins=30, kde=True, ax=axes[2], color='salmon')
axes[2].set_title('Monetary Distribution (log-transformed)')
axes[2].set_xlabel('log(1 + Monetary)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming your RFM dataframe is named 'rfm' with columns: 'Recency', 'Frequency', 'Monetary'

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recency - linear scale
sns.histplot(rfm['Recency'], bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days since last purchase')
axes[0].set_ylabel('Count')

# Frequency - log scale
sns.histplot(np.log1p(rfm['Frequency']), bins=30, kde=True, ax=axes[1], color='blue')
axes[1].set_title('Frequency Distribution (log-transformed)')
axes[1].set_xlabel('log(1 + Frequency)')
axes[1].set_ylabel('Count')

# Monetary - log scale
sns.histplot(np.log1p(rfm['Monetary']), bins=30, kde=True, ax=axes[2], color='blue')
axes[2].set_title('Monetary Distribution (log-transformed)')
axes[2].set_xlabel('log(1 + Monetary)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(15,4))

# Recency - blue
sns.histplot(rfm['Recency'], bins=30, ax=axes[0], kde=True, color='blue')
axes[0].set_title('Recency Distribution')

# Frequency - blue
sns.histplot(rfm['Frequency'], bins=30, ax=axes[1], kde=True, color='blue')
axes[1].set_title('Frequency Distribution')

# Monetary - blue
sns.histplot(rfm['Monetary'], bins=30, ax=axes[2], kde=True, color='blue')
axes[2].set_title('Monetary Distribution')

plt.tight_layout()
plt.show()


In [ ]:
# Select RFM features
X_rfm = rfm[['Recency', 'Frequency', 'Monetary']]

# Optional: log-transform skewed features (Frequency and Monetary)
X_rfm['Frequency_log'] = np.log1p(X_rfm['Frequency'])
X_rfm['Monetary_log'] = np.log1p(X_rfm['Monetary'])
X_rfm_transformed = X_rfm[['Recency', 'Frequency_log', 'Monetary_log']]

# Standardise
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled_rfm = scaler.fit_transform(X_rfm_transformed)


In [ ]:
# Since RFM dataset is small, we can use all rows
X_sample_rfm = X_scaled_rfm  # 4339 rows#

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import Birch, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------- K-Means --------------------
kmeans_rfm = KMeans(n_clusters=3, random_state=42)
labels_km_rfm = kmeans_rfm.fit_predict(X_sample_rfm)

sil_km_rfm = silhouette_score(X_sample_rfm, labels_km_rfm)
dbi_km_rfm = davies_bouldin_score(X_sample_rfm, labels_km_rfm)

print(f"K-Means Silhouette Score: {sil_km_rfm:.3f}")
print(f"K-Means Davies-Bouldin Index: {dbi_km_rfm:.3f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare DataFrame for plotting
plot_df_rfm = pd.DataFrame(X_scaled_rfm, columns=['Recency', 'Frequency_log', 'Monetary_log'])
plot_df_rfm['Cluster'] = labels_km_rfm.astype(str)  # K-Means cluster labels as string

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_rfm,
    x='Frequency_log',
    y='Monetary_log',
    hue='Cluster',
    palette='Set2',  # distinct colors for clusters
    s=50,            # slightly bigger points
    alpha=0.7,       # slightly transparent
    edgecolor='w'    # white edges for clarity
)

plt.title('K-Means Clustering on RFM Data', fontsize=14)
plt.xlabel('Frequency (log-transformed)', fontsize=12)
plt.ylabel('Monetary (log-transformed)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare DataFrame for plotting
plot_df_rfm = pd.DataFrame(X_scaled_rfm, columns=['Recency', 'Frequency_log', 'Monetary_log'])
plot_df_rfm['Cluster'] = labels_km_rfm.astype(str)  # K-Means cluster labels as string

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_rfm,
    x='Frequency_log',
    y='Monetary_log',
    hue='Cluster',
    palette='Set2',  # distinct colors for clusters
    s=50,            # slightly bigger points
    alpha=0.7,       # slightly transparent
    edgecolor='w'    # white edges for clarity
)

plt.title('K-Means Clustering on RFM Data', fontsize=14)
plt.xlabel('Frequency (log-transformed)', fontsize=12)
plt.ylabel('Monetary (log-transformed)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare DataFrame for plotting
plot_df_rfm = pd.DataFrame(X_scaled_rfm, columns=['Recency', 'Frequency_log', 'Monetary_log'])
plot_df_rfm['Cluster'] = labels_km_rfm.astype(str)  # K-Means cluster labels

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_rfm,
    x='Frequency_log',
    y='Monetary_log',
    hue='Cluster',
    palette='Set2',
    s=50,          # slightly bigger points
    alpha=0.7,     # slightly transparent
    edgecolor='w'  # white edges for clarity
)

plt.title('K-Means Clustering on RFM Data', fontsize=14)
plt.xlabel('Frequency (log-transformed)', fontsize=12)
plt.ylabel('Monetary (log-transformed)', fontsize=12)

# Legend outside top-right
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, 
           bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare DataFrame for plotting
plot_df_rfm = pd.DataFrame(X_scaled_rfm, columns=['Recency', 'Frequency_log', 'Monetary_log'])
plot_df_rfm['Cluster'] = labels_km_rfm.astype(str)  # K-Means cluster labels

# Scale Recency to use as point size (optional: multiply to increase visibility)
size_scale = 50  # adjust as needed
plot_df_rfm['Recency_size'] = (plot_df_rfm['Recency'] - plot_df_rfm['Recency'].min() + 1) / \
                              (plot_df_rfm['Recency'].max() - plot_df_rfm['Recency'].min() + 1) * size_scale + 10

# Journal-style scatter plot with Recency as point size
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_rfm,
    x='Frequency_log',
    y='Monetary_log',
    hue='Cluster',
    size='Recency_size',      # point size encodes Recency
    sizes=(10, 200),          # min/max point size
    palette='Set2',
    alpha=0.7,
    edgecolor='w'
)

plt.title('K-Means Clustering on RFM Data (Recency as Point Size)', fontsize=14)
plt.xlabel('Frequency (log-transformed)', fontsize=12)
plt.ylabel('Monetary (log-transformed)', fontsize=12)

# Legend outside top-right
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, 
           bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Assuming your RFM dataframe is 'rfm' with columns Recency, Frequency, Monetary
scaler = StandardScaler()
X_scaled_rfm = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

kmeans_rfm = KMeans(n_clusters=3, random_state=42)
labels_km_rfm = kmeans_rfm.fit_predict(X_scaled_rfm)

sil_score = silhouette_score(X_scaled_rfm, labels_km_rfm)
print(f"K-Means Silhouette Score: {sil_score:.4f}")


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Standardize RFM features
scaler = StandardScaler()
X_scaled_rfm = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

# K-Means clustering
kmeans_rfm = KMeans(n_clusters=3, random_state=42)
labels_km_rfm = kmeans_rfm.fit_predict(X_scaled_rfm)

# Metrics
sil_score = silhouette_score(X_scaled_rfm, labels_km_rfm)
dbi_score = davies_bouldin_score(X_scaled_rfm, labels_km_rfm)

print(f"K-Means Silhouette Score: {sil_score:.4f}")
print(f"K-Means Davies–Bouldin Index: {dbi_score:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Standardize RFM features
scaler = StandardScaler()
X_scaled_rfm = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

# K-Means clustering
kmeans_rfm = KMeans(n_clusters=3, random_state=42)
labels_km_rfm = kmeans_rfm.fit_predict(X_scaled_rfm)

# Prepare plotting DataFrame
plot_df_km = pd.DataFrame(
    X_scaled_rfm,
    columns=['Recency','Frequency','Monetary']
)
plot_df_km['Cluster'] = labels_km_rfm.astype(str)  # Cluster labels as strings

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_km,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,         # point size
    alpha=0.7,    # transparency
    edgecolor='w' # white edge for clarity
)

plt.title('K-Means Clustering on RFM Data', fontsize=14)
plt.xlabel('Recency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')  # legend top-right
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Add cluster labels to RFM dataframe
rfm_clusters = rfm.copy()
rfm_clusters['Cluster'] = labels_km_rfm

# Calculate cluster means
cluster_summary = rfm_clusters.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(2)

# Add interpretation (optional)
cluster_summary['Interpretation'] = [
    'High recency, low frequency, low spend',      # Cluster 0
    'Low recency, high frequency, moderate spend',# Cluster 1
    'Moderate recency, moderate frequency, high spend'  # Cluster 2
]

# Reset index for nicer display
cluster_summary = cluster_summary.reset_index()

print(cluster_summary)


In [ ]:
import pandas as pd

# Assuming:
# rfm_scaled = standardized RFM dataframe (columns: 'Recency', 'Frequency', 'Monetary')
# labels_km_rfm = cluster labels from K-Means

# Create a copy of the standardized RFM data and add cluster labels
rfm_std_clusters = rfm_scaled.copy()
rfm_std_clusters['Cluster'] = labels_km_rfm

# Calculate cluster means (standardized values) and round for clarity
cluster_summary_std = rfm_std_clusters.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(2)

# Add interpretation (adjust descriptions if needed)
cluster_summary_std['Interpretation'] = [
    'High recency, low frequency, low spend',
    'Low recency, high frequency, moderate spend',
    'Moderate recency, moderate frequency, high spend'
]

# Reset index for nicer display
cluster_summary_std = cluster_summary_std.reset_index()

# Display table
print(cluster_summary_std)


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Assuming your RFM dataframe is called 'rfm'
# Columns: 'Recency', 'Frequency', 'Monetary'

# 1. Standardize the RFM data
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm[['Recency','Frequency','Monetary']]),
                          columns=['Recency','Frequency','Monetary'])

# 2. Fit K-Means on standardized data
kmeans = KMeans(n_clusters=3, random_state=42)
labels_km_rfm = kmeans.fit_predict(rfm_scaled)

# 3. Add cluster labels to standardized dataframe
rfm_std_clusters = rfm_scaled.copy()
rfm_std_clusters['Cluster'] = labels_km_rfm

# 4. Calculate cluster means (standardized) and round for clarity
cluster_summary_std = rfm_std_clusters.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(2)

# 5. Add interpretation (you can adjust based on your analysis)
cluster_summary_std['Interpretation'] = [
    'High recency, low frequency, low spend',
    'Low recency, high frequency, moderate spend',
    'Moderate recency, moderate frequency, high spend'
]

# Reset index for nicer display
cluster_summary_std = cluster_summary_std.reset_index()

# 6. Display the standardized cluster summary table
print(cluster_summary_std)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assuming 'rfm' is your RFM dataframe
X_rfm = rfm[['Recency', 'Frequency', 'Monetary']]

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)


In [ ]:
# Fit GMM
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)


In [ ]:
sil_gmm = silhouette_score(X_scaled, labels_gmm)
dbi_gmm = davies_bouldin_score(X_scaled, labels_gmm)

print(f"GMM Silhouette Score: {sil_gmm:.4f}")
print(f"GMM Davies–Bouldin Index: {dbi_gmm:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare DataFrame for plotting
plot_df_gmm = pd.DataFrame(X_scaled, columns=['Recency', 'Frequency', 'Monetary'])
plot_df_gmm['Cluster'] = labels_gmm.astype(str)  # labels as strings

# Plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_gmm,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('GMM Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Frequency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
rfm_std_clusters = pd.DataFrame(X_scaled, columns=['Recency','Frequency','Monetary'])
rfm_std_clusters['Cluster'] = labels_gmm
cluster_summary = rfm_std_clusters.groupby('Cluster').mean().round(2)
cluster_summary


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from matplotlib.patches import Ellipse

# Assuming standardized RFM data is in rfm_scaled (columns: 'Recency','Frequency','Monetary')
# And GMM is fitted with 3 components
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(rfm_scaled)

# Prepare plotting dataframe
plot_df = rfm_scaled.copy()
plot_df['Cluster'] = labels_gmm.astype(str)

# Scatter plot: Frequency vs Monetary
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

# Add ellipses to show Gaussian components
for i in range(3):
    mean = gmm.means_[i][[1,2]]  # Frequency & Monetary
    cov = gmm.covariances_[i][[1,2]][:,[1,2]]  # 2x2 covariance
    eigvals, eigvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:,0][::-1]))
    width, height = 2 * np.sqrt(eigvals)  # 2 standard deviations
    ellipse = Ellipse(mean, width, height, angle, edgecolor='black', fc='None', lw=2)
    plt.gca().add_patch(ellipse)

plt.title('GMM Clustering with Gaussian Ellipses (RFM)', fontsize=14)
plt.xlabel('Frequency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# Add ellipses to show Gaussian components
for i in range(3):
    mean = gmm.means_[i][[1,2]]  # Frequency & Monetary
    cov = gmm.covariances_[i][[1,2]][:,[1,2]]  # 2x2 covariance
    eigvals, eigvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:,0][::-1]))
    width, height = 2 * np.sqrt(eigvals)  # 2 standard deviations
    ellipse = Ellipse(xy=(mean[0], mean[1]), width=width, height=height,
                      angle=angle, edgecolor='black', fc='None', lw=2)
    plt.gca().add_patch(ellipse)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from matplotlib.patches import Ellipse

# Assuming standardized RFM data is in rfm_scaled (columns: 'Recency','Frequency','Monetary')
# And GMM is fitted with 3 components
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(rfm_scaled)

# Prepare plotting dataframe
plot_df = rfm_scaled.copy()
plot_df['Cluster'] = labels_gmm.astype(str)

# Scatter plot: Frequency vs Monetary
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

# Add ellipses to show Gaussian components
for i in range(3):
    mean = gmm.means_[i][[1,2]]  # Frequency & Monetary
    cov = gmm.covariances_[i][[1,2]][:,[1,2]]  # 2x2 covariance
    eigvals, eigvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:,0][::-1]))
    width, height = 2 * np.sqrt(eigvals)  # 2 standard deviations
    ellipse = Ellipse(mean, width, height, angle, edgecolor='black', fc='None', lw=2)
    plt.gca().add_patch(ellipse)

plt.title('GMM Clustering with Gaussian Ellipses (RFM)', fontsize=14)
plt.xlabel('Frequency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd

# Assuming:
# rfm_scaled is your standardized RFM dataframe with columns: 'Recency', 'Frequency', 'Monetary'
# labels_gmm is your cluster labels from GMM

# Prepare DataFrame for plotting
plot_df_gmm = rfm_scaled.copy()
plot_df_gmm['Cluster'] = labels_gmm.astype(str)  # convert to string for color mapping

# Create 3D scatter plot
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

# Scatter points
for cluster in plot_df_gmm['Cluster'].unique():
    cluster_data = plot_df_gmm[plot_df_gmm['Cluster'] == cluster]
    ax.scatter(
        cluster_data['Recency'],
        cluster_data['Frequency'],
        cluster_data['Monetary'],
        label=f'Cluster {cluster}',
        s=50,       # point size
        alpha=0.7   # transparency
    )

# Labels and title
ax.set_xlabel('Recency (standardized)')
ax.set_ylabel('Frequency (standardized)')
ax.set_zlabel('Monetary (standardized)')
ax.set_title('GMM Clustering on Standardized RFM Data (3D)', fontsize=14)
ax.legend(title='Cluster', loc='upper right')

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assuming your RFM dataframe is named 'rfm' with columns: 'Recency', 'Frequency', 'Monetary'

# Standardize RFM
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency','Frequency','Monetary'])

# BIRCH clustering
birch = Birch(n_clusters=3)
labels_birch = birch.fit_predict(rfm_scaled_df)

# Add cluster labels
rfm_scaled_df['Cluster'] = labels_birch.astype(str)

# Metrics
sil_birch = silhouette_score(rfm_scaled_df[['Recency','Frequency','Monetary']], labels_birch)
dbi_birch = davies_bouldin_score(rfm_scaled_df[['Recency','Frequency','Monetary']], labels_birch)

print(f"BIRCH Silhouette Score: {sil_birch:.4f}")
print(f"BIRCH Davies–Bouldin Index: {dbi_birch:.4f}")

# Plot (2D: Frequency vs Monetary)
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=rfm_scaled_df,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('BIRCH Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Frequency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assuming your RFM DataFrame is named 'rfm' with columns: 'Recency', 'Frequency', 'Monetary'

# Standardize RFM
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm), columns=rfm.columns)

# Fit BIRCH
birch = Birch(n_clusters=3)
labels_birch = birch.fit_predict(rfm_scaled)

# Add cluster labels to standardized RFM
rfm_std_clusters = rfm_scaled.copy()
rfm_std_clusters['Cluster'] = labels_birch

# Cluster summary (means of standardized RFM)
cluster_summary = rfm_std_clusters.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    "High recency, low frequency, low spend",
    "Low recency, moderate frequency, low spend",
    "Moderate recency, high frequency, high spend"
]

print("Cluster Summary (Standardized RFM):")
print(cluster_summary)

# Cluster results table
sil_score = silhouette_score(rfm_scaled, labels_birch)
dbi_score = davies_bouldin_score(rfm_scaled, labels_birch)
num_clusters = len(set(labels_birch))

cluster_results = pd.DataFrame({
    'Model': ['BIRCH'],
    'Number of Clusters': [num_clusters],
    'Silhouette Score': [round(sil_score, 4)],
    'Davies–Bouldin Index': [round(dbi_score, 4)]
})

print("\nCluster Results Table:")
print(cluster_results)


In [ ]:
import pandas as pd

# Assume:
# rfm_scaled -> standardized RFM dataframe with columns ['Recency', 'Frequency', 'Monetary']
# labels_birch -> cluster labels from BIRCH

# 1. Create a copy of standardized RFM and add cluster labels
rfm_std_clusters = rfm_scaled.copy()
rfm_std_clusters['Cluster'] = labels_birch

# 2. Compute cluster means (standardized values)
cluster_summary_std = rfm_std_clusters.groupby('Cluster').mean().round(2)

# 3. Add interpretation column manually
interpretations = {
    0: 'High recency, low frequency, low spend',
    1: 'Low recency, moderate frequency, low spend',
    2: 'Moderate recency, high frequency, high spend'
}

cluster_summary_std['Interpretation'] = cluster_summary_std.index.map(interpretations)

# 4. Reset index for nicer display
cluster_summary_std = cluster_summary_std.reset_index()

# Display the table
print(cluster_summary_std)


In [ ]:
cluster_summary_std = cluster_summary_std.drop(columns=['CustomerID'])


In [ ]:

# Assume:
# rfm_scaled -> standardized RFM dataframe with columns ['Recency', 'Frequency', 'Monetary']
# labels_birch -> cluster labels from BIRCH

# 1. Create a copy of standardized RFM and add cluster labels
rfm_std_clusters = rfm_scaled.copy()
rfm_std_clusters['Cluster'] = labels_birch

# 2. Compute cluster means (standardized values)
cluster_summary_std = rfm_std_clusters.groupby('Cluster').mean().round(2)

# 3. Add interpretation column manually
interpretations = {
    0: 'High recency, low frequency, low spend',
    1: 'Low recency, moderate frequency, low spend',
    2: 'Moderate recency, high frequency, high spend'
}

cluster_summary_std['Interpretation'] = cluster_summary_std.index.map(interpretations)

# 4. Reset index for nicer display
cluster_summary_std = cluster_summary_std.reset_index()

# Display the table
print(cluster_summary_std)

In [ ]:
cluster_summary_std = cluster_summary_std.drop(columns=['CustomerID'])

In [ ]:
# Assume:
# rfm_scaled -> standardized RFM dataframe with columns ['Recency', 'Frequency', 'Monetary']
# labels_birch -> cluster labels from BIRCH

# 1. Create a copy of standardized RFM and add cluster labels
rfm_std_clusters = rfm_scaled.copy()
rfm_std_clusters['Cluster'] = labels_birch

# 2. Compute cluster means (standardized values)
cluster_summary_std = rfm_std_clusters.groupby('Cluster').mean().round(2)

# 3. Add interpretation column manually
interpretations = {
    0: 'High recency, low frequency, low spend',
    1: 'Low recency, moderate frequency, low spend',
    2: 'Moderate recency, high frequency, high spend'
}

cluster_summary_std['Interpretation'] = cluster_summary_std.index.map(interpretations)
cluster_summary_std = cluster_summary_std.drop(columns=['CustomerID'])

# 4. Reset index for nicer display
cluster_summary_std = cluster_summary_std.reset_index()

# Display the table
print(cluster_summary_std)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import numpy as np

# Assuming your standardized RFM data is in rfm_scaled (columns: 'Recency', 'Frequency', 'Monetary')

# Fit DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=10)  # adjust eps and min_samples as needed
labels_db = dbscan.fit_predict(rfm_scaled)

# Prepare DataFrame for plotting
plot_df_db = rfm_scaled.copy()
plot_df_db['Cluster'] = labels_db.astype(str)  # convert to string for seaborn

# Set noise points label to '-1' (already labeled by DBSCAN)
plot_df_db['Cluster'] = plot_df_db['Cluster'].replace('-1', 'Noise')

# Plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Recency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Tune DBSCAN parameters
dbscan = DBSCAN(eps=1.0, min_samples=5)  # smaller eps to merge points less aggressively
labels_db = dbscan.fit_predict(rfm_scaled)

# Check how many clusters (excluding noise)
unique_labels = set(labels_db)
num_clusters = len(unique_labels) - (1 if -1 in labels_db else 0)
num_noise = list(labels_db).count(-1)

print(f"Number of clusters: {num_clusters}, Number of noise points: {num_noise}")

# Only calculate Silhouette and DBI if at least 2 clusters exist
mask = labels_db != -1  # exclude noise
if num_clusters >= 2:
    sil_db = silhouette_score(rfm_scaled[mask], labels_db[mask])
    dbi_db = davies_bouldin_score(rfm_scaled[mask], labels_db[mask])
    print(f"Silhouette Score: {sil_db:.4f}")
    print(f"Davies–Bouldin Index: {dbi_db:.4f}")
else:
    print("Too few clusters for Silhouette Score/DBI calculation. Adjust eps or min_samples.")


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import numpy as np

# Fit DBSCAN with tuned parameters
dbscan = DBSCAN(eps=0.9, min_samples=5)  # tuned for standardized RFM
labels_db = dbscan.fit_predict(rfm_scaled)

# Count clusters and noise
unique_labels = set(labels_db)
num_clusters = len(unique_labels) - (1 if -1 in labels_db else 0)
num_noise = list(labels_db).count(-1)
print(f"Number of clusters: {num_clusters}, Number of noise points: {num_noise}")

# Calculate metrics if at least 2 clusters
mask = labels_db != -1  # exclude noise
if num_clusters >= 2:
    sil_db = silhouette_score(rfm_scaled[mask], labels_db[mask])
    dbi_db = davies_bouldin_score(rfm_scaled[mask], labels_db[mask])
    print(f"DBSCAN Silhouette Score: {sil_db:.4f}")
    print(f"DBSCAN Davies–Bouldin Index: {dbi_db:.4f}")
else:
    print("Too few clusters for metrics calculation. Adjust eps or min_samples.")

# Prepare DataFrame for plotting
plot_df_db = rfm_scaled.copy()
plot_df_db['Cluster'] = labels_db.astype(str)
plot_df_db['Cluster'] = plot_df_db['Cluster'].replace('-1', 'Noise')

# Plot DBSCAN clusters
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Recency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# Try larger eps for DBSCAN
dbscan = DBSCAN(eps=2.0, min_samples=5)  # increase eps
labels_db = dbscan.fit_predict(rfm_scaled)

# Count clusters and noise
unique_labels = set(labels_db)
num_clusters = len(unique_labels) - (1 if -1 in labels_db else 0)
num_noise = list(labels_db).count(-1)
print(f"Number of clusters: {num_clusters}, Number of noise points: {num_noise}")

# Compute metrics if >=2 clusters
mask = labels_db != -1
if num_clusters >= 2:
    sil_db = silhouette_score(rfm_scaled[mask], labels_db[mask])
    dbi_db = davies_bouldin_score(rfm_scaled[mask], labels_db[mask])
    print(f"DBSCAN Silhouette Score: {sil_db:.4f}")
    print(f"DBSCAN Davies-Bouldin Index: {dbi_db:.4f}")
else:
    print("Still too few clusters. Try increasing eps further.")

# Prepare DataFrame for plotting
plot_df_db = rfm_scaled.copy()
plot_df_db['Cluster'] = labels_db.astype(str)
plot_df_db['Cluster'] = plot_df_db['Cluster'].replace('-1', 'Noise')

# Plot
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Recency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

# Try eps from 2.0 to 5.0 in increments
for eps_val in np.arange(2.0, 5.1, 0.25):
    dbscan = DBSCAN(eps=eps_val, min_samples=5)
    labels_db = dbscan.fit_predict(rfm_scaled)
    num_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    num_noise = list(labels_db).count(-1)
    print(f"eps={eps_val:.2f} -> clusters={num_clusters}, noise={num_noise}")
    if num_clusters >= 2:
        mask = labels_db != -1
        sil_db = silhouette_score(rfm_scaled[mask], labels_db[mask])
        dbi_db = davies_bouldin_score(rfm_scaled[mask], labels_db[mask])
        print(f"  Silhouette: {sil_db:.4f}, DBI: {dbi_db:.4f}")
        break


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

# X_scaled is your standardized RFM data
eps_value = 8  # increase until DBSCAN forms multiple clusters
min_samples = 5

db = DBSCAN(eps=eps_value, min_samples=min_samples)
labels_db = db.fit_predict(X_scaled)

# Only keep non-noise points for metrics
mask = labels_db != -1
if len(np.unique(labels_db[mask])) >= 2:
    sil_db = silhouette_score(X_scaled[mask], labels_db[mask])
    dbi_db = davies_bouldin_score(X_scaled[mask], labels_db[mask])
    print("DBSCAN Silhouette Score:", round(sil_db, 3))
    print("DBSCAN Davies-Bouldin Index:", round(dbi_db, 3))
else:
    print("Too few clusters for metrics calculation")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns

# Assume 'rfm' is your RFM dataframe
rfm_scaled = StandardScaler().fit_transform(rfm[['Recency','Frequency','Monetary']])

# Tune eps manually until at least 2 clusters form
eps_value = 6.0  # you can increase/decrease to get 2-3 clusters
min_samples = 5

dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
labels_db = dbscan.fit_predict(rfm_scaled)

# Count clusters and noise
unique_labels = np.unique(labels_db)
n_clusters = len(unique_labels) - (1 if -1 in labels_db else 0)
n_noise = np.sum(labels_db == -1)

print(f"DBSCAN Number of clusters: {n_clusters}, Noise points: {n_noise}")

# Only calculate metrics if 2 or more clusters
if n_clusters >= 2:
    mask = labels_db != -1  # exclude noise
    sil_score = silhouette_score(rfm_scaled[mask], labels_db[mask])
    dbi_score = davies_bouldin_score(rfm_scaled[mask], labels_db[mask])
    print(f"DBSCAN Silhouette Score: {sil_score:.3f}")
    print(f"DBSCAN Davies-Bouldin Index: {dbi_score:.3f}")
else:
    print("Too few clusters for Silhouette Score/DBI calculation.")

# Plot
plot_df = pd.DataFrame(rfm_scaled, columns=['Recency','Frequency','Monetary'])
plot_df['Cluster'] = labels_db.astype(str)

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RFM')
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your RFM data is in 'rfm' with columns ['Recency','Frequency','Monetary']

# 1. Standardize RFM
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

# 2. Try DBSCAN with increasing eps until we get 2–3 clusters
# You can adjust min_samples if needed
eps_values = np.arange(5, 15, 0.5)
min_samples = 5

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=min_samples)
    labels = db.fit_predict(rfm_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    print(f"eps={eps:.2f} -> clusters={n_clusters}, noise={n_noise}")
    if n_clusters >= 2:  # stop when we have at least 2 clusters
        break

# 3. Calculate metrics (exclude noise points)
mask = labels != -1
sil = silhouette_score(rfm_scaled[mask], labels[mask])
dbi = davies_bouldin_score(rfm_scaled[mask], labels[mask])
print(f"\nDBSCAN Silhouette Score: {sil:.3f}")
print(f"DBSCAN Davies–Bouldin Index: {dbi:.3f}")

# 4. Create DataFrame with cluster labels
rfm_db = rfm.copy()
rfm_db['Cluster'] = labels

# 5. Cluster summary (mean values in standardized RFM)
rfm_std = pd.DataFrame(rfm_scaled, columns=['Recency','Frequency','Monetary'])
rfm_std['Cluster'] = labels
cluster_summary = rfm_std.groupby('Cluster').mean().round(2)
print("\nCluster Summary (Standardized RFM):")
print(cluster_summary)

# 6. Optional: scatter plot (2D PCA projection)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)
plot_df = pd.DataFrame(rfm_pca, columns=['PC1','PC2'])
plot_df['Cluster'] = labels.astype(str)

plt.figure(figsize=(8,6))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='Cluster', palette='Set2', s=50, alpha=0.7, edgecolor='w')
plt.title('DBSCAN Clustering on Standardized RFM')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Sample a subset of your RFM data
sample_size = 3000  # adjust if needed
rfm_sample = rfm.sample(n=sample_size, random_state=42)

# 2. Standardize
scaler = StandardScaler()
rfm_scaled_sample = scaler.fit_transform(rfm_sample[['Recency','Frequency','Monetary']])

# 3. Tune DBSCAN
eps = 6.0      # you may adjust
min_samples = 5
db = DBSCAN(eps=eps, min_samples=min_samples)
labels = db.fit_predict(rfm_scaled_sample)

# 4. Check number of clusters and noise
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
print(f"Number of clusters: {n_clusters}, Number of noise points: {n_noise}")

# 5. Calculate Silhouette and DBI if at least 2 clusters
mask = labels != -1
if n_clusters >= 2:
    sil = silhouette_score(rfm_scaled_sample[mask], labels[mask])
    dbi = davies_bouldin_score(rfm_scaled_sample[mask], labels[mask])
    print(f"DBSCAN Silhouette Score: {sil:.3f}")
    print(f"DBSCAN Davies–Bouldin Index: {dbi:.3f}")
else:
    print("Too few clusters for Silhouette Score/DBI calculation.")

# 6. Cluster summary
rfm_std = pd.DataFrame(rfm_scaled_sample, columns=['Recency','Frequency','Monetary'])
rfm_std['Cluster'] = labels
cluster_summary = rfm_std.groupby('Cluster').mean().round(2)
print("\nCluster Summary (Standardized RFM):")
print(cluster_summary)

# 7. 2D scatter plot using PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled_sample)
plot_df = pd.DataFrame(rfm_pca, columns=['PC1','PC2'])
plot_df['Cluster'] = labels.astype(str)

plt.figure(figsize=(8,6))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='Cluster', palette='Set2',
                s=50, alpha=0.7, edgecolor='w')
plt.title('DBSCAN Clustering on Standardized RFM (Sample)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN

# Sample 2000 points for visualization and computation feasibility
rfm_sample = rfm_scaled.sample(n=2000, random_state=42)

# Fit DBSCAN
db = DBSCAN(eps=5, min_samples=5)
labels = db.fit_predict(rfm_sample)

# Prepare plotting DataFrame
plot_df_db = rfm_sample.copy()
plot_df_db['Cluster'] = labels.astype(str)  # noise = -1

# Journal-style scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('DBSCAN Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Frequency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cluster import DBSCAN

# Convert scaled array to DataFrame
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

# Sample 2000 points for visualization
rfm_sample = rfm_scaled_df.sample(n=2000, random_state=42)

# Fit DBSCAN
db = DBSCAN(eps=5, min_samples=5)
labels = db.fit_predict(rfm_sample)

# Prepare plotting DataFrame
plot_df_db = rfm_sample.copy()
plot_df_db['Cluster'] = labels.astype(str)  # noise = -1

# Scatter plot
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_db,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('DBSCAN Clustering on Standardized RFM Data', fontsize=14)
plt.xlabel('Frequency (standardized)', fontsize=12)
plt.ylabel('Monetary (standardized)', fontsize=12)
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

# Assuming rfm_scaled is your standardized RFM numpy array
# Sample 20,000 points
if rfm_scaled.shape[0] > 20000:
    rfm_sample_20k = rfm_scaled[np.random.choice(rfm_scaled.shape[0], 20000, replace=False)]
else:
    rfm_sample_20k = rfm_scaled

# Fit DBSCAN
db = DBSCAN(eps=5, min_samples=5)  # You might need to tune eps
labels_20k = db.fit_predict(rfm_sample_20k)

# Count clusters and noise
n_clusters = len(set(labels_20k)) - (1 if -1 in labels_20k else 0)
n_noise = list(labels_20k).count(-1)

print(f"DBSCAN Number of clusters: {n_clusters}, Noise points: {n_noise}")


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sample 20,000 points (or all if < 20k)
if rfm_scaled.shape[0] > 20000:
    rfm_sample_20k = rfm_scaled[np.random.choice(rfm_scaled.shape[0], 20000, replace=False)]
else:
    rfm_sample_20k = rfm_scaled

# Automatically tune eps to get ~3 clusters
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rfm_sample_20k)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2 and n_clusters <= 4:  # target around 3 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# Fit final DBSCAN
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rfm_sample_20k)

# Prepare DataFrame for plotting
plot_df_db = pd.DataFrame(rfm_sample_20k, columns=['Recency','Frequency','Monetary'])
plot_df_db['Cluster'] = labels_final.astype(str)

# Plot (2D: Frequency vs Monetary)
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RFM (20k sample)', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Mask to exclude noise points (-1)
mask = labels_final != -1
filtered_labels = labels_final[mask]
filtered_data = rfm_sample_20k[mask]

# Silhouette Score
sil_score = silhouette_score(filtered_data, filtered_labels)
# Davies-Bouldin Index
dbi_score = davies_bouldin_score(filtered_data, filtered_labels)

print(f"DBSCAN Silhouette Score: {sil_score:.4f}")
print(f"DBSCAN Davies-Bouldin Index: {dbi_score:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

# Assume your full RFM dataframe is called 'rfm' with columns: 'Recency', 'Frequency', 'Monetary'

# -------------------------------
# 1. Standardize the RFM data
# -------------------------------
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

# -------------------------------
# 2. Sample 20,000 points for DBSCAN
# -------------------------------
if rfm_scaled.shape[0] > 20000:
    idx = np.random.choice(rfm_scaled.shape[0], 20000, replace=False)
    rfm_sample = rfm_scaled[idx]
else:
    rfm_sample = rfm_scaled

# -------------------------------
# 3. Fit DBSCAN
# -------------------------------
dbscan = DBSCAN(eps=5, min_samples=5)
labels_db = dbscan.fit_predict(rfm_sample)

# -------------------------------
# 4. Create cluster summary table
# -------------------------------
rfm_clusters = pd.DataFrame(rfm_sample, columns=['Recency','Frequency','Monetary'])
rfm_clusters['Cluster'] = labels_db

# Calculate mean of standardized RFM values for each cluster
cluster_summary = rfm_clusters.groupby('Cluster').mean().round(2)

# -------------------------------
# 5. Optional: add interpretation
# -------------------------------
def interpret_cluster(row):
    recency, freq, monetary = row['Recency'], row['Frequency'], row['Monetary']
    if recency > 0 and freq < 0 and monetary < 0:
        return 'High recency, low frequency, low spend'
    elif recency < 0 and freq > 0 and monetary > 0:
        return 'Low recency, high frequency, high spend'
    else:
        return 'Moderate/other behavior'

cluster_summary['Interpretation'] = cluster_summary.apply(interpret_cluster, axis=1)

print("DBSCAN Cluster Summary (Standardized RFM):\n")
print(cluster_summary)


In [ ]:
import pandas as pd
import numpy as np

# rfm_sample: standardized RFM data (used in the plot)
# labels_db: DBSCAN cluster labels from the plot

# Create a DataFrame from the standardized RFM sample
rfm_clusters = pd.DataFrame(rfm_sample, columns=['Recency','Frequency','Monetary'])
rfm_clusters['Cluster'] = labels_db

# Calculate mean standardized RFM values per cluster
cluster_summary = rfm_clusters.groupby('Cluster').mean().round(2)

# Optional: add interpretation
def interpret_cluster(row):
    recency, freq, monetary = row['Recency'], row['Frequency'], row['Monetary']
    if row.name == -1:
        return 'Noise points / outliers'
    if recency > 0 and freq < 0 and monetary < 0:
        return 'High recency, low frequency, low spend'
    elif recency < 0 and freq > 0 and monetary > 0:
        return 'Low recency, high frequency, high spend'
    else:
        return 'Moderate/other behavior'

cluster_summary['Interpretation'] = cluster_summary.apply(interpret_cluster, axis=1)

print("DBSCAN Cluster Summary (Standardized RFM - Plot Data):\n")
print(cluster_summary)


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sample 20,000 points (or all if < 20k)
if rfm_scaled.shape[0] > 20000:
    rfm_sample_20k = rfm_scaled[np.random.choice(rfm_scaled.shape[0], 20000, replace=False)]
else:
    rfm_sample_20k = rfm_scaled

# Automatically tune eps to get ~3 clusters
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rfm_sample_20k)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2 and n_clusters <= 4:  # target around 3 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# Fit final DBSCAN
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rfm_sample_20k)

# Prepare DataFrame for plotting
plot_df_db = pd.DataFrame(rfm_sample_20k, columns=['Recency','Frequency','Monetary'])
plot_df_db['Cluster'] = labels_final.astype(str)

# Plot (2D: Frequency vs Monetary)
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RFM (20k sample)', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Calculate mean standardized RFM per cluster
cluster_summary_db = plot_df_db.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(2)

# Add an interpretation column (manual description based on the means)
def interpret_db(row):
    recency = 'High' if row['Recency'] > 0 else 'Low'
    frequency = 'High' if row['Frequency'] > 0 else 'Low'
    monetary = 'High' if row['Monetary'] > 0 else 'Low'
    return f"{recency} recency, {frequency} frequency, {monetary} spend"

cluster_summary_db['Interpretation'] = cluster_summary_db.apply(interpret_db, axis=1)

# Reset index for nicer display
cluster_summary_db = cluster_summary_db.reset_index()
cluster_summary_db


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

# Standardized RFM dataset
# Assuming rfm_scaled is your standardized RFM numpy array
# Columns: ['Recency', 'Frequency', 'Monetary']

# Fit Hierarchical Clustering
hier = AgglomerativeClustering(n_clusters=3, affinity='euclidean', linkage='ward')
labels_hier = hier.fit_predict(rfm_scaled)

# Prepare DataFrame for plotting
plot_df_hier = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

# Plot 2D scatter (Frequency vs Monetary)
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_hier,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('Hierarchical Clustering (Agglomerative) on Standardized RFM', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import AgglomerativeClustering
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fit Hierarchical Clustering
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')  # 'ward' implies Euclidean distance
labels_hier = hier.fit_predict(rfm_scaled)

# Prepare DataFrame for plotting
plot_df_hier = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

# Plot 2D scatter (Frequency vs Monetary)
plt.figure(figsize=(8,6))
sns.set_style("whitegrid")
sns.scatterplot(
    data=plot_df_hier,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('Hierarchical Clustering (Agglomerative) on Standardized RFM', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import numpy as np

# Assuming:
# rfm_scaled -> standardized RFM numpy array
# labels_hier -> cluster labels from Agglomerative Clustering

# 1. Cluster Summary Table (Standardized RFM)
rfm_hier_df = pd.DataFrame(rfm_scaled, columns=['Recency','Frequency','Monetary'])
rfm_hier_df['Cluster'] = labels_hier

cluster_summary = rfm_hier_df.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'High recency, low frequency, low spend',
    'Low recency, high frequency, moderate spend',
    'Low recency, high frequency, high spend'
]
print("Cluster Summary (Standardized RFM):")
print(cluster_summary)

# 2. Silhouette Score and Davies-Bouldin Index
sil_score = silhouette_score(rfm_scaled, labels_hier)
dbi_score = davies_bouldin_score(rfm_scaled, labels_hier)
print(f"\nHierarchical Clustering Silhouette Score: {sil_score:.4f}")
print(f"Hierarchical Clustering Davies–Bouldin Index: {dbi_score:.4f}")

# 3. Cluster Results Table
cluster_results = pd.DataFrame({
    'Model': ['Hierarchical (Agglomerative)'],
    'Number of Clusters': [len(np.unique(labels_hier))],
    'Silhouette Score': [round(sil_score, 4)],
    'Davies–Bouldin Index': [round(dbi_score, 4)]
})
print("\nCluster Results Table:")
print(cluster_results)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create DataFrame for comparison
df_compare = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Silhouette Score': [0.5943, 0.5408, 0.9158, 0.8329, 0.5868],
    'Davies-Bouldin Index': [0.7102, 0.9516, 0.5395, 0.1880, 0.6990]
})

# Plot
fig, ax = plt.subplots(figsize=(10,6))

# Silhouette Score bars
ax.bar(df_compare['Model'], df_compare['Silhouette Score'], color='skyblue', label='Silhouette Score')

# Davies-Bouldin Index bars (overlaid)
ax.bar(df_compare['Model'], df_compare['Davies-Bouldin Index'], color='lightcoral', alpha=0.6, label='Davies-Bouldin Index')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparison of Clustering Models on Standardized RFM', fontsize=14)
ax.legend(loc='upper right', fontsize=10, title_fontsize=11)
plt.xticks(rotation=0)  # Keep x-labels straight
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics for standardized RFM
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.5943, 0.5408, 0.9158, 0.8329, 0.5868]
dbi_scores = [0.7102, 0.9516, 0.5395, 0.1880, 0.6990]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

# Create figure
fig, ax1 = plt.subplots(figsize=(10,6))

# Turn off grid completely
ax1.grid(False)

# Plot Silhouette Score
bar1 = ax1.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
# Plot Davies-Bouldin Index
bar2 = ax1.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Add labels and title
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparison of Clustering Models on Standardized RFM', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=0, fontsize=11)  # straight labels
ax1.legend(fontsize=11)

# Show values on top of bars
for rect in bar1 + bar2:
    height = rect.get_height()
    ax1.text(rect.get_x() + rect.get_width()/2., 1.01*height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------------------------
# 1. Select RF features
# -------------------------------------------------
rf = rfm[['Recency', 'Frequency']].copy()

# -------------------------------------------------
# 2. Standardize RF
# -------------------------------------------------
scaler = StandardScaler()
rf_scaled = scaler.fit_transform(rf)

# -------------------------------------------------
# 3. K-Means clustering (K = 3)
# -------------------------------------------------
kmeans_rf = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_rf = kmeans_rf.fit_predict(rf_scaled)

# -------------------------------------------------
# 4. Evaluation metrics
# -------------------------------------------------
sil_rf = silhouette_score(rf_scaled, labels_rf)
dbi_rf = davies_bouldin_score(rf_scaled, labels_rf)

print(f"RF K-Means Silhouette Score: {sil_rf:.4f}")
print(f"RF K-Means Davies–Bouldin Index: {dbi_rf:.4f}")

# -------------------------------------------------
# 5. Prepare DataFrame for plotting
# -------------------------------------------------
plot_df_rf = pd.DataFrame(
    rf_scaled,
    columns=['Recency (Std)', 'Frequency (Std)']
)
plot_df_rf['Cluster'] = labels_rf.astype(str)

# -------------------------------------------------
# 6. Plot: Recency vs Frequency
# -------------------------------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_rf,
    x='Recency (Std)',
    y='Frequency (Std)',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('K-Means Clustering on RF (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)', fontsize=12)
plt.ylabel('Frequency (standardized)', fontsize=12)
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------------------------------
# RF K-Means Cluster Summary (Standardized)
# -------------------------------------------------

# Create DataFrame with standardized RF and cluster labels
rf_std_clusters = pd.DataFrame(
    rf_scaled,
    columns=['Recency', 'Frequency']
)
rf_std_clusters['Cluster'] = labels_rf

# Compute cluster means
rf_cluster_summary = (
    rf_std_clusters
    .groupby('Cluster')
    .mean()
    .round(2)
    .reset_index()
)

rf_cluster_summary


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# Fit GMM
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm_rf = gmm.fit_predict(rfm_scaled[:, [0,1]])  # RF columns

# Prepare DataFrame for plotting
plot_df_gmm_rf = pd.DataFrame(rfm_scaled[:, [0,1]], columns=['Recency','Frequency'])
plot_df_gmm_rf['Cluster'] = labels_gmm_rf.astype(str)

# Silhouette Score & Davies-Bouldin Index
sil_gmm_rf = silhouette_score(rfm_scaled[:, [0,1]], labels_gmm_rf)
dbi_gmm_rf = davies_bouldin_score(rfm_scaled[:, [0,1]], labels_gmm_rf)

print(f"RF GMM Silhouette Score: {sil_gmm_rf:.4f}")
print(f"RF GMM Davies–Bouldin Index: {dbi_gmm_rf:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_gmm_rf,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('GMM Clustering on Standardized RF', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Create a DataFrame with standardized RF values and cluster labels
rf_scaled_df = pd.DataFrame(rfm_scaled[:, [0,1]], columns=['Recency','Frequency'])
rf_scaled_df['Cluster'] = labels_gmm_rf

# Calculate cluster means (standardized values)
cluster_summary_rf_gmm = rf_scaled_df.groupby('Cluster').mean().round(2)

# Add simple interpretations
interpretations = [
    "High recency, low frequency",
    "Low recency, low frequency",
    "Low recency, high frequency"
]

# Assign interpretations to clusters
cluster_summary_rf_gmm['Interpretation'] = interpretations

# Reset index for cleaner table
cluster_summary_rf_gmm = cluster_summary_rf_gmm.reset_index()
print(cluster_summary_rf_gmm)


In [ ]:
from sklearn.cluster import Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. Sample or use full RF dataset
# -----------------------------
# Assuming rfm_scaled contains standardized Recency and Frequency columns
if rfm_scaled.shape[0] > 20000:
    rf_sample = rfm_scaled[np.random.choice(rfm_scaled.shape[0], 20000, replace=False)]
else:
    rf_sample = rfm_scaled

# -----------------------------
# 2. Fit BIRCH
# -----------------------------
birch_model = Birch(n_clusters=3)
labels_birch = birch_model.fit_predict(rf_sample)

# -----------------------------
# 3. Prepare DataFrame for plotting
# -----------------------------
plot_df_birch = pd.DataFrame(rf_sample, columns=['Recency','Frequency'])
plot_df_birch['Cluster'] = labels_birch.astype(str)

# -----------------------------
# 4. Plot clusters (2D: Recency vs Frequency)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_birch,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('BIRCH Clustering on Standardized RF (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()

# -----------------------------
# 5. Calculate metrics
# -----------------------------
sil = silhouette_score(rf_sample, labels_birch)
dbi = davies_bouldin_score(rf_sample, labels_birch)

print(f"RF BIRCH Silhouette Score: {sil:.4f}")
print(f"RF BIRCH Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Cluster summary table
# -----------------------------
rf_df = pd.DataFrame(rf_sample, columns=['Recency','Frequency'])
rf_df['Cluster'] = labels_birch

cluster_summary = rf_df.groupby('Cluster').mean().round(2)
# Optional: add interpretation
cluster_summary['Interpretation'] = [
    'High recency, low frequency',
    'Low recency, low frequency',
    'Low recency, high frequency'
]
print("\nRF BIRCH Cluster Summary Table:")
print(cluster_summary)


In [ ]:
from sklearn.cluster import Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. Prepare RF data (only Recency & Frequency)
# -----------------------------
# Assuming rfm_scaled has columns: ['Recency','Frequency','Monetary']
rf_scaled = rfm_scaled[:, :2]  # take only Recency & Frequency

# Sample 20,000 if needed
if rf_scaled.shape[0] > 20000:
    rf_sample = rf_scaled[np.random.choice(rf_scaled.shape[0], 20000, replace=False)]
else:
    rf_sample = rf_scaled

# -----------------------------
# 2. Fit BIRCH
# -----------------------------
birch_model = Birch(n_clusters=3)
labels_birch = birch_model.fit_predict(rf_sample)

# -----------------------------
# 3. Prepare DataFrame for plotting
# -----------------------------
plot_df_birch = pd.DataFrame(rf_sample, columns=['Recency','Frequency'])
plot_df_birch['Cluster'] = labels_birch.astype(str)

# -----------------------------
# 4. Plot clusters
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_birch,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('BIRCH Clustering on Standardized RF (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()

# -----------------------------
# 5. Calculate metrics
# -----------------------------
sil = silhouette_score(rf_sample, labels_birch)
dbi = davies_bouldin_score(rf_sample, labels_birch)

print(f"RF BIRCH Silhouette Score: {sil:.4f}")
print(f"RF BIRCH Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Cluster summary table
# -----------------------------
rf_df = pd.DataFrame(rf_sample, columns=['Recency','Frequency'])
rf_df['Cluster'] = labels_birch

cluster_summary = rf_df.groupby('Cluster').mean().round(2)
# Optional: add interpretation
cluster_summary['Interpretation'] = [
    'High recency, low frequency',
    'Low recency, low frequency',
    'Low recency, high frequency'
]
print("\nRF BIRCH Cluster Summary Table:")
print(cluster_summary)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Sample the data (up to 20k)
# -----------------------------
if rfm_scaled.shape[0] > 20000:
    rf_sample = rfm_scaled[np.random.choice(rfm_scaled.shape[0], 20000, replace=False)]
else:
    rf_sample = rfm_scaled  # use all if less than 20k

# -----------------------------
# 2. Automatically tune eps for ~3 clusters
# -----------------------------
eps_values = np.arange(0.5, 10, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rf_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if 2 <= n_clusters <= 4:  # target 2-3 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 3. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rf_sample)

# -----------------------------
# 4. Metrics (exclude noise)
# -----------------------------
mask = labels_final != -1
if len(set(labels_final[mask])) > 1:
    sil = silhouette_score(rf_sample[mask], labels_final[mask])
    dbi = davies_bouldin_score(rf_sample[mask], labels_final[mask])
    print(f"RF DBSCAN Silhouette Score: {sil:.4f}")
    print(f"RF DBSCAN Davies-Bouldin Index: {dbi:.4f}")
else:
    print("Too few clusters for Silhouette/DBI calculation.")

# -----------------------------
# 5. Prepare DataFrame for plotting
# -----------------------------
plot_df_db = pd.DataFrame(rf_sample, columns=['Recency','Frequency'])
plot_df_db['Cluster'] = labels_final.astype(str)

# -----------------------------
# 6. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on Standardized RF Data', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()

# -----------------------------
# 7. Cluster summary table
# -----------------------------
rf_df = pd.DataFrame(rf_sample, columns=['Recency','Frequency'])
rf_df['Cluster'] = labels_final
cluster_summary = rf_df.groupby('Cluster').mean().round(2)

# Optional: add interpretation column
interpretation = {
    0: "High recency, low frequency",
    1: "Low recency, low frequency",
    2: "Low recency, high frequency",
    -1: "Noise / outliers"
}
cluster_summary['Interpretation'] = cluster_summary.index.map(lambda x: interpretation.get(x, ''))
print(cluster_summary)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# -----------------------------
# 1. Standardize RF data
# -----------------------------
# Assuming your RF dataframe is called 'rf' with columns ['Recency','Frequency']
scaler = StandardScaler()
rf_scaled = scaler.fit_transform(rf)

# -----------------------------
# 2. Sample 20,000 points for DBSCAN
# -----------------------------
if rf_scaled.shape[0] > 20000:
    rf_sample = rf_scaled[np.random.choice(rf_scaled.shape[0], 20000, replace=False)]
else:
    rf_sample = rf_scaled

# -----------------------------
# 3. Tune eps to get at least 2 clusters
# -----------------------------
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rf_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:  # want at least 2 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 4. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rf_sample)

# -----------------------------
# 5. Prepare DataFrame for plotting and summary
# -----------------------------
plot_df_db = pd.DataFrame(rf_sample[:, :2], columns=['Recency','Frequency'])
plot_df_db['Cluster'] = labels_final.astype(str)  # convert to string for hue

# Cluster summary table (standardized)
cluster_summary = plot_df_db.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Low recency, Low frequency' if c=='0' else
    'Low recency, High frequency' if c=='1' else
    'Noise / Outliers'
    for c in cluster_summary.index
]

print("\nRF DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 6. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RF DBSCAN Clustering (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Exclude noise points (-1) from metrics calculation
mask = labels_final != -1
if len(set(labels_final[mask])) > 1:  # make sure at least 2 clusters
    sil_score = silhouette_score(rf_sample[mask], labels_final[mask])
    dbi_score = davies_bouldin_score(rf_sample[mask], labels_final[mask])
    print(f"RF DBSCAN Silhouette Score: {sil_score:.4f}")
    print(f"RF DBSCAN Davies–Bouldin Index: {dbi_score:.4f}")
else:
    print("Too few clusters for Silhouette Score/DBI calculation.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assuming rfm_scaled is your standardized RF numpy array (Recency, Frequency)

# -----------------------------
# 1. Fit Hierarchical Clustering
# -----------------------------
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')  # 'affinity' removed for sklearn >=0.24
labels_hier = hier.fit_predict(rfm_scaled)

# -----------------------------
# 2. Calculate metrics
# -----------------------------
sil = silhouette_score(rfm_scaled, labels_hier)
dbi = davies_bouldin_score(rfm_scaled, labels_hier)
print(f"RF Hierarchical Silhouette Score: {sil:.4f}")
print(f"RF Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 3. Prepare DataFrame for plotting
# -----------------------------
plot_df_hier = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

# -----------------------------
# 4. Plot clusters (2D: Recency vs Frequency)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('Hierarchical Clustering on Standardized RF', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()

# -----------------------------
# 5. Create cluster summary table
# -----------------------------
rf_hier_clusters = pd.DataFrame(rfm_scaled, columns=['Recency','Frequency'])
rf_hier_clusters['Cluster'] = labels_hier

cluster_summary = rf_hier_clusters.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'High recency, low frequency',    # Cluster 0 (adjust based on actual values)
    'Low recency, low frequency',     # Cluster 1
    'Low recency, high frequency'     # Cluster 2
]
print(cluster_summary)


In [ ]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assume rfm_scaled has all 3 standardized RFM columns
# For RF combination, only take Recency and Frequency
rf_scaled = rfm_scaled[:, :2]  # first two columns: Recency, Frequency

# Fit Hierarchical Clustering
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hier = hier.fit_predict(rf_scaled)

# Calculate Silhouette Score and Davies-Bouldin Index
sil = silhouette_score(rf_scaled, labels_hier)
dbi = davies_bouldin_score(rf_scaled, labels_hier)
print(f"RF Hierarchical Silhouette Score: {sil:.4f}")
print(f"RF Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# Prepare DataFrame for plotting
plot_df_hier = pd.DataFrame(rf_scaled, columns=['Recency','Frequency'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

# Plot clusters (2D: Recency vs Frequency)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Recency',
    y='Frequency',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('Hierarchical Clustering on RF (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Frequency (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Assume rf_scaled = standardized RF data (Recency, Frequency)
# labels_hier = cluster labels from Hierarchical Clustering

# Create DataFrame with cluster labels
rf_hier_clusters = pd.DataFrame(rf_scaled, columns=['Recency','Frequency'])
rf_hier_clusters['Cluster'] = labels_hier

# Calculate cluster means (standardized values)
cluster_summary = rf_hier_clusters.groupby('Cluster')[['Recency','Frequency']].mean().round(2)

# Add interpretation column
def interpret_rf(row):
    recency = 'Low' if row['Recency'] < 0 else 'High'
    frequency = 'High' if row['Frequency'] > 0 else 'Low'
    return f"{recency} recency, {frequency} frequency"

cluster_summary['Interpretation'] = cluster_summary.apply(interpret_rf, axis=1)

print("RF Hierarchical Cluster Summary Table:")
print(cluster_summary)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 1. Comparison Table
# -----------------------------
rf_comparison = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Number of Clusters': [3, 3, 3, 3, 3],  # Replace with actual numbers if different
    'Silhouette Score': [0.6176, 0.4874, 0.8819, 0.8784, 0.5958],
    'Davies–Bouldin Index': [0.5467, 0.7353, 0.1823, 0.1233, 0.5306]
})

print(rf_comparison)

# -----------------------------
# 2. Bar Plot
# -----------------------------
models = rf_comparison['Model']
sil_scores = rf_comparison['Silhouette Score']
dbi_scores = rf_comparison['Davies–Bouldin Index']

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10,6))
ax.grid(False)  # remove grid

# Plot bars
bar1 = ax.bar(x - width/2, sil_scores, width, label='Silhouette Score', color='#4C72B0')
bar2 = ax.bar(x + width/2, dbi_scores, width, label='Davies-Bouldin Index', color='#55A868')

# Labels and title
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparison of Clustering Models (RF)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.legend(fontsize=11)

# Display values on top
for rect in bar1 + bar2:
    height = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2., 1.01*height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Extract Recency and Monetary only (RM)
# Index: 0 = Recency, 2 = Monetary
rm_scaled = rfm_scaled[:, [0, 2]]


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# K-Means with 3 clusters (consistent with methodology)
kmeans_rm = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_rm = kmeans_rm.fit_predict(rm_scaled)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df_rm = rm_df.copy()
plot_df_rm['Cluster'] = plot_df_rm['Cluster'].astype(str)

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_rm,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('K-Means Clustering on RM (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# Create DataFrame for summary
rm_df = pd.DataFrame(rm_scaled, columns=['Recency', 'Monetary'])
rm_df['Cluster'] = labels_rm

# Cluster means
cluster_summary_rm = (
    rm_df
    .groupby('Cluster')
    .mean()
    .round(2)
)

cluster_summary_rm


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df_rm = rm_df.copy()
plot_df_rm['Cluster'] = plot_df_rm['Cluster'].astype(str)

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_rm,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('K-Means Clustering on RM (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
sil_rm = silhouette_score(rm_scaled, labels_rm)
dbi_rm = davies_bouldin_score(rm_scaled, labels_rm)

print(f"RM K-Means Silhouette Score: {sil_rm:.4f}")
print(f"RM K-Means Davies–Bouldin Index: {dbi_rm:.4f}")


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1. Fit GMM (RM)
# -----------------------------
gmm_rm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    random_state=42
)

labels_gmm_rm = gmm_rm.fit_predict(rm_scaled)

# -----------------------------
# 2. Evaluation metrics
# -----------------------------
sil_rm_gmm = silhouette_score(rm_scaled, labels_gmm_rm)
dbi_rm_gmm = davies_bouldin_score(rm_scaled, labels_gmm_rm)

print(f"RM GMM Silhouette Score: {sil_rm_gmm:.4f}")
print(f"RM GMM Davies–Bouldin Index: {dbi_rm_gmm:.4f}")


In [ ]:
# -----------------------------
# 3. Cluster summary table
# -----------------------------
rm_gmm_df = pd.DataFrame(
    rm_scaled,
    columns=['Recency', 'Monetary']
)

rm_gmm_df['Cluster'] = labels_gmm_rm

cluster_summary_gmm_rm = (
    rm_gmm_df
    .groupby('Cluster')[['Recency', 'Monetary']]
    .mean()
)

cluster_summary_gmm_rm


In [ ]:
# -----------------------------
# 4. Plot clusters
# -----------------------------
plt.figure(figsize=(7,5))

for cluster in sorted(rm_gmm_df['Cluster'].unique()):
    subset = rm_gmm_df[rm_gmm_df['Cluster'] == cluster]
    plt.scatter(
        subset['Recency'],
        subset['Monetary'],
        label=f'Cluster {cluster}',
        alpha=0.6
    )

plt.xlabel('Recency')
plt.ylabel('Monetary')
plt.title('RM – GMM Clustering')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Copy dataframe
plot_df_rm = rm_df.copy()
plot_df_rm['Cluster'] = plot_df_rm['Cluster'].astype(str)

# Create figure
plt.figure(figsize=(8, 6))

# Plot each cluster separately
for cluster in plot_df_rm['Cluster'].unique():
    cluster_data = plot_df_rm[plot_df_rm['Cluster'] == cluster]
    plt.scatter(
        cluster_data['Recency'],
        cluster_data['Monetary'],
        label=f'Cluster {cluster}',
        s=50,
        alpha=0.7,
        edgecolors='white'
    )

# Titles and labels
plt.title('Clustering on RM (Standardized)', fontsize=14)
plt.xlabel('Recency (Standardized)', fontsize=12)
plt.ylabel('Monetary (Standardized)', fontsize=12)

# Legend
plt.legend(title='Cluster')

# Layout
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming `rm_birch_df` has 'Recency', 'Monetary', 'Cluster' columns
plot_df_birch = rm_birch_df.copy()
plot_df_birch['Cluster'] = plot_df_birch['Cluster'].astype(str)

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_birch,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('BIRCH Clustering on RM (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import Birch

# -----------------------------
# 1. Assume RM data is standardized
# Columns: 'Recency', 'Monetary'
# rm_scaled = your standardized RM numpy array
# -----------------------------

# Fit BIRCH
birch_model = Birch(n_clusters=3)
labels_birch = birch_model.fit_predict(rm_scaled)

# Create DataFrame with cluster labels
rm_birch_df = pd.DataFrame(rm_scaled, columns=['Recency','Monetary'])
rm_birch_df['Cluster'] = labels_birch

# -----------------------------
# 2. Cluster summary table
# -----------------------------
cluster_summary = rm_birch_df.groupby('Cluster')[['Recency','Monetary']].mean().round(2)
cluster_summary['Interpretation'] = [
    'High recency, Low monetary spend',
    'Low recency, High monetary spend',
    'Low recency, Moderate monetary'
]
print(cluster_summary)

# -----------------------------
# 3. Plot clusters
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=rm_birch_df,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('BIRCH Clustering on RM (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()

# -----------------------------
# 4. Silhouette Score and DBI
# -----------------------------
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil = silhouette_score(rm_scaled, labels_birch)
dbi = davies_bouldin_score(rm_scaled, labels_birch)
print(f"RM BIRCH Silhouette Score: {sil:.4f}")
print(f"RM BIRCH Davies–Bouldin Index: {dbi:.4f}")


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assume rm_scaled is your standardized RM numpy array with columns [Recency, Monetary]

# -----------------------------
# 1. Tune DBSCAN eps to get ~3 clusters
# -----------------------------
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2 and n_clusters <= 4:  # target 2–3 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 2. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rm_scaled)

# -----------------------------
# 3. Metrics (exclude noise for Silhouette/DBI)
# -----------------------------
from sklearn.metrics import silhouette_score, davies_bouldin_score

mask = labels_final != -1
if len(set(labels_final[mask])) > 1:
    sil = silhouette_score(rm_scaled[mask], labels_final[mask])
    dbi = davies_bouldin_score(rm_scaled[mask], labels_final[mask])
    print(f"RM DBSCAN Silhouette Score: {sil:.4f}")
    print(f"RM DBSCAN Davies–Bouldin Index: {dbi:.4f}")
else:
    sil, dbi = None, None
    print("Too few clusters for metrics calculation.")

# -----------------------------
# 4. Prepare DataFrame for plotting
# -----------------------------
rm_db_df = pd.DataFrame(rm_scaled, columns=['Recency','Monetary'])
rm_db_df['Cluster'] = labels_final.astype(str)

# -----------------------------
# 5. Plot DBSCAN clusters
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=rm_db_df,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on RM (Standardized)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()

# -----------------------------
# 6. Cluster Summary Table
# -----------------------------
cluster_summary = pd.DataFrame(rm_scaled, columns=['Recency','Monetary'])
cluster_summary['Cluster'] = labels_final
cluster_summary = cluster_summary.groupby('Cluster').mean().round(2)

# Add interpretations manually based on standardized values
cluster_summary['Interpretation'] = [
    'Low recency, Low monetary spend',   # Cluster 0 (example)
    'Low recency, High monetary spend',  # Cluster 1
    'Noise / Outliers'                   # Cluster -1 if present
]

print(cluster_summary)


In [ ]:
best_eps = None
best_clusters = 0
closest_diff = 10  # start with a large number

for eps in np.arange(1.0, 10.0, 0.25):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:
        diff = abs(n_clusters - 3)  # we want ~3 clusters
        if diff < closest_diff:
            closest_diff = diff
            best_eps = eps
            best_clusters = n_clusters

if best_eps is None:
    print("No eps produced ≥ 2 clusters. You may need to increase the eps range.")
else:
    print(f"Chosen eps={best_eps}, clusters={best_clusters}")


In [ ]:
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rm_scaled)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Standardize RM data
# -----------------------------
# Assuming your RM dataframe is called 'rm' with columns ['Recency','Monetary']
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 2. Sample 20,000 points for DBSCAN
# -----------------------------
if rm_scaled.shape[0] > 20000:
    rm_sample = rm_scaled[np.random.choice(rm_scaled.shape[0], 20000, replace=False)]
else:
    rm_sample = rm_scaled

# -----------------------------
# 3. Tune eps to get at least 2 clusters
# -----------------------------
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:  # want at least 2 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 4. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rm_sample)

# -----------------------------
# 5. Compute metrics (exclude noise points)
# -----------------------------
mask = labels_final != -1
if len(set(labels_final[mask])) >= 2:  # ensure at least 2 clusters for metrics
    sil = silhouette_score(rm_sample[mask], labels_final[mask])
    dbi = davies_bouldin_score(rm_sample[mask], labels_final[mask])
else:
    sil = np.nan
    dbi = np.nan

print(f"RM DBSCAN Silhouette Score: {sil:.4f}")
print(f"RM DBSCAN Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Prepare cluster summary table
# -----------------------------
plot_df_db = pd.DataFrame(rm_sample[:, :2], columns=['Recency','Monetary'])
plot_df_db['Cluster'] = labels_final.astype(str)

cluster_summary = plot_df_db.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Low recency, Low monetary' if c=='0' else
    'Low recency, High monetary' if c=='1' else
    'Noise / Outliers'
    for c in cluster_summary.index
]

print("\nRM DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters (2D: Recency vs Monetary)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM DBSCAN Clustering (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Extract RM columns from your main dataset
rm = df[['Recency', 'Monetary']].copy()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Standardize RM data
# -----------------------------
# Assuming your RM dataframe is called 'rm' with columns ['Recency','Monetary']
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 2. Sample 20,000 points for DBSCAN
# -----------------------------
if rm_scaled.shape[0] > 20000:
    rm_sample = rm_scaled[np.random.choice(rm_scaled.shape[0], 20000, replace=False)]
else:
    rm_sample = rm_scaled

# -----------------------------
# 3. Tune eps to get at least 2 clusters
# -----------------------------
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:  # want at least 2 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 4. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rm_sample)

# -----------------------------
# 5. Compute metrics (exclude noise points)
# -----------------------------
mask = labels_final != -1
if len(set(labels_final[mask])) >= 2:  # ensure at least 2 clusters for metrics
    sil = silhouette_score(rm_sample[mask], labels_final[mask])
    dbi = davies_bouldin_score(rm_sample[mask], labels_final[mask])
else:
    sil = np.nan
    dbi = np.nan

print(f"RM DBSCAN Silhouette Score: {sil:.4f}")
print(f"RM DBSCAN Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Prepare cluster summary table
# -----------------------------
plot_df_db = pd.DataFrame(rm_sample[:, :2], columns=['Recency','Monetary'])
plot_df_db['Cluster'] = labels_final.astype(str)

cluster_summary = plot_df_db.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Low recency, Low monetary' if c=='0' else
    'Low recency, High monetary' if c=='1' else
    'Noise / Outliers'
    for c in cluster_summary.index
]

print("\nRM DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters (2D: Recency vs Monetary)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM DBSCAN Clustering (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 0. Prepare RM data
# -----------------------------
# Replace 'df' with your main dataset
rm = df[['Recency', 'Monetary']].copy()

# -----------------------------
# 1. Standardize RM data
# -----------------------------
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 2. Sample 20,000 points if dataset is large
# -----------------------------
if rm_scaled.shape[0] > 20000:
    rm_sample = rm_scaled[np.random.choice(rm_scaled.shape[0], 20000, replace=False)]
else:
    rm_sample = rm_scaled

# -----------------------------
# 3. Tune eps to get at least 2 clusters
# -----------------------------
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:  # want at least 2 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 4. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rm_sample)

# -----------------------------
# 5. Calculate metrics (exclude noise for silhouette/DBI)
# -----------------------------
mask = labels_final != -1
sil = silhouette_score(rm_sample[mask], labels_final[mask])
dbi = davies_bouldin_score(rm_sample[mask], labels_final[mask])
print(f"\nRM DBSCAN Silhouette Score: {sil:.4f}")
print(f"RM DBSCAN Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Prepare cluster summary table
# -----------------------------
plot_df_rm = pd.DataFrame(rm_sample, columns=['Recency','Monetary'])
plot_df_rm['Cluster'] = labels_final.astype(str)

cluster_summary = plot_df_rm.groupby('Cluster').mean().round(2)

# Add interpretations
cluster_summary['Interpretation'] = [
    'Low recency, Low monetary spend' if c=='0' else
    'Low recency, High monetary spend' if c=='1' else
    'Noise / Outliers'
    for c in cluster_summary.index
]

print("\nRM DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_rm,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on RM (Standardized, 20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 0. Prepare RM data
# -----------------------------
rm = rm_df[['Recency', 'Monetary']].copy()

# -----------------------------
# 1. Standardize RM data
# -----------------------------
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 2. Sample 20,000 points for DBSCAN (if dataset is smaller, use all)
# -----------------------------
if rm_scaled.shape[0] > 20000:
    rm_sample = rm_scaled[np.random.choice(rm_scaled.shape[0], 20000, replace=False)]
else:
    rm_sample = rm_scaled

# -----------------------------
# 3. Tune eps to get at least 2 clusters
# -----------------------------
eps_values = np.arange(1.0, 10.0, 0.25)
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:  # want at least 2 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")

# -----------------------------
# 4. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(rm_sample)

# -----------------------------
# 5. Calculate metrics
# -----------------------------
mask = labels_final != -1  # exclude noise for metrics
if len(set(labels_final[mask])) >= 2:  # check at least 2 clusters
    sil = silhouette_score(rm_sample[mask], labels_final[mask])
    dbi = davies_bouldin_score(rm_sample[mask], labels_final[mask])
    print(f"\nRM DBSCAN Silhouette Score: {sil:.4f}")
    print(f"RM DBSCAN Davies–Bouldin Index: {dbi:.4f}")
else:
    print("\nToo few clusters to calculate Silhouette Score or DBI.")

# -----------------------------
# 6. Cluster Summary Table
# -----------------------------
plot_df_db = pd.DataFrame(rm_sample, columns=['Recency','Monetary'])
plot_df_db['Cluster'] = labels_final.astype(str)

cluster_summary = plot_df_db.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Noise / Outliers' if c=='-1' else
    'Low recency, Low monetary spend' if c=='0' else
    'Low recency, High monetary spend' if c=='1' else
    'Other'
    for c in cluster_summary.index
]

print("\nRM DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM DBSCAN Clustering (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# 3. Tune eps to get at least 2 clusters
# -----------------------------
eps_values = np.arange(0.5, 10.0, 0.25)  # try slightly smaller start
best_eps = None
best_clusters = 0
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(rm_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:  # want at least 2 clusters
        best_eps = eps
        best_clusters = n_clusters
        break

# If tuning fails, pick a default small value
if best_eps is None:
    best_eps = 2.5
    db = DBSCAN(eps=best_eps, min_samples=5)
    labels = db.fit_predict(rm_sample)
    best_clusters = len(set(labels)) - (1 if -1 in labels else 0)

print(f"Tuned DBSCAN: eps={best_eps}, clusters={best_clusters}, noise points={list(labels).count(-1)}")


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN

# -----------------------------
# 1. Fit final DBSCAN with tuned eps
# -----------------------------
db_final = DBSCAN(eps=0.5, min_samples=5)
labels_final = db_final.fit_predict(rm_sample)

# -----------------------------
# 2. Calculate metrics (exclude noise for Silhouette/DBI)
# -----------------------------
mask = labels_final != -1  # exclude noise points
sil = silhouette_score(rm_sample[mask], labels_final[mask])
dbi = davies_bouldin_score(rm_sample[mask], labels_final[mask])
print(f"RM DBSCAN Silhouette Score: {sil:.4f}")
print(f"RM DBSCAN Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 3. Cluster summary table
# -----------------------------
plot_df_db = pd.DataFrame(rm_sample[:, :2], columns=['Recency','Monetary'])
plot_df_db['Cluster'] = labels_final.astype(str)

cluster_summary = plot_df_db.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Noise / Outliers' if c == '-1' else
    'Low recency, Low monetary spend' if c == '0' else
    'Low recency, High monetary spend' if c == '1' else
    'High recency, Low monetary spend'
    for c in cluster_summary.index
]

print("\nRM DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 4. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM DBSCAN Clustering (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare RM data
# -----------------------------
rm = df[['Recency', 'Monetary']].copy()

scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 2. Fit Hierarchical clustering
# -----------------------------
hier = AgglomerativeClustering(
    n_clusters=3,
    linkage='ward'
)

labels_hier = hier.fit_predict(rm_scaled)

# -----------------------------
# 3. Metrics
# -----------------------------
sil = silhouette_score(rm_scaled, labels_hier)
dbi = davies_bouldin_score(rm_scaled, labels_hier)

print(f"RM Hierarchical Silhouette Score: {sil:.4f}")
print(f"RM Hierarchical Davies–Bouldin Index: {dbi:.4f}")


In [ ]:
# -----------------------------
# 1. Prepare RM data
# -----------------------------
rm = rm_df[['Recency', 'Monetary']].copy()

scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. Fit Hierarchical Clustering
# -----------------------------
hier = AgglomerativeClustering(
    n_clusters=3,
    linkage='ward'
)
labels_hier = hier.fit_predict(rm_sample)

# -----------------------------
# 2. Calculate metrics
# -----------------------------
sil = silhouette_score(rm_sample, labels_hier)
dbi = davies_bouldin_score(rm_sample, labels_hier)

print(f"RM Hierarchical Silhouette Score: {sil:.4f}")
print(f"RM Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 3. Cluster summary table
# -----------------------------
plot_df_hier = pd.DataFrame(rm_sample[:, :2], columns=['Recency', 'Monetary'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

cluster_summary = (
    plot_df_hier
    .groupby('Cluster')
    .mean()
    .round(2)
)

cluster_summary['Interpretation'] = [
    'High recency, Low monetary spend' if c == '0' else
    'Low recency, Low monetary spend' if c == '1' else
    'Low recency, High monetary spend'
    for c in cluster_summary.index
]

print("\nRM Hierarchical Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 4. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('RM Hierarchical Clustering (20k sample)', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# -----------------------------
# Prepare RM data
# -----------------------------
# Replace rm_df with the dataframe you actually used for RM
rm = rm_df[['Recency', 'Monetary']].copy()

scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# Sample 20,000 points (for consistency)
# -----------------------------
if rm_scaled.shape[0] > 20000:
    rm_sample = rm_scaled[
        np.random.choice(rm_scaled.shape[0], 20000, replace=False)
    ]
else:
    rm_sample = rm_scaled

print("RM sample shape:", rm_sample.shape)


In [ ]:
print(globals().keys())


In [ ]:
data = pd.read_excel("Online Retail.xlsx")


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# -----------------------------
# 1. Prepare RM dataset
# -----------------------------
rm = df[['Recency', 'Monetary']].copy()

# -----------------------------
# 2. Standardize RM
# -----------------------------
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 3. Sample 20,000 points (if dataset is larger)
# -----------------------------
if rm_scaled.shape[0] > 20000:
    rm_sample = rm_scaled[np.random.choice(rm_scaled.shape[0], 20000, replace=False)]
else:
    rm_sample = rm_scaled

print("RM sample shape:", rm_sample.shape)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Load the Excel file
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"  # adjust path if needed
df = pd.read_excel(file_path)

# -----------------------------
# 2. Prepare RM data
# -----------------------------
# Ensure the columns exist in your dataset
rm = df[['Recency', 'Monetary']].copy()

# -----------------------------
# 3. Standardize the data
# -----------------------------
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 4. Fit Hierarchical Clustering
# -----------------------------
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hier = hier.fit_predict(rm_scaled)

# -----------------------------
# 5. Calculate metrics
# -----------------------------
sil = silhouette_score(rm_scaled, labels_hier)
dbi = davies_bouldin_score(rm_scaled, labels_hier)
print(f"RM Hierarchical Silhouette Score: {sil:.4f}")
print(f"RM Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Create cluster summary table
# -----------------------------
plot_df_hier = pd.DataFrame(rm_scaled, columns=['Recency','Monetary'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

cluster_summary = plot_df_hier.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'High recency, Low monetary spend' if c=='0' else
    'Low recency, Low monetary spend' if c=='1' else
    'Low recency, High monetary spend'
    for c in cluster_summary.index
]

print("\nRM Hierarchical Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM Hierarchical Clustering', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

# See all column names
print(df.columns)


In [ ]:
import pandas as pd

file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"
df = pd.read_excel(file_path)

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [ ]:
import numpy as np

# Set a snapshot date (usually one day after the last invoice)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group by CustomerID
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,   # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'Quantity': lambda x: (x * df.loc[x.index, 'UnitPrice']).sum()  # Monetary
}).reset_index()

# Rename columns
rfm.rename(columns={
    'InvoiceDate':'Recency',
    'InvoiceNo':'Frequency',
    0:'Monetary'
}, inplace=True)

# Remove customers with negative or zero monetary
rfm = rfm[rfm['Monetary'] > 0]

print(rfm.head())


In [ ]:
import pandas as pd

# Full path to your file
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"

# Load the Excel file (default loads first sheet)
df = pd.read_excel(file_path)

# Check first few rows
print(df.head())

# Optional: check data types and missing values
print(df.info())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Use one day after the last invoice date as the snapshot date
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)


In [ ]:
# Group by CustomerID to calculate RFM
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency (number of invoices)
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # Monetary (total spending)
}).rename(columns={'InvoiceDate':'Recency', 'InvoiceNo':'Frequency', 'UnitPrice':'Monetary'})

# Optional: view the first few rows
print(rfm.head())

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Full RFM
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm), columns=rfm.columns, index=rfm.index)

# RFM subsets
rf_scaled = pd.DataFrame(scaler.fit_transform(rf[['Recency','Frequency']]), columns=['Recency','Frequency'], index=rf.index)
fm_scaled = pd.DataFrame(scaler.fit_transform(fm[['Frequency','Monetary']]), columns=['Frequency','Monetary'], index=fm.index)
rm_scaled = pd.DataFrame(scaler.fit_transform(rm[['Recency','Monetary']]), columns=['Recency','Monetary'], index=rm.index)

# Raw features
raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features), columns=raw_features.columns, index=raw_features.index)

In [ ]:
# Assuming you already have `rfm` dataframe from RFM calculation

# RF subset
rf = rfm[['Recency', 'Frequency']]

# FM subset
fm = rfm[['Frequency', 'Monetary']]

# RM subset
rm = rfm[['Recency', 'Monetary']]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Load the Excel file
# -----------------------------
file_path = r"C:\Users\ASUS\Downloads\Online Retail.xlsx"  # adjust path if needed
df = pd.read_excel(file_path)

# -----------------------------
# 2. Prepare RM data
# -----------------------------
# Ensure the columns exist in your dataset
rm = df[['Recency', 'Monetary']].copy()

# -----------------------------
# 3. Standardize the data
# -----------------------------
scaler = StandardScaler()
rm_scaled = scaler.fit_transform(rm)

# -----------------------------
# 4. Fit Hierarchical Clustering
# -----------------------------
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hier = hier.fit_predict(rm_scaled)

# -----------------------------
# 5. Calculate metrics
# -----------------------------
sil = silhouette_score(rm_scaled, labels_hier)
dbi = davies_bouldin_score(rm_scaled, labels_hier)
print(f"RM Hierarchical Silhouette Score: {sil:.4f}")
print(f"RM Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 6. Create cluster summary table
# -----------------------------
plot_df_hier = pd.DataFrame(rm_scaled, columns=['Recency','Monetary'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

cluster_summary = plot_df_hier.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'High recency, Low monetary spend' if c=='0' else
    'Low recency, Low monetary spend' if c=='1' else
    'Low recency, High monetary spend'
    for c in cluster_summary.index
]

print("\nRM Hierarchical Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM Hierarchical Clustering', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Full RFM
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm), columns=rfm.columns, index=rfm.index)

# RFM subsets
rf_scaled = pd.DataFrame(scaler.fit_transform(rf), columns=rf.columns, index=rf.index)
fm_scaled = pd.DataFrame(scaler.fit_transform(fm), columns=fm.columns, index=fm.index)
rm_scaled = pd.DataFrame(scaler.fit_transform(rm), columns=rm.columns, index=rm.index)

# Raw features
raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features), columns=raw_features.columns, index=raw_features.index)


In [ ]:
# Assuming you already have `rfm` dataframe from RFM calculation

# RF subset
rf = rfm[['Recency', 'Frequency']]

# FM subset
fm = rfm[['Frequency', 'Monetary']]

# RM subset
rm = rfm[['Recency', 'Monetary']]

In [ ]:
# Aggregate transactional data per customer
raw_features = df.groupby('CustomerID').agg({
    'Quantity': 'sum',   # Total quantity purchased
    'InvoiceNo': 'nunique',  # Total number of invoices/orders
    'UnitPrice': lambda x: np.sum(x * df.loc[x.index, 'Quantity'])  # Total spend
}).rename(columns={'InvoiceNo':'TotalOrders', 'UnitPrice':'TotalSpend'})

# Optional: add average basket value
raw_features['AvgBasket'] = raw_features['TotalSpend'] / raw_features['TotalOrders']

# Check the first few rows
print(raw_features.head())


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

raw_scaled = pd.DataFrame(scaler.fit_transform(raw_features), 
                          columns=raw_features.columns, 
                          index=raw_features.index)

# Check
print(raw_scaled.head())


In [ ]:
print(rfm_scaled.head())   # Full RFM
print(rf_scaled.head())    # RF subset
print(fm_scaled.head())    # FM subset
print(rm_scaled.head())    # RM subset
print(raw_scaled.head())   # Raw features

In [ ]:
# -----------------------------
# RM Hierarchical Clustering
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare RM dataset
# -----------------------------
# Assuming 'rm_scaled' already exists (Recency & Monetary standardized)
X = rm_scaled.copy()  # shape (n_samples, 2)

# -----------------------------
# 2. Fit Hierarchical Clustering
# -----------------------------
hier = AgglomerativeClustering(
    n_clusters=3,  # you can adjust
    linkage='ward'
)
labels_hier = hier.fit_predict(X)

# -----------------------------
# 3. Calculate metrics
# -----------------------------
sil = silhouette_score(X, labels_hier)
dbi = davies_bouldin_score(X, labels_hier)
print(f"RM Hierarchical Silhouette Score: {sil:.4f}")
print(f"RM Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 4. Prepare Cluster Summary Table
# -----------------------------
plot_df_hier = pd.DataFrame(X, columns=['Recency','Monetary'])
plot_df_hier['Cluster'] = labels_hier.astype(str)

cluster_summary = plot_df_hier.groupby('Cluster').mean().round(2)

# Add interpretation based on Recency & Monetary
cluster_summary['Interpretation'] = [
    'High recency, low monetary spend' if c=='0' else
    'Low recency, high monetary spend' if c=='1' else
    'Low recency, low monetary spend'
    for c in cluster_summary.index
]

print("\nRM Hierarchical Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 5. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('RM Hierarchical Clustering', fontsize=14)
plt.xlabel('Recency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Models and metrics
models = ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical']
silhouette_scores = [0.6824, 0.3785, 0.9274, 0.6915, 0.6658]
dbi_scores = [0.4369, 1.3940, 0.2726, 0.3453, 0.4145]

x = np.arange(len(models))  # label locations
width = 0.35  # width of bars

fig, ax = plt.subplots(figsize=(10,6))

# Turn off grid completely
ax.grid(False)

# Plot bars
bar1 = ax.bar(x - width/2, silhouette_scores, width, label='Silhouette Score', color='#4C72B0')
bar2 = ax.bar(x + width/2, dbi_scores, width, label='Davies–Bouldin Index', color='#55A868')

# Labels and title
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparison of Clustering Models (RM dataset)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=0, fontsize=11)
ax.legend(fontsize=11)

# Show values on top
for rect in bar1 + bar2:
    height = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2., 1.01*height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare FM data (Frequency & Monetary)
# -----------------------------
# Assuming 'fm_scaled' is your standardized FM DataFrame
X = fm_scaled.values  # convert to numpy array for clustering

# -----------------------------
# 2. Fit K-Means
# -----------------------------
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X)

# -----------------------------
# 3. Metrics
# -----------------------------
sil = silhouette_score(X, labels)
dbi = davies_bouldin_score(X, labels)
print(f"FM K-Means Silhouette Score: {sil:.4f}")
print(f"FM K-Means Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 4. Cluster Summary Table
# -----------------------------
plot_df = pd.DataFrame(fm_scaled, columns=['Frequency','Monetary'])
plot_df['Cluster'] = labels.astype(str)  # as string for consistency

cluster_summary = plot_df.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Low frequency, Low monetary spend' if c=='0' else
    'Low frequency, High monetary spend' if c=='1' else
    'High frequency, High monetary spend'
    for c in cluster_summary.index
]

print("\nFM K-Means Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 5. Plot Clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('FM K-Means Clustering', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------
# 1. Prepare FM data (already scaled)
# --------------------------------
# fm_scaled already exists based on your output
# Columns: ['Frequency', 'Monetary']

# --------------------------------
# 2. Fit GMM
# --------------------------------
gmm = GaussianMixture(n_components=3, random_state=42)
labels_gmm = gmm.fit_predict(fm_scaled)

# --------------------------------
# 3. Metrics
# --------------------------------
sil = silhouette_score(fm_scaled, labels_gmm)
dbi = davies_bouldin_score(fm_scaled, labels_gmm)

print(f"FM GMM Silhouette Score: {sil:.4f}")
print(f"FM GMM Davies–Bouldin Index: {dbi:.4f}")

# --------------------------------
# 4. Cluster summary table
# --------------------------------
plot_df_gmm = fm_scaled.copy()
plot_df_gmm['Cluster'] = labels_gmm

cluster_summary = (
    plot_df_gmm
    .groupby('Cluster')[['Frequency', 'Monetary']]
    .mean()
    .round(2)
)

cluster_summary['Interpretation'] = [
    'Low frequency, Low monetary spend',
    'Low frequency, High monetary spend',
    'High frequency, High monetary spend'
]

print("\nFM GMM Cluster Summary Table:")
print(cluster_summary)

# --------------------------------
# 5. Plot clusters
# --------------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_gmm,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('FM GMM Clustering (Standardized)', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score

# --------------------------------
# 1. Use FM standardized data
# --------------------------------
# fm_scaled already exists with columns ['Frequency','Monetary']
fm_data = fm_scaled[['Frequency', 'Monetary']].copy()

# --------------------------------
# 2. Fit BIRCH
# --------------------------------
birch = Birch(n_clusters=3)
labels_birch = birch.fit_predict(fm_data)

# --------------------------------
# 3. Metrics
# --------------------------------
sil = silhouette_score(fm_data, labels_birch)
dbi = davies_bouldin_score(fm_data, labels_birch)

print(f"FM BIRCH Silhouette Score: {sil:.4f}")
print(f"FM BIRCH Davies–Bouldin Index: {dbi:.4f}")

# --------------------------------
# 4. Cluster summary table
# --------------------------------
plot_df_fm = fm_data.copy()
plot_df_fm['Cluster'] = labels_birch.astype(str)

cluster_summary = plot_df_fm.groupby('Cluster').mean().round(2)

cluster_summary['Interpretation'] = [
    'Low frequency, Low monetary spend' if c == '0' else
    'Low frequency, High monetary spend' if c == '1' else
    'High frequency, High monetary spend'
    for c in cluster_summary.index
]

print("\nFM BIRCH Cluster Summary Table:")
print(cluster_summary)

# --------------------------------
# 5. Plot
# --------------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_fm,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('FM BIRCH Clustering (Standardized)', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare FM data
# -----------------------------
# fm_scaled already exists and is standardized
# Columns: ['Frequency', 'Monetary']

X = fm_scaled.values

# -----------------------------
# 2. Sample (DBSCAN scalability)
# -----------------------------
if X.shape[0] > 20000:
    fm_sample = X[np.random.choice(X.shape[0], 20000, replace=False)]
else:
    fm_sample = X

print(f"DBSCAN running on {fm_sample.shape[0]} samples")

# -----------------------------
# 3. Tune eps to get ≥2 clusters
# -----------------------------
eps_values = np.arange(0.2, 5.0, 0.1)
best_eps = None

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(fm_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    if n_clusters >= 2:
        best_eps = eps
        break

print(f"Tuned DBSCAN eps = {best_eps}")

# -----------------------------
# 4. Fit final DBSCAN
# -----------------------------
db_final = DBSCAN(eps=best_eps, min_samples=5)
labels_final = db_final.fit_predict(fm_sample)

# -----------------------------
# 5. Metrics (exclude noise)
# -----------------------------
mask = labels_final != -1

if len(set(labels_final[mask])) > 1:
    sil = silhouette_score(fm_sample[mask], labels_final[mask])
    dbi = davies_bouldin_score(fm_sample[mask], labels_final[mask])

    print(f"FM DBSCAN Silhouette Score: {sil:.4f}")
    print(f"FM DBSCAN Davies–Bouldin Index: {dbi:.4f}")
else:
    print("Too few clusters for metric calculation.")

# -----------------------------
# 6. Cluster Summary Table
# -----------------------------
plot_df_db = pd.DataFrame(
    fm_sample,
    columns=['Frequency', 'Monetary']
)
plot_df_db['Cluster'] = labels_final.astype(str)

cluster_summary = plot_df_db.groupby('Cluster').mean().round(2)

cluster_summary['Interpretation'] = [
    'Noise / Outliers' if c == '-1' else
    'Low frequency, Low monetary spend' if c == '0' else
    'Low frequency, High monetary spend' if c == '1' else
    'High frequency, High monetary spend'
    for c in cluster_summary.index
]

print("\nFM DBSCAN Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 7. Plot clusters
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_db,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)

plt.title('FM DBSCAN Clustering (Standardized)', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare FM dataset
# -----------------------------
# Assuming you already have FM features (Frequency, Monetary) in a dataframe called fm_scaled
# fm_scaled should be standardized

# -----------------------------
# 2. Fit Hierarchical Clustering
# -----------------------------
hier = AgglomerativeClustering(
    n_clusters=3,  # as per previous models
    linkage='ward'
)
labels_hier = hier.fit_predict(fm_scaled)

# -----------------------------
# 3. Calculate metrics
# -----------------------------
sil = silhouette_score(fm_scaled, labels_hier)
dbi = davies_bouldin_score(fm_scaled, labels_hier)
print(f"FM Hierarchical Silhouette Score: {sil:.4f}")
print(f"FM Hierarchical Davies–Bouldin Index: {dbi:.4f}")

# -----------------------------
# 4. Cluster summary table
# -----------------------------
plot_df_hier = fm_scaled.copy()
plot_df_hier['Cluster'] = labels_hier.astype(str)

cluster_summary = plot_df_hier.groupby('Cluster').mean().round(2)
cluster_summary['Interpretation'] = [
    'Low frequency, Low monetary spend' if c=='0' else
    'Low frequency, High monetary spend' if c=='1' else
    'High frequency, High monetary spend'
    for c in cluster_summary.index
]

print("\nFM Hierarchical Cluster Summary Table:")
print(cluster_summary)

# -----------------------------
# 5. Plot clusters (2D)
# -----------------------------
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=plot_df_hier,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('FM Hierarchical Clustering', fontsize=14)
plt.xlabel('Frequency (standardized)')
plt.ylabel('Monetary (standardized)')
plt.legend(title='Cluster', loc='upper right', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Create DataFrame
fm_comparison = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Silhouette Score': [0.8230, 0.4818, 0.9569, 0.8908, 0.9435],
    'Davies–Bouldin Index': [0.7477, 0.9634, 0.4380, 0.1155, 0.5754]
})

# Set figure size
plt.figure(figsize=(10,6))

# Set width of bar
bar_width = 0.35
index = np.arange(len(fm_comparison))

# Plot Silhouette Score
plt.bar(index, fm_comparison['Silhouette Score'], bar_width, label='Silhouette Score', color='skyblue')

# Plot DBI
plt.bar(index + bar_width, fm_comparison['Davies–Bouldin Index'], bar_width, label='Davies–Bouldin Index', color='salmon')

# Labels and ticks
plt.xlabel('Clustering Model', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)
plt.title('Comparison of FM Clustering Models', fontsize=14)
plt.xticks(index + bar_width/2, fm_comparison['Model'])
plt.ylim(0, 1.2)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Comparison DataFrame
fm_comparison = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Silhouette Score': [0.8230, 0.4818, 0.9569, 0.8908, 0.9435],
    'Davies–Bouldin Index': [0.7477, 0.9634, 0.4380, 0.1155, 0.5754]
})

# Figure size
plt.figure(figsize=(10,6))

# Bar width
bar_width = 0.35
index = np.arange(len(fm_comparison))

# Plot Silhouette Score (green)
plt.bar(index, fm_comparison['Silhouette Score'], bar_width, label='Silhouette Score', color='#4CAF50')

# Plot Davies–Bouldin Index (blue)
plt.bar(index + bar_width, fm_comparison['Davies–Bouldin Index'], bar_width, label='Davies–Bouldin Index', color='#2196F3')

# Labels and title
plt.xlabel('Clustering Model', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)
plt.title('Comparison of FM Clustering Models', fontsize=14)
plt.xticks(index + bar_width/2, fm_comparison['Model'])
plt.ylim(0, 1.2)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Comparison DataFrame
fm_comparison = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Silhouette Score': [0.8230, 0.4818, 0.9569, 0.8908, 0.9435],
    'Davies–Bouldin Index': [0.7477, 0.9634, 0.4380, 0.1155, 0.5754]
})

# Figure size
plt.figure(figsize=(10,6))

# Bar width
bar_width = 0.35
index = np.arange(len(fm_comparison))

# Plot Silhouette Score (green)
plt.bar(index, fm_comparison['Silhouette Score'], bar_width, label='Silhouette Score', color='##4C72B0')

# Plot Davies–Bouldin Index (blue)
plt.bar(index + bar_width, fm_comparison['Davies–Bouldin Index'], bar_width, label='Davies–Bouldin Index', color='#55A868')

# Labels and title
plt.xlabel('Clustering Model', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)
plt.title('Comparison of FM Clustering Models', fontsize=14)
plt.xticks(index + bar_width/2, fm_comparison['Model'])
plt.ylim(0, 1.2)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Comparison DataFrame
fm_comparison = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Silhouette Score': [0.8230, 0.4818, 0.9569, 0.8908, 0.9435],
    'Davies–Bouldin Index': [0.7477, 0.9634, 0.4380, 0.1155, 0.5754]
})

# Figure size
plt.figure(figsize=(10,6))

# Bar width
bar_width = 0.35
index = np.arange(len(fm_comparison))

# Plot Silhouette Score (green)
plt.bar(index, fm_comparison['Silhouette Score'], bar_width, label='Silhouette Score', color='#4C72B0')

# Plot Davies–Bouldin Index (blue)
plt.bar(index + bar_width, fm_comparison['Davies–Bouldin Index'], bar_width, label='Davies–Bouldin Index', color='#55A868')

# Labels and title
plt.xlabel('Clustering Model', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)
plt.title('Comparison of FM Clustering Models', fontsize=14)
plt.xticks(index + bar_width/2, fm_comparison['Model'])
plt.ylim(0, 1.2)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Comparison DataFrame
fm_comparison = pd.DataFrame({
    'Model': ['K-Means', 'GMM', 'BIRCH', 'DBSCAN', 'Hierarchical'],
    'Silhouette Score': [0.8230, 0.4818, 0.9569, 0.8908, 0.9435],
    'Davies–Bouldin Index': [0.7477, 0.9634, 0.4380, 0.1155, 0.5754]
})

# Figure size
plt.figure(figsize=(10,6))

# Bar width
bar_width = 0.35
index = np.arange(len(fm_comparison))

# Plot Silhouette Score (green)
bars1 = plt.bar(index, fm_comparison['Silhouette Score'], bar_width, label='Silhouette Score', color='#4C72B0')
# Plot Davies–Bouldin Index (blue)
bars2 = plt.bar(index + bar_width, fm_comparison['Davies–Bouldin Index'], bar_width, label='Davies–Bouldin Index', color='#55A868')

# Add data labels on top of bars
for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{height:.2f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{height:.2f}', ha='center', va='bottom', fontsize=10)

# Labels and title
plt.xlabel('Clustering Model', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)
plt.title('Comparison of FM Clustering Models', fontsize=14)
plt.xticks(index + bar_width/2, fm_comparison['Model'])
plt.ylim(0, 1.2)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

best_sil = -1
best_dbi = None
best_params = None

n_components_range = [2, 3, 4, 5]  # try 2–5 clusters
cov_types = ['full', 'tied', 'diag', 'spherical']

for n in n_components_range:
    for cov in cov_types:
        gmm = GaussianMixture(n_components=n, covariance_type=cov, random_state=42, n_init=10)
        labels = gmm.fit_predict(fm_scaled)
        sil = silhouette_score(fm_scaled, labels)
        dbi = davies_bouldin_score(fm_scaled, labels)
        
        if sil > best_sil:  # choose the one with highest silhouette
            best_sil = sil
            best_dbi = dbi
            best_params = (n, cov)

print(f"Best GMM parameters: n_components={best_params[0]}, covariance_type={best_params[1]}")
print(f"Silhouette Score: {best_sil:.4f}")
print(f"Davies–Bouldin Index: {best_dbi:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Load or define RFM dataset
# -----------------------------
# Replace this with your actual RFM dataframe if it exists
# Example structure:
# rfm = pd.DataFrame({
#     'Recency': [...],
#     'Frequency': [...],
#     'Monetary': [...]
# })

# -----------------------------
# 2. Prepare RFM data
# -----------------------------
X_rfm = rfm[['Recency', 'Frequency', 'Monetary']].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# -----------------------------
# 3. Hyperparameter tuning for GMM
# -----------------------------
best_sil = -1
best_dbi = np.inf
best_params = None

n_components_list = [2, 3, 4, 5]
covariance_types = ['full', 'tied', 'diag', 'spherical']

for n in n_components_list:
    for cov_type in covariance_types:
        gmm = GaussianMixture(n_components=n, covariance_type=cov_type, random_state=42)
        labels = gmm.fit_predict(X_scaled)
        if len(set(labels)) < 2:
            continue  # Skip if less than 2 clusters
        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)
        if sil > best_sil:
            best_sil = sil
            best_dbi = dbi
            best_params = (n, cov_type)

# -----------------------------
# 4. Output best results
# -----------------------------
print(f"Best GMM parameters: n_components={best_params[0]}, covariance_type={best_params[1]}")
print(f"Silhouette Score: {best_sil:.4f}")
print(f"Davies–Bouldin Index: {best_dbi:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare RF dataset
# -----------------------------
# Make sure you have your RF dataframe (Recency + Frequency)
# Example: rf = pd.DataFrame({'Recency': [...], 'Frequency': [...]})
X_rf = rf[['Recency', 'Frequency']].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rf)

# -----------------------------
# 2. Hyperparameter tuning for GMM
# -----------------------------
best_sil = -1
best_dbi = np.inf
best_params = None

n_components_list = [2, 3, 4, 5]
covariance_types = ['full', 'tied', 'diag', 'spherical']

for n in n_components_list:
    for cov_type in covariance_types:
        gmm = GaussianMixture(n_components=n, covariance_type=cov_type, random_state=42)
        labels = gmm.fit_predict(X_scaled)
        if len(set(labels)) < 2:
            continue  # Skip if less than 2 clusters
        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)
        if sil > best_sil:
            best_sil = sil
            best_dbi = dbi
            best_params = (n, cov_type)

# -----------------------------
# 3. Output best results
# -----------------------------
print(f"Best GMM parameters for RF: n_components={best_params[0]}, covariance_type={best_params[1]}")
print(f"Silhouette Score: {best_sil:.4f}")
print(f"Davies–Bouldin Index: {best_dbi:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -----------------------------
# 1. Prepare RM dataset
# -----------------------------
# Make sure you have your RM dataframe (Recency + Monetary)
# Example: rm = pd.DataFrame({'Recency': [...], 'Monetary': [...]})
X_rm = rm[['Recency', 'Monetary']].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rm)

# -----------------------------
# 2. Hyperparameter tuning for GMM
# -----------------------------
best_sil = -1
best_dbi = np.inf
best_params = None

n_components_list = [2, 3, 4, 5]
covariance_types = ['full', 'tied', 'diag', 'spherical']

for n in n_components_list:
    for cov_type in covariance_types:
        gmm = GaussianMixture(n_components=n, covariance_type=cov_type, random_state=42)
        labels = gmm.fit_predict(X_scaled)
        if len(set(labels)) < 2:
            continue  # Skip if less than 2 clusters
        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)
        if sil > best_sil:
            best_sil = sil
            best_dbi = dbi
            best_params = (n, cov_type)

# -----------------------------
# 3. Output best results
# -----------------------------
print(f"Best GMM parameters for RM: n_components={best_params[0]}, covariance_type={best_params[1]}")
print(f"Silhouette Score: {best_sil:.4f}")
print(f"Davies–Bouldin Index: {best_dbi:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# Load your RFM dataset
# -------------------------------
# Example: rfm = pd.read_csv("rfm_data.csv")
# Assuming rfm has columns ['Recency', 'Frequency', 'Monetary']

X = rfm[['Recency', 'Frequency', 'Monetary']]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# K-Means Clustering
# -------------------------------
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

# -------------------------------
# DBSCAN Clustering
# -------------------------------
dbscan = DBSCAN(eps=1.5, min_samples=5)  # adjust eps as needed
dbscan_labels = dbscan.fit_predict(X_scaled)

# -------------------------------
# Optional: PCA for 2D Visualization
# -------------------------------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# -------------------------------
# Plot K-Means
# -------------------------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=kmeans_labels, palette='Set2', s=60)
plt.title("K-Means Clustering (RFM) - PCA Projection")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title='Cluster')

# -------------------------------
# Plot DBSCAN
# -------------------------------
plt.subplot(1,2,2)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=dbscan_labels, palette='Set1', s=60)
plt.title("DBSCAN Clustering (RFM) - PCA Projection")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# -------------------------------
# Load your RFM dataset
# -------------------------------
# Example: rfm = pd.read_csv("rfm_data.csv")
# Assuming rfm has columns ['Recency', 'Frequency', 'Monetary']

X = rfm[['Recency', 'Frequency', 'Monetary']]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# K-Means Clustering
# -------------------------------
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

# -------------------------------
# DBSCAN Clustering
# -------------------------------
dbscan = DBSCAN(eps=1.5, min_samples=5)  # adjust eps as needed
dbscan_labels = dbscan.fit_predict(X_scaled)

# -------------------------------
# 3D Plotting function
# -------------------------------
def plot_3d_clusters(X_scaled, labels, title):
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(X_scaled[:,0], X_scaled[:,1], X_scaled[:,2], c=labels, cmap='Set2', s=60)
    ax.set_xlabel('Recency (scaled)')
    ax.set_ylabel('Frequency (scaled)')
    ax.set_zlabel('Monetary (scaled)')
    ax.set_title(title)
    legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
    ax.add_artist(legend1)
    plt.show()

# -------------------------------
# Plot K-Means clusters
# -------------------------------
plot_3d_clusters(X_scaled, kmeans_labels, "K-Means Clustering (RFM)")

# -------------------------------
# Plot DBSCAN clusters
# -------------------------------
plot_3d_clusters(X_scaled, dbscan_labels, "DBSCAN Clustering (RFM)")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming X_scaled is your scaled RFM dataframe
# And labels_kmeans, labels_dbscan are the cluster labels from your models

# Choose two features for 2D plotting: Frequency vs Monetary
plot_features = ['Frequency', 'Monetary']

# K-Means 2D Plot
plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_scaled[plot_features[0]],
    y=X_scaled[plot_features[1]],
    hue=labels_kmeans,
    palette='Set2',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('K-Means Clustering on RFM (Frequency vs Monetary)', fontsize=14)
plt.xlabel(plot_features[0])
plt.ylabel(plot_features[1])
plt.legend(title='Cluster')
plt.show()

# DBSCAN 2D Plot
plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_scaled[plot_features[0]],
    y=X_scaled[plot_features[1]],
    hue=labels_dbscan,
    palette='Set1',
    s=50,
    alpha=0.7,
    edgecolor='w'
)
plt.title('DBSCAN Clustering on RFM (Frequency vs Monetary)', fontsize=14)
plt.xlabel(plot_features[0])
plt.ylabel(plot_features[1])
plt.legend(title='Cluster')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Column indices
freq_idx = 1
mon_idx = 2

plt.figure(figsize=(8,6))
plt.scatter(
    X_scaled[:, freq_idx],
    X_scaled[:, mon_idx],
    c=labels_kmeans,
    alpha=0.7
)
plt.title('K-Means Clustering on RFM (Frequency vs Monetary)')
plt.xlabel('Frequency')
plt.ylabel('Monetary')
plt.show()


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
labels_kmeans = kmeans.fit_predict(X_scaled)


In [ ]:
import matplotlib.pyplot as plt

# Column indices
freq_idx = 1
mon_idx = 2

plt.figure(figsize=(8,6))
plt.scatter(
    X_scaled[:, freq_idx],
    X_scaled[:, mon_idx],
    c=labels_kmeans,
    alpha=0.7
)
plt.title('K-Means Clustering on RFM (Frequency vs Monetary)')
plt.xlabel('Frequency')
plt.ylabel('Monetary')
plt.show()
